# Renown Combat Lab — Reorganized

End-to-end combat analytics for *Renown*. Structured to compute once, analyze many times.

**Section 1 (Compute)** runs the heavy simulations and writes CSVs to disk:
- Main tournament (Random vs Random) with first-skirmish tactic pair logging
- Playstyle-aware tournament (each loadout uses its default playstyle)
- Forced-tactics matrix sweep on a stratified sample
- Horde-mode survival on a stratified sample

All later sections read those CSVs — no re-running sims to look at a new angle.

**Section 2+** organizes the analysis by:
- Pursuit / Domain — does spending correlate with winning?
- Retinue performance
- Equipment (weapons, shields, armor, bastard, lance)
- Tactics (empirical matrix from main run + forced-matrix sweep)
- Playstyles (style-aware tournament results)
- Horde Mode
- Per-loadout rollups (durability, kill power, robustness, counters, casualty sources)

---
**Pool model (current):** uses `balanced_validation_pool` — the CSV-derived innate/mastery pursuit model. Tier/retinue unlocks require the spec MASTERED (MaA→Coliseum, Sgt→War College, KT→Preceptory, each carrying its mastery chain). Regen is a 4-level axis (none/Apothecary/+Infirmary/+Hospitaller). MPC = settlement space (reduced by satisfied *Efficient X*); `total_investment` = raw pursuit count (action economy). Every generated build is checked by `validate_pool` (expects 0 invalid). Set `PER_CELL` in Section 0 to cap each (retinue×MPC) cell; `None` = full ~27k-build pool.


## 0. Setup — imports and global config

Change `LOADOUT_SOURCE` to `"csv"` to load a pre-computed pool from `loadouts.csv` (faster startup, reproducible) or `"generator"` to regenerate from `archetype_pool()`.

Tournament sizes: `N_RUNS_MAIN` is the most expensive (n_loadouts² × N_RUNS battles). `N_RUNS_PLAYSTYLE` runs a second tournament with style-aware loadouts. `N_RUNS_FORCED_TACTICS` runs a smaller forced-tactics sweep for the head-to-head tactic matrix.

In [1]:
import sys, os
sys.path.insert(0, r"C:\Users\Matt\OneDrive\Desktop\Game\Combatv3")
import os, sys, time, json, random, importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter, defaultdict

# Project modules
import renown_combat, vectorized_combat, loadouts, playstyles, tournament_vec, analysis, batch_engine
import tactics_analysis
for m in [renown_combat, vectorized_combat, loadouts, playstyles, tournament_vec, analysis, tactics_analysis, batch_engine]:
    importlib.reload(m)
vectorized_combat.invalidate_tactic_tables()

from renown_combat import TACTICS

# ── CONFIG ──────────────────────────────────────────────────────────────────
OUT_DIR        = r"C:\Users\Matt\OneDrive\Desktop\Game\Combatv3\lab_out"
os.makedirs(OUT_DIR, exist_ok=True)

# ── Loadout source ───────────────────────────────────────────────────────────
# "csv": load a frozen pool you saved earlier (set LOADOUT_CSV to its path).
# "generator": build a fresh pool from the current rules, using the MPC bracket below,
#              and SAVE it to a parquet+csv you can re-pull later (see cell 4).
LOADOUT_SOURCE = "csv"     # "csv" or "generator"
LOADOUT_CSV    = r"C:\Users\Matt\OneDrive\Desktop\Game\Combatv3\lab_out\loadouts_gen_8_12.csv"   # used only when LOADOUT_SOURCE == "csv"

# ── Editable MPC bracket (generation only) ─────────────────────────────────────
# The pool spans builds whose Military Pursuit Count falls in [MPC_MIN, MPC_MAX].
# These feed loadouts.balanced_validation_pool(mpc_min=MPC_MIN, mpc_max=MPC_MAX, per_cell=PER_CELL).
MPC_MIN = 4
MPC_MAX = 13
# PER_CELL caps each (retinue x MPC) cell. None = full pool (large: ~27k builds, heavy round-robin).
# Set an integer (e.g. 50) to keep matchup counts tractable; the .bat uses 50.
PER_CELL = 50
# Where to save a freshly generated pool (so you can reload it later via LOADOUT_SOURCE="csv").
# The MPC range is baked into the name so saved pools are self-identifying.
SAVE_GENERATED_POOL = True
GEN_POOL_BASENAME   = os.path.join(OUT_DIR, f"loadouts_gen_{MPC_MIN}_{MPC_MAX}")  # .parquet + .csv

# ── Run counts ─────────────────────────────────────────────────────────────────
# N_RUNS_MAIN is the single source of truth; every other tournament's run count
# derives from it. Edit N_RUNS_MAIN to scale them all together.
N_RUNS_MAIN      = 100
N_RUNS_PLAYSTYLE = N_RUNS_MAIN
N_RUNS_FORCED    = N_RUNS_MAIN
N_RUNS_HORDE     = N_RUNS_MAIN

# ── Tournament / sampling knobs ─────────────────────────────────────────────────
dont = False # ignore playstyle tournament
USE_BATCH      = True     # batched engine (~7-8x faster)
POOL_SIZE      = None     # None = full pool; set e.g. 200 for a fast smoke run
# Stratified subsample: cap builds PER MPC bucket so the whole range is covered without the
# high-MPC combinatorial blowup. Rows scale ~quadratically with build count; 80->40 = ~1/4 the
# matchup rows + runtime + RAM. None disables.
STRATIFY_PER_MPC = None   # (legacy; balanced_validation_pool uses PER_CELL instead — leave None)
N_WORKERS_MAIN = max(1, (os.cpu_count() or 4) - 2)   # leave 2 cores free

FORCED_SAMPLE_SIZE = 200
HORDE_BATTLES      = 8
HORDE_SAMPLE_SIZE  = 10

# Reproducibility
random.seed(2026)
np.random.seed(2026)

def _have_matchups(path):
    """True if the matchups file exists as .csv OR its .parquet sibling (batch writes parquet)."""
    p=str(path)
    return os.path.exists(p) or os.path.exists(p[:-4]+".parquet" if p.endswith(".csv") else p+".parquet")

## 0.1 Load loadout pool

In [2]:
# ── Build or load the loadout pool ─────────────────────────────────────────────
# Uses balanced_validation_pool: the analytical pool spanning all gear tiers x retinues,
# with the CSV-derived innate/mastery model, retinue-unlock enforcement (MaA->Coliseum,
# Sgt->War College, KT->Preceptory, each carrying its mastery chain), and the 4-level regen
# axis (none / Apothecary / Apo+Infirmary / Apo+Inf+Hospitaller). per_cell caps each
# (retinue x MPC) cell so the round-robin stays tractable; None = full pool.
if LOADOUT_SOURCE == "csv" and os.path.exists(LOADOUT_CSV):
    pool = loadouts.archetype_pool(csv_path=LOADOUT_CSV)
    print(f"Loaded {len(pool)} loadouts from {LOADOUT_CSV}")
else:
    pool = loadouts.balanced_validation_pool(
        mpc_min=MPC_MIN, mpc_max=MPC_MAX, per_cell=PER_CELL, verbose=True)
    print(f"Generated {len(pool)} loadouts from balanced_validation_pool "
          f"(MPC {MPC_MIN}-{MPC_MAX}, per_cell={PER_CELL})")
    if LOADOUT_SOURCE == "csv":
        print(f"  ({LOADOUT_CSV} not found — fell back to generator)")

    # Validate every generated build against the full rule set (structural + buildings +
    # prereqs + efficiency). Should print 0 invalid; any violations are a generation bug.
    _bad = loadouts.validate_pool(pool)

    # Save the freshly generated pool so it can be re-pulled later via LOADOUT_SOURCE="csv".
    if SAVE_GENERATED_POOL:
        _rows = [{
            "name": l.name, "retinue": l.retinue, "weapon": l.weapon,
            "shield": l.shield or "", "armor": l.armor, "ranged": l.ranged or "",
            "has_tiltyard": l.has_tiltyard, "size": l.size,
            "extra_tags": ",".join(l.extra_tags),
            "upkeep_per_retinue": l.upkeep_per_retinue,
            "playstyle": l.playstyle or "",
            "tiltyard_mastery": getattr(l, "tiltyard_mastery", False),
            "pursuits": "|".join(sorted(l.pursuits)),
            "military_pursuit_count": l.military_pursuit_count,
            "domain_count": l.domain_count,
        } for l in pool]
        _df = pd.DataFrame(_rows)
        _df.to_parquet(GEN_POOL_BASENAME + ".parquet", index=False)
        _df.to_csv(GEN_POOL_BASENAME + ".csv", index=False)
        print(f"  Saved pool -> {GEN_POOL_BASENAME}.parquet (+ .csv)")
        print(f"  To reuse later: set LOADOUT_SOURCE='csv' and LOADOUT_CSV=r'{GEN_POOL_BASENAME}.csv'")

# Quick summary
print("\nBy retinue:")
for r, n in Counter(ld.retinue for ld in pool).most_common():
    print(f"  {r:<15} {n}")
print(f"\nBy MPC (military pursuit count):")
mpc_counts = Counter(ld.military_pursuit_count for ld in pool)
for mpc in sorted(mpc_counts):
    print(f"  mpc={mpc:<3} {mpc_counts[mpc]}")

if POOL_SIZE is not None and POOL_SIZE < len(pool):
    random.seed(2026); pool = random.sample(pool, POOL_SIZE)
    print(f"Subsampled pool to {len(pool)} (POOL_SIZE={POOL_SIZE})")

# Stable handle to the pool that generated the main tournament data.
MAIN_POOL = list(pool)


Loaded 1512 loadouts from C:\Users\Matt\OneDrive\Desktop\Game\Combatv3\lab_out\loadouts_gen_8_12.csv

By retinue:
  Man-at-Arms     714
  Levy            398
  Sergeant        338
  Knight Templar  62

By MPC (military pursuit count):
  mpc=8   458
  mpc=9   208
  mpc=10  241
  mpc=11  275
  mpc=12  330


# 1. Compute — run all simulations once

Each subsection writes CSVs to `OUT_DIR`. Skip subsections you've already run if the CSVs are on disk.

## 1A. Main tournament (Random vs Random)

This is the canonical data source. Every loadout plays every other loadout `N_RUNS_MAIN` times with Random playstyles on both sides. Outputs:
- `summary.csv` — one row per loadout with aggregated metrics
- `matchups.csv` — one row per (a_loadout × b_loadout) pair
- `tactic_matrix.csv` — empirical 7×7 tactic-pair matrix (NEW)

The `tactic_matrix.csv` is produced for free from the main tournament — every first-skirmish tactic pair is logged and attributed to final outcomes. This means we don't need a separate forced-tactic sweep to see how tactics interact in real (random) play.

In [3]:
MAIN_SUMMARY = os.path.join(OUT_DIR, "summary.csv")
MAIN_MATCHUPS = os.path.join(OUT_DIR, "matchups.csv")
MAIN_TACTIC_MATRIX = os.path.join(OUT_DIR, "tactic_matrix.csv")

if os.path.exists(MAIN_SUMMARY) and _have_matchups(MAIN_MATCHUPS):
    print(MAIN_SUMMARY,MAIN_MATCHUPS)
    print(f"SKIP - main tournament CSVs already exist in {OUT_DIR}")
    print(f"  Delete those files and re-run this cell to recompute.")

else:
    t0 = time.time()
    if USE_BATCH:
        # Batched RANDOM-vs-RANDOM tournament. Writes matchups.csv, summary.csv, tactic_matrix.csv.
        batch_engine.run_mode_batched(pool, mode="random", n_runs=N_RUNS_MAIN,
                                      output_dir=OUT_DIR, suffix="", base_seed=2026, verbose=True, n_workers=N_WORKERS_MAIN)
    else:
        tournament_vec.run_tournament_vec(
            pool, n_runs=N_RUNS_MAIN, output_dir=OUT_DIR,
            filename_suffix="", n_workers=N_WORKERS_MAIN, modes=("random",),
            verbose=True, print_every=max(1, len(pool)//20))
    print(f"\nMain tournament complete in {time.time()-t0:.0f}s")

C:\Users\Matt\OneDrive\Desktop\Game\Combatv3\lab_out\summary.csv C:\Users\Matt\OneDrive\Desktop\Game\Combatv3\lab_out\matchups.csv
SKIP - main tournament CSVs already exist in C:\Users\Matt\OneDrive\Desktop\Game\Combatv3\lab_out
  Delete those files and re-run this cell to recompute.


## 1B. Playstyle-aware tournament

Each loadout is assigned its theoretically-optimal playstyle by `assign_default_playstyle()`, then the tournament reruns with everyone using their assigned style instead of Random. This is the comparison set for evaluating playstyle quality.

In [4]:
PLAYSTYLE_SUMMARY = os.path.join(OUT_DIR, "summary_playstyle.csv")
PLAYSTYLE_MATCHUPS = os.path.join(OUT_DIR, "matchups_playstyle.csv")
PLAYSTYLE_TACTIC_MATRIX = os.path.join(OUT_DIR, "tactic_matrix_playstyle.csv")

if dont:
    pass
else:
    if os.path.exists(PLAYSTYLE_SUMMARY) and _have_matchups(PLAYSTYLE_MATCHUPS):
        print(f"SKIP - playstyle tournament CSVs already exist in {OUT_DIR}")
    else:
        # Assign defaults; build the playstyle pool (+ KT-twins for the within-KT playstyle isolation).
        style_pool = [ld._replace(playstyle=playstyles.assign_default_playstyle(ld)) for ld in pool]
        kt_twins = loadouts.kt_twins(pool)
        style_pool.extend(kt_twins)
        print(f"Added {len(kt_twins)} KT-twin loadouts (same KT, equipment-natural playstyle)")
        print(f"Playstyle distribution across {len(style_pool)} loadouts:")
        for ps, n in Counter(ld.playstyle for ld in style_pool).most_common():
            print(f"  {ps:<14} {n}")
        print()
    
        t0 = time.time()
        if USE_BATCH:
            batch_engine.run_mode_batched(style_pool, mode="playstyle", n_runs=N_RUNS_PLAYSTYLE,
                                          output_dir=OUT_DIR, suffix="_playstyle", base_seed=2026, verbose=True, n_workers=N_WORKERS_MAIN)
        else:
            tournament_vec.run_tournament_vec(
                style_pool, n_runs=N_RUNS_PLAYSTYLE, output_dir=OUT_DIR,
                filename_suffix="_playstyle", n_workers=N_WORKERS_MAIN, modes=("playstyle",),
                verbose=True, print_every=max(1, len(style_pool)//20))
        print(f"\nPlaystyle tournament complete in {time.time()-t0:.0f}s")


SKIP - playstyle tournament CSVs already exist in C:\Users\Matt\OneDrive\Desktop\Game\Combatv3\lab_out


## 1C. Forced-tactics matrix (stratified sample)

For a smaller stratified sample (~60 loadouts), force every (a_tactic, b_tactic) pair and measure win rates. This complements the empirical tactic matrix from 1A — same numbers, different angle (conditional on tactic *forced* vs *organically chosen*).

In [5]:
FORCED_TACTICS_CSV = os.path.join(OUT_DIR, "forced_tactics.csv")

if os.path.exists(FORCED_TACTICS_CSV):
    print(f"SKIP — forced tactics CSV already exists.")
else:
    # Stratified sample: 1-3 per (retinue, MPC) bucket
    by_bucket = defaultdict(list)
    for ld in pool:
        by_bucket[(ld.retinue, ld.military_pursuit_count)].append(ld)
    sample = []
    rng = random.Random(2026)
    for key, lds in by_bucket.items():
        sample.extend(rng.sample(lds, min(len(lds), 2)))
    if len(sample) > FORCED_SAMPLE_SIZE:
        sample = rng.sample(sample, FORCED_SAMPLE_SIZE)
    print(f"Forced-tactics sample: {len(sample)} loadouts")

    t0 = time.time()
    result = tactics_analysis.empirical_tactic_matrix(
        sample, n_runs=N_RUNS_FORCED, verbose=False, n_workers=N_WORKERS_MAIN)
    print(f"Forced-tactics sweep complete in {time.time()-t0:.0f}s")

    # Flatten into a long-form CSV
    rows = []
    for i, a_t in enumerate(TACTICS):
        for j, b_t in enumerate(TACTICS):
            rows.append({
                "a_tactic": a_t, "b_tactic": b_t,
                "win_rate": result["win_rate"].iloc[i, j],
                "survival": result["survival"].iloc[i, j],
                "skirm": result["skirm"].iloc[i, j],
                "indecisive_rate": result["indecisive"].iloc[i, j],
            })
    pd.DataFrame(rows).to_csv(FORCED_TACTICS_CSV, index=False)
    print(f"Wrote {FORCED_TACTICS_CSV}")

Forced-tactics sample: 40 loadouts
Forced-tactics sweep complete in 44s
Wrote C:\Users\Matt\OneDrive\Desktop\Game\Combatv3\lab_out\forced_tactics.csv


## 1D. Horde mode (multi-battle survival)

Each loadout fights `HORDE_BATTLES` consecutive battles with carry-over fatigue, Strain accumulation, and Apothecary heal between battles. Measures sustained performance, not just one-battle wins.

In [6]:
HORDE_CSV = os.path.join(OUT_DIR, "horde_survival.csv")
if os.path.exists(HORDE_CSV):
    print(f"SKIP — horde survival CSV already exists.")
else:
    try:
        import horde_mode
        importlib.reload(horde_mode)
    except ImportError:
        print("horde_mode.py not found — skipping. (Section 7 will be empty.)")
    else:
        rng = random.Random(2026)
        by_bucket = defaultdict(list)
        for ld in pool:
            by_bucket[(ld.retinue, ld.military_pursuit_count)].append(ld)
        sample = []
        for key, lds in by_bucket.items():
            sample.extend(rng.sample(lds, min(len(lds), 2)))
        if len(sample) > HORDE_SAMPLE_SIZE:
            sample = rng.sample(sample, HORDE_SAMPLE_SIZE)
        print(f"Horde sample: {len(sample)} loadouts × {HORDE_BATTLES} waves × {N_RUNS_HORDE} runs")
        t0 = time.time()
        rows = []
        for i, ld in enumerate(sample):
            survival = horde_mode.run_horde(
                ld, sample,
                max_waves=HORDE_BATTLES,
                n_runs=N_RUNS_HORDE,
                seed=2026+i*1009,
            )
            # Derive summary stats from the returned arrays
            wave_of_wipe = survival["wave_of_wipe"]            # per-run wave index of wipe (or max_waves+1)
            final_sizes  = survival["per_run_final_size"]      # per-run final size
            rows.append({
                "name": ld.name, "retinue": ld.retinue,
                "mpc": ld.military_pursuit_count, "domain_count": ld.domain_count,
                "battles_survived_mean":   float(np.mean(wave_of_wipe)),
                "battles_survived_median": float(np.median(wave_of_wipe)),
                "size_remaining_mean":     float(np.mean(final_sizes)),
            })
        pd.DataFrame(rows).to_csv(HORDE_CSV, index=False)
        print(f"Horde mode complete in {time.time()-t0:.0f}s — wrote {HORDE_CSV}")

Horde sample: 10 loadouts × 8 waves × 100 runs
Horde mode complete in 1s — wrote C:\Users\Matt\OneDrive\Desktop\Game\Combatv3\lab_out\horde_survival.csv


In [7]:
# Sanity check: confirm the CURRENT loadouts.py is loaded and the pool validates.
import loadouts, inspect
importlib.reload(loadouts)
print("FILE BEING IMPORTED:", loadouts.__file__)
print("has validate_loadout:", hasattr(loadouts, "validate_loadout"))
print("has master_effect_closure:", hasattr(loadouts, "master_effect_closure"))
print("Master Workshop mastery_tags:", loadouts.PURSUITS_INFO["Master Workshop"]["mastery_tags"], "(expect ['Rend'])")
print("Conditioning Field mastery_tags:", loadouts.PURSUITS_INFO["Conditioning Field"]["mastery_tags"], "(expect ['Cond Field'])")
import vectorized_combat as vc
vc.invalidate_tactic_tables()
_p = loadouts.balanced_validation_pool(4, 13, per_cell=None)
print("builds:", len(_p))
loadouts.validate_pool(_p)   # expect 0 invalid


FILE BEING IMPORTED: C:\Users\Matt\OneDrive\Desktop\Game\Combatv3\loadouts.py
has validate_loadout: True
has master_effect_closure: True
Master Workshop mastery_tags: ['Rend'] (expect ['Rend'])
Conditioning Field mastery_tags: ['Cond Field'] (expect ['Cond Field'])
builds: 34710
validate_pool: 34710 builds, 0 invalid (0.0%)
  ALL VALID ✓


{}

## 1E. Load all CSVs into DataFrames for analysis

All sections below read from these dataframes — no further computation needed.

In [ ]:
t0 = time.time()
# Main tournament summary (per-loadout aggregates) — small, load raw
df_main_summary = pd.read_csv(MAIN_SUMMARY)

# ── Memory control ──────────────────────────────────────────────────────────
# matchups.parquet is hundreds of MB → multiple GB in RAM. load_tournament_pruned reads ONLY the
# ~30 columns the analyses need, straight from parquet (the heavy a_pursuits/b_pursuits/_name/_tags
# columns never load), and does NO downcasting pass. Fast and low-memory.
LOAD_PS_MATCHUPS = False   # playstyle matchups DOUBLES RAM; load only if you need the KT-twin cell
df_main_matchups = (analysis.load_tournament_lean(MAIN_MATCHUPS)
                    if _have_matchups(MAIN_MATCHUPS) else None)
df_tactic_matrix = pd.read_csv(MAIN_TACTIC_MATRIX) if os.path.exists(MAIN_TACTIC_MATRIX) else None

# Playstyle tournament (summary always; matchups only if LOAD_PS_MATCHUPS)
df_ps_summary = pd.read_csv(PLAYSTYLE_SUMMARY) if os.path.exists(PLAYSTYLE_SUMMARY) else None
df_ps_matchups = (analysis.load_tournament_pruned(PLAYSTYLE_MATCHUPS)
                  if (LOAD_PS_MATCHUPS and _have_matchups(PLAYSTYLE_MATCHUPS)) else None)

# Forced tactics
df_forced_tactics = pd.read_csv(FORCED_TACTICS_CSV) if os.path.exists(FORCED_TACTICS_CSV) else None

# Horde
df_horde = pd.read_csv(HORDE_CSV) if os.path.exists(HORDE_CSV) else None

def _mem(df):
    return f"{df.shape} ({df.memory_usage(deep=True).sum()/1e6:.0f} MB)" if df is not None else "missing"
print(f"Main summary:    {df_main_summary.shape}")
print(f"Main matchups:   {_mem(df_main_matchups)}")
print(f"Tactic matrix:   {df_tactic_matrix.shape if df_tactic_matrix is not None else 'missing'}")
print(f"Playstyle summ:  {df_ps_summary.shape if df_ps_summary is not None else 'missing'}")
print(f"Playstyle match: {_mem(df_ps_matchups)}{'  (skipped: LOAD_PS_MATCHUPS=False)' if not LOAD_PS_MATCHUPS else ''}")
print(f"Forced tactics:  {df_forced_tactics.shape if df_forced_tactics is not None else 'missing'}")
print(f"Horde:           {df_horde.shape if df_horde is not None else 'missing'}")
print(f"Import complete in {time.time()-t0:.0f}s")

# 2. Pursuit & Domain Analysis

**Design target:** win rate should be tightly correlated to Military Pursuit Count (MPC). A player spending 13pts on military should beat a player spending 5pts. This section evaluates how cleanly that correlation holds, and whether Domain Count (the breadth-of-investment metric) tells the same story or a different one.

## 2.1 Correlation: Win Rate ↔ MPC

The headline number. Pearson and Spearman correlations of per-loadout win rate vs MPC.

In [ ]:
from scipy import stats as sp_stats

g = df_main_summary
pearson_r, pearson_p = sp_stats.pearsonr(g["military_pursuit_count"], g["win_rate"])
spearman_r, spearman_p = sp_stats.spearmanr(g["military_pursuit_count"], g["win_rate"])

print(f"=== Win Rate ↔ Military Pursuit Count ===")
print(f"  Pearson:  r = {pearson_r:+.3f}  (p = {pearson_p:.2e})")
print(f"  Spearman: r = {spearman_r:+.3f}  (p = {spearman_p:.2e})")
print()
print(f"Interpretation:")
print(f"  +1.0  perfect linear correlation: MPC fully predicts win rate.")
print(f"  +0.7+ strong: design target met.")
print(f"  +0.4-0.7 moderate: budget matters but other factors strong.")
print(f"  <+0.4 weak: budget barely matters — investigate why.")

In [ ]:
# Visualize: MPC vs Win Rate scatter + binned mean line
fig, ax = plt.subplots(figsize=(11, 6))
ax.scatter(g["military_pursuit_count"], g["win_rate"], alpha=0.15, s=20, color="steelblue")
binned = g.groupby("military_pursuit_count")["win_rate"].agg(["mean", "std", "count"])
ax.errorbar(binned.index, binned["mean"], yerr=binned["std"], fmt="o-", color="darkred",
            markersize=10, linewidth=2, capsize=4, label="Mean ± std per MPC")
ax.axhline(0.5, color="gray", linestyle=":", alpha=0.5, label="50% win rate")
ax.set_xlabel("Military Pursuit Count (MPC)")
ax.set_ylabel("Win Rate")
ax.set_title(f"Win Rate vs Military Pursuit Count  (Pearson r = {pearson_r:+.3f})")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 2.2 MPC bucket summary — does higher always win?

Test: for every loadout at MPC=N, what fraction of opponents at MPC<N do they beat? At MPC>N? Cross-budget head-to-head is the strict test of "spending more should win more."

In [ ]:
# Build cross-MPC win rate matrix from matchups
df_m = df_main_matchups
df_m["a_mpc_bucket"] = df_m["a_military_pursuit_count"]
df_m["b_mpc_bucket"] = df_m["b_military_pursuit_count"]
# Aggregate wins/losses per (a_mpc, b_mpc) pair
grp = df_m.groupby(["a_mpc_bucket", "b_mpc_bucket"]).agg(
    a_wins=("a_wins", "sum"), b_wins=("b_wins", "sum"),
    mut=("mut_wipe", "sum"), indec=("indecisive", "sum"),
).reset_index()
grp["total"] = grp[["a_wins","b_wins","mut","indec"]].sum(axis=1)
grp["a_win_rate"] = grp["a_wins"] / grp["total"]
grp["decisive_rate"] = (grp["a_wins"] + grp["b_wins"]) / grp["total"]

# Pivot: rows = A's MPC, cols = B's MPC, value = A's win rate
pivot_wr = grp.pivot(index="a_mpc_bucket", columns="b_mpc_bucket", values="a_win_rate")
pivot_dec = grp.pivot(index="a_mpc_bucket", columns="b_mpc_bucket", values="decisive_rate")

print("=== A's Win Rate by (A_MPC, B_MPC) — values show how often A beats B ===")
print(pivot_wr.round(3).to_string())
print()
print("Diagonal = mirror MPC. Above diagonal: A has more budget than B (should win).")

In [ ]:
# Heatmap
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

ax = axes[0]
im = ax.imshow(pivot_wr.values, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(len(pivot_wr.columns))); ax.set_xticklabels(pivot_wr.columns)
ax.set_yticks(range(len(pivot_wr.index))); ax.set_yticklabels(pivot_wr.index)
ax.set_xlabel("Opponent (B) MPC"); ax.set_ylabel("Player (A) MPC")
ax.set_title("A's Win Rate by MPC matchup")
plt.colorbar(im, ax=ax)
# Annotate
for i in range(len(pivot_wr.index)):
    for j in range(len(pivot_wr.columns)):
        v = pivot_wr.values[i, j]
        if not np.isnan(v):
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", color="black", fontsize=8)

ax = axes[1]
# Average win rate over LOWER vs HIGHER MPC opponents per row
rows = []
for a_mpc in sorted(pivot_wr.index):
    higher = pivot_wr.loc[a_mpc, pivot_wr.columns > a_mpc].dropna()
    lower  = pivot_wr.loc[a_mpc, pivot_wr.columns < a_mpc].dropna()
    same   = pivot_wr.loc[a_mpc, [a_mpc] if a_mpc in pivot_wr.columns else []]
    rows.append({
        "mpc": a_mpc,
        "vs_lower_mean": lower.mean() if len(lower) else np.nan,
        "vs_higher_mean": higher.mean() if len(higher) else np.nan,
        "vs_same_mean": same.mean() if len(same) else np.nan,
    })
df_vs = pd.DataFrame(rows)
ax.plot(df_vs["mpc"], df_vs["vs_lower_mean"], "o-", color="green", label="vs lower MPC")
ax.plot(df_vs["mpc"], df_vs["vs_same_mean"], "s--", color="gray", label="vs same MPC")
ax.plot(df_vs["mpc"], df_vs["vs_higher_mean"], "^-", color="red", label="vs higher MPC")
ax.axhline(0.5, color="black", linestyle=":", alpha=0.5)
ax.set_xlabel("Player's MPC"); ax.set_ylabel("Win rate")
ax.set_title("Performance by opponent's budget bracket")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print("\nKey check: does the GREEN line stay above 0.5 (you beat lower-MPC opponents)?")
print("Does the RED line stay below 0.5 (you lose to higher-MPC opponents)?")

## 2.1b Correlation: Win Rate ↔ Total Investment (action economy)

`total_investment` counts **every** pursuit a build owns (each = 1), with NO *Efficient X* discount —
it measures the action economy (how many pursuit-decisions were spent), whereas MPC measures
settlement space (and IS reduced by satisfied *Efficient X*). The hypothesis: total_investment is a
*more monotonic* predictor of win rate than MPC, because MPC's space-discount lets deep-teched gear
look artificially cheap. This cell compares the two head-to-head.

In [ ]:
from scipy import stats as sp_stats
import numpy as np

# total_investment = raw count of pursuits (every pursuit = 1; Efficient-X does NOT reduce it).
# Derived from the "pursuits" column ("|"-joined). MPC is military_pursuit_count (space, discounted).
gi = df_main_summary.copy()
gi["total_investment"] = gi["pursuits"].fillna("").map(
    lambda s: len([x for x in s.split("|") if x]) if s else 0)

ti_r,  ti_p  = sp_stats.pearsonr(gi["total_investment"], gi["win_rate"])
ti_sr, ti_sp = sp_stats.spearmanr(gi["total_investment"], gi["win_rate"])
mpc_r,  _    = sp_stats.pearsonr(gi["military_pursuit_count"], gi["win_rate"])
mpc_sr, _    = sp_stats.spearmanr(gi["military_pursuit_count"], gi["win_rate"])

print("=== Win Rate ↔ Total Investment  vs  ↔ MPC ===")
print(f"  total_investment : Pearson {ti_r:+.3f} (p={ti_p:.2e}) | Spearman {ti_sr:+.3f}")
print(f"  MPC (space)      : Pearson {mpc_r:+.3f}              | Spearman {mpc_sr:+.3f}")
print()
print(f"  Spearman is the monotonicity measure (rank correlation). Higher = more monotonic.")
print(f"  total_investment Spearman {ti_sr:+.3f}  vs  MPC Spearman {mpc_sr:+.3f}"
      f"  -> {'total_investment MORE monotonic' if ti_sr>mpc_sr else 'MPC more monotonic'}")
print()

# How far apart MPC and total_investment run (the Efficient-X gap)
gap = (gi["total_investment"] - gi["military_pursuit_count"])
print(f"  total_investment − MPC gap: mean {gap.mean():.2f}, max {gap.max()} "
      f"(0 = no efficiency discount; larger = deeper Efficient-X stacking)")


In [ ]:
# Visualize: Total Investment vs Win Rate scatter + binned mean line (compare to the MPC plot)
fig, ax = plt.subplots(figsize=(11, 6))
ax.scatter(gi["total_investment"], gi["win_rate"], alpha=0.15, s=20, color="seagreen")
binned = gi.groupby("total_investment")["win_rate"].agg(["mean", "std", "count"])
ax.errorbar(binned.index, binned["mean"], yerr=binned["std"], fmt="o-", color="darkred",
            markersize=9, linewidth=2, capsize=4, label="Mean ± std per investment")
ax.axhline(0.5, color="gray", linestyle=":", alpha=0.5, label="50% win rate")
ax.set_xlabel("Total Investment (every pursuit = 1, no Efficient-X discount)")
ax.set_ylabel("Win Rate")
ax.set_title(f"Win Rate vs Total Investment  (Pearson r = {ti_r:+.3f}, Spearman = {ti_sr:+.3f})")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

# Side-by-side per-bucket means: does investment climb more smoothly than MPC?
inv_means = gi.groupby("total_investment")["win_rate"].mean()
mpc_means = gi.groupby("military_pursuit_count")["win_rate"].mean()
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(mpc_means.index, mpc_means.values, "o-", color="steelblue", label="by MPC (space)")
ax.plot(inv_means.index, inv_means.values, "s-", color="seagreen", label="by total_investment (actions)")
ax.axhline(0.5, color="gray", linestyle=":", alpha=0.5)
ax.set_xlabel("Count"); ax.set_ylabel("Mean win rate")
ax.set_title("Monotonicity check: mean win rate vs MPC vs total_investment")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


## 2.3 Can the 30-DomainCount player lose to the 16-DomainCount player?

Same question for Domain Count: does broader domain investment correlate with winning, and can a heavily-invested player be beaten by a lean one?

In [ ]:
pearson_d, _ = sp_stats.pearsonr(g["domain_count"], g["win_rate"])
spearman_d, _ = sp_stats.spearmanr(g["domain_count"], g["win_rate"])
print(f"=== Win Rate ↔ Domain Count ===")
print(f"  Pearson:  r = {pearson_d:+.3f}")
print(f"  Spearman: r = {spearman_d:+.3f}")

# MPC↔DC are highly correlated by construction. Partial: residual win rate AFTER removing MPC effect.
from sklearn.linear_model import LinearRegression
X_mpc = g[["military_pursuit_count"]].values
y = g["win_rate"].values
mpc_model = LinearRegression().fit(X_mpc, y)
y_residual = y - mpc_model.predict(X_mpc)
# Now correlate residual with DC
res_pearson, _ = sp_stats.pearsonr(g["domain_count"], y_residual)
print(f"\nWin Rate residual (after MPC removed) ↔ Domain Count:")
print(f"  Pearson:  r = {res_pearson:+.3f}")
print(f"  → If close to 0, DC doesn't add independent predictive value beyond MPC.")
print(f"  → If positive, broader domain investment HELPS independently of military spend.")
print(f"  → If negative, broader investment HURTS (suggests pursuit redundancy / wasted breadth).")

In [ ]:
# Specific test: how often does a high-DC loader lose to a low-DC loader?
# Build cross-DC matchup table
df_m["a_dc_bucket"] = (df_m["a_domain_count"] // 5) * 5  # 5-pt buckets
df_m["b_dc_bucket"] = (df_m["b_domain_count"] // 5) * 5
grp_dc = df_m.groupby(["a_dc_bucket", "b_dc_bucket"]).agg(
    a_wins=("a_wins","sum"), b_wins=("b_wins","sum"),
    mut=("mut_wipe","sum"), indec=("indecisive","sum")).reset_index()
grp_dc["total"] = grp_dc[["a_wins","b_wins","mut","indec"]].sum(axis=1)
grp_dc["a_win_rate"] = grp_dc["a_wins"] / grp_dc["total"]
pivot_dc = grp_dc.pivot(index="a_dc_bucket", columns="b_dc_bucket", values="a_win_rate")

fig, ax = plt.subplots(figsize=(10, 7))
im = ax.imshow(pivot_dc.values, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(len(pivot_dc.columns))); ax.set_xticklabels(pivot_dc.columns)
ax.set_yticks(range(len(pivot_dc.index))); ax.set_yticklabels(pivot_dc.index)
ax.set_xlabel("Opponent (B) Domain Count bucket")
ax.set_ylabel("Player (A) Domain Count bucket")
ax.set_title("A's Win Rate by Domain Count matchup")
plt.colorbar(im, ax=ax)
for i in range(len(pivot_dc.index)):
    for j in range(len(pivot_dc.columns)):
        v = pivot_dc.values[i, j]
        if not np.isnan(v):
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", color="black", fontsize=8)
plt.tight_layout(); plt.show()

## 2.4 Outliers — punch above weight on MPC

Loadouts whose win rate is most above what their MPC bucket would predict — "value loadouts." Conversely, loadouts that under-perform their budget.

In [ ]:
over, under = analysis.mpc_outliers(df_main_matchups, top_n=15)
print("=== Top punch-above-weight loadouts (most over-perform their MPC peers) ===")
print(over[["retinue","military_pursuit_count","win_rate","bucket_mean_wr","z_score"]].round(3).to_string())
print("\n=== Top under-performers (waste budget) ===")
print(under[["retinue","military_pursuit_count","win_rate","bucket_mean_wr","z_score"]].round(3).to_string())

# 3. Retinue Analysis

How does win rate, durability, kill efficiency, and stalemate rate vary across the four retinues? Are retinue cost-efficiency ratios appropriate?

In [ ]:
print("=== Per-Retinue summary (all loadouts, Random playstyle) ===")
ret_summary = df_main_summary.groupby("retinue").agg(
    n_loadouts=("name", "count"),
    win_rate=("win_rate", "mean"),
    loss_rate=("loss_rate", "mean"),
    decisive_win_rate=("decisive_win_rate", "mean"),
    kill_eff=("kill_efficiency", "mean"),
    survivors=("avg_self_survivors", "mean"),
    opp_survivors=("avg_opp_survivors", "mean"),
    wins_per_1k_upkeep=("wins_per_1000_upkeep", "mean"),
    upkeep=("upkeep_per_retinue", "mean"),
)
print(ret_summary.round(3).to_string())

In [ ]:
# Win rate by retinue, faceted by MPC
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
for ret, color in zip(["Levy","Man-at-Arms","Sergeant","Knight Templar"],
                       ["#888","#4a90e2","#d4a800","#c43838"]):
    sub = df_main_summary[df_main_summary["retinue"] == ret]
    binned = sub.groupby("military_pursuit_count")["win_rate"].agg(["mean","std","count"])
    binned = binned[binned["count"] >= 3]
    if len(binned) > 0:
        ax.errorbar(binned.index, binned["mean"], yerr=binned["std"],
                    fmt="o-", color=color, label=ret, capsize=3, markersize=8)
ax.axhline(0.5, color="gray", linestyle=":", alpha=0.5)
ax.set_xlabel("Military Pursuit Count")
ax.set_ylabel("Win Rate")
ax.set_title("Win Rate by MPC × Retinue")
ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
ret_means = ret_summary["win_rate"].sort_values(ascending=False)
colors = {"Levy":"#888","Man-at-Arms":"#4a90e2","Sergeant":"#d4a800","Knight Templar":"#c43838"}
ax.bar(ret_means.index, ret_means.values, color=[colors[r] for r in ret_means.index])
ax.axhline(0.5, color="gray", linestyle=":", alpha=0.5)
ax.set_ylabel("Mean Win Rate")
ax.set_title("Mean Win Rate by Retinue")
ax.grid(alpha=0.3, axis="y")
plt.tight_layout(); plt.show()

In [ ]:
# ── 4×4 retinue matrix: win rate + kills by CAUSE (inflicted vs suffered) ──
# Each cell = ROW retinue (as A) vs COLUMN retinue (as B).
#   win_rate  : row's win rate vs column
#   INFLICTED : avg of the COLUMN's soldiers the row kills, split combat / shake / rout
#   SUFFERED  : avg of the ROW's own soldiers lost,        split combat / shake / rout
# Mirror cells (diagonal) shown for reference. Values are mean per-battle casualty COUNTS, not outcomes.
import numpy as np, pandas as pd, matplotlib.pyplot as plt

RETS = ["Levy", "Man-at-Arms", "Sergeant", "Knight Templar"]
df = df_main_matchups   # <-- your loaded tournament

g = (df.groupby(["a_retinue", "b_retinue"])
       .agg(win_rate   =("a_win_rate", "mean"),
            infl_combat=("avg_b_killed_combat", "mean"),
            infl_shake =("avg_b_killed_shake",  "mean"),
            infl_rout  =("avg_b_killed_rout",   "mean"),
            suff_combat=("avg_a_killed_combat", "mean"),
            suff_shake =("avg_a_killed_shake",  "mean"),
            suff_rout  =("avg_a_killed_rout",   "mean"),
            n          =("a_win_rate", "size"))
       .reset_index())
G = g.set_index(["a_retinue", "b_retinue"])

# numeric matrices for the heatmap backdrop (win rate) + net combat performance
win = pd.DataFrame(index=RETS, columns=RETS, dtype=float)
for r in RETS:
    for c in RETS:
        win.loc[r, c] = G.loc[(r, c), "win_rate"] if (r, c) in G.index else np.nan

# ---- visual grid: win-rate heatmap with inflicted/suffered text in each cell ----
fig, ax = plt.subplots(figsize=(13, 11))
im = ax.imshow(win.values.astype(float), cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(4)); ax.set_xticklabels(RETS, fontsize=10)
ax.set_yticks(range(4)); ax.set_yticklabels(RETS, fontsize=10)
ax.set_xlabel("opponent (column = B)", fontsize=11)
ax.set_ylabel("retinue (row = A)", fontsize=11)
ax.set_title("Retinue matchup matrix — win rate (color) + kills by cause\n"
             "INFLICTED on opponent  /  SUFFERED  (combat · shake · rout)", fontsize=12)
for i, r in enumerate(RETS):
    for j, c in enumerate(RETS):
        if (r, c) not in G.index:
            ax.text(j, i, "—", ha="center", va="center", color="#666"); continue
        row = G.loc[(r, c)]
        wr = row["win_rate"]
        diag = " (mirror)" if r == c else ""
        txt = (f"WR {wr*100:.0f}%{diag}\n"
               f"inflict  {row['infl_combat']:.1f}·{row['infl_shake']:.1f}·{row['infl_rout']:.1f}\n"
               f"suffer   {row['suff_combat']:.1f}·{row['suff_shake']:.1f}·{row['suff_rout']:.1f}")
        ax.text(j, i, txt, ha="center", va="center", fontsize=8.5,
                color="black" if 0.25 < wr < 0.8 else "white")
fig.colorbar(im, ax=ax, label="row win rate", fraction=0.046, pad=0.04)
plt.tight_layout(); plt.show()

# ---- the full table (off-diagonal only, for sorting/inspection) ----
tbl = g[g.a_retinue != g.b_retinue].copy()
tbl["win%"] = (tbl.win_rate*100).round(0)
tbl = tbl[["a_retinue","b_retinue","win%",
           "infl_combat","infl_shake","infl_rout",
           "suff_combat","suff_shake","suff_rout","n"]].round(2)
display(tbl.sort_values(["a_retinue","b_retinue"]).reset_index(drop=True))

## 3.1 Retinue cost efficiency

Wins per 1000 gold upkeep — does spending more on better troops pay off?

In [ ]:
ret_costs = df_main_summary.groupby("retinue").agg(
    army_upkeep=("army_upkeep", "first"),
    wins_per_1k=("wins_per_1000_upkeep", "mean"),
    win_rate=("win_rate", "mean"),
).round(2)
ret_costs["upkeep_25_men"] = ret_costs["army_upkeep"]
print("=== Retinue cost-efficiency ===")
print(ret_costs.to_string())

# 4. Equipment Analysis

Weapons, shields, armor, and 1H/2H/Bastard tradeoffs.

## 4.1 Weapon performance

In [ ]:
df = df_main_summary.copy()
print("=== Melee weapon performance (≥5 loadouts) ===")
m = df[df["weapon"].notna() & (df["weapon"] != "")].groupby("weapon").agg(
    n_loadouts=("name","count"), win_rate=("win_rate","mean"),
    decisive_win_rate=("decisive_win_rate","mean"), kill_eff=("kill_efficiency","mean"),
).sort_values("win_rate", ascending=False)
print(m[m["n_loadouts"]>=5].round(3).to_string())

print("\n=== Ranged weapon performance (≥5 loadouts) ===")
r = df[df["ranged"].notna() & (df["ranged"] != "")].groupby("ranged").agg(
    n_loadouts=("name","count"), win_rate=("win_rate","mean"),
    decisive_win_rate=("decisive_win_rate","mean"), kill_eff=("kill_efficiency","mean"),
).sort_values("win_rate", ascending=False)
print(r[r["n_loadouts"]>=5].round(3).to_string())

## 4.2 Shield performance

In [ ]:
sh = df_main_summary.copy()
sh["shield"] = sh["shield"].fillna("None")
sh = sh.groupby("shield").agg(
    n=("name","count"),
    win_rate=("win_rate","mean"),
    decisive_win_rate=("decisive_win_rate","mean"),
    kill_eff=("kill_efficiency","mean"),
).sort_values("win_rate", ascending=False)
print("=== Shield performance ===")
print(sh.round(3).to_string())

## 4.3 Armor performance

In [ ]:
arm = df_main_summary.groupby("armor").agg(
    n=("name","count"),
    win_rate=("win_rate","mean"),
    decisive_win_rate=("decisive_win_rate","mean"),
    survivors=("avg_self_survivors","mean"),
).sort_values("win_rate", ascending=False)
print("=== Armor performance ===")
print(arm.round(3).to_string())

In [ ]:
# Diagnose what's actually in df_main_summary
print("ARMORS present:", df_main_summary["armor"].value_counts().to_dict())
print("\nRETINUES present:", df_main_summary["retinue"].value_counts().to_dict())
print("\nMPC range:", df_main_summary["military_pursuit_count"].min(), "-", df_main_summary["military_pursuit_count"].max())
print("Total builds:", len(df_main_summary))
# Cross: which retinues have which armors (shows if tier floors are excluding light armor)
print("\nArmor x Retinue:")
print(df_main_summary.groupby(["retinue","armor"]).size().unstack(fill_value=0))

## 4.4 Bastard Sword — does the dual-profile actually help?

Compare Bastard 1H+shield builds vs Bastard 2H builds vs other 2H weapons (Poleaxe, Halberd, etc.).

In [ ]:
bast = df_main_summary[df_main_summary["weapon"] == "Bastard Sword"].copy()
bast["mode"] = np.where(bast["shield"].fillna("None") == "None", "Bastard 2H", "Bastard 1H+shield")
b_summary = bast.groupby("mode").agg(
    n=("name","count"),
    win_rate=("win_rate","mean"),
    decisive_win_rate=("decisive_win_rate","mean"),
    kill_eff=("kill_efficiency","mean"),
).round(3)
print("=== Bastard Sword variants ===")
print(b_summary.to_string())

# Compare to other heavy weapons
print()
print("=== Other heavy 2H weapons for reference ===")
other = df_main_summary[df_main_summary["weapon"].isin(["Poleaxe","Halberd","War Hammer","Battle Axe","2HBastard"])]
print(other.groupby("weapon").agg(
    n=("name","count"),
    win_rate=("win_rate","mean"),
    decisive_win_rate=("decisive_win_rate","mean"),
).round(3).to_string())

## 4.5 Lance — does Charge synergy pay off?

In [ ]:
lance = df_main_summary[df_main_summary["weapon"] == "Lance"]
non_lance = df_main_summary[df_main_summary["weapon"] != "Lance"]
print(f"Lance loadouts:  n={len(lance):3d}  win_rate={lance['win_rate'].mean():.3f}  decisive_wr={lance['decisive_win_rate'].mean():.3f}")
print(f"Non-Lance:       n={len(non_lance):3d}  win_rate={non_lance['win_rate'].mean():.3f}  decisive_wr={non_lance['decisive_win_rate'].mean():.3f}")
# Best lance shields
if len(lance) > 0:
    print("\nLance shield breakdown:")
    print(lance.groupby("shield").agg(n=("name","count"), wr=("win_rate","mean")).round(3).to_string())

## 4.6 Shield deep-dive

A focused analysis of shield builds across all metrics. Shield types map to gear tiers (Wooden=Crude, Kite=Cast, Scutum=Wrought, Tower=Forged, Heater=Crafted) per the loadout rules — so this also doubles as a tier-by-tier comparison of shielded builds.

### 4.6.1 Win rate by shield type

In [ ]:
# Filter to shield-bearing matchup rows (A side)
shield_df = df_main_matchups[df_main_matchups['a_shield'].notna() & (df_main_matchups['a_shield'] != '')].copy()
shield_df['n_battles'] = shield_df['a_wins'] + shield_df['b_wins'] + shield_df['mut_wipe'] + shield_df['indecisive']
print(f'Shield-bearing matchup rows: {len(shield_df):,} (of {len(df_main_matchups):,} total)')

# Per-loadout aggregation
shield_winrate = shield_df.groupby(['a_name', 'a_shield', 'a_retinue', 'a_weapon', 'a_armor']).agg(
    n_battles=('n_battles', 'sum'),
    wins=('a_wins', 'sum'),
    losses=('b_wins', 'sum'),
    mutual=('mut_wipe', 'sum'),
).reset_index()
shield_winrate['win_rate'] = shield_winrate['wins'] / shield_winrate['n_battles']
shield_winrate['decisive_win_rate'] = shield_winrate['wins'] / shield_winrate[['wins','losses']].sum(axis=1).clip(lower=1)
shield_winrate = shield_winrate.sort_values('win_rate', ascending=False)

print('\n=== Win rate by shield type (aggregate) ===')
by_type = shield_winrate.groupby('a_shield').agg(
    n_loadouts=('a_name', 'count'),
    avg_win_rate=('win_rate', 'mean'),
    avg_decisive_wr=('decisive_win_rate', 'mean'),
).sort_values('avg_win_rate', ascending=False)
print(by_type.round(3).to_string())

### 4.6.2 Durability — survival rates by shield

In [ ]:
# Use analysis.durability and join shield info back in
dur = analysis.durability(df_main_matchups)
shield_lookup = df_main_matchups.groupby('a_name')['a_shield'].first().to_dict()
dur['shield'] = dur['a_name'].map(shield_lookup).fillna('None')

dur_by_shield = dur.groupby('shield').agg(
    n_loadouts=('a_name', 'count'),
    avg_survival_rate=('avg_survival_rate', 'mean'),
).sort_values('avg_survival_rate', ascending=False)
print('=== Survival rate by shield type ===')
print(dur_by_shield.round(3).to_string())

print('\n=== Top 10 most durable shield builds ===')
top_shield_dur = dur[dur['shield'] != 'None'].nlargest(10, 'avg_survival_rate')[
    ['a_name','shield','avg_survival_rate']]
print(top_shield_dur.round(3).to_string(index=False))

### 4.6.3 Shield destruction rate

How often does each shield get smashed mid-battle? Wooden + Cast tier shields are most fragile; Heater the toughest.

In [ ]:
# Drop mirror matchups
sd_df = shield_df[shield_df['a_name'] != shield_df['b_name']].copy()
destroy_by_type = sd_df.groupby('a_shield').agg(
    n_matchups=('a_shield', 'count'),
    avg_destroy_pct=('a_shield_destroyed_rate', lambda x: x.mean() * 100),
).sort_values('avg_destroy_pct', ascending=False)
print('=== Avg shield destruction rate by type ===')
print(destroy_by_type.round(1).to_string())

# Top 10 most-destroyed shield builds
print('\n=== Top 10 most-destroyed shield builds ===')
top_destroyed = sd_df.groupby(['a_name', 'a_shield']).agg(
    avg_destroy_pct=('a_shield_destroyed_rate', lambda x: x.mean() * 100),
).reset_index().sort_values('avg_destroy_pct', ascending=False).head(10)
print(top_destroyed.round(1).to_string(index=False))

### 4.6.4 What counters shield builds?

For each shield type, which weapons/loadouts beat them most reliably?

In [ ]:
def top_counters_for_shield(shield_name, top_n=8):
    rows = df_main_matchups[
        (df_main_matchups['a_shield'] == shield_name) &
        (df_main_matchups['a_name'] != df_main_matchups['b_name'])
    ].copy()
    if len(rows) == 0:
        return None
    rows['n_battles'] = rows['a_wins'] + rows['b_wins'] + rows['mut_wipe'] + rows['indecisive']
    rows['a_winrate'] = rows['a_wins'] / rows['n_battles']
    rows['b_winrate'] = rows['b_wins'] / rows['n_battles']
    counters = rows.groupby(['b_weapon', 'b_retinue', 'b_shield']).agg(
        b_wr=('b_winrate', 'mean'),
        a_wr=('a_winrate', 'mean'),
        n=('a_name', 'count'),
    ).reset_index()
    counters['margin'] = counters['b_wr'] - counters['a_wr']
    counters = counters[counters['n'] >= 5].sort_values('margin', ascending=False).head(top_n)
    return counters

for shield_type in ['Wooden Shield','Kite Shield','Scutum Shield','Tower Shield','Heater Shield']:
    c = top_counters_for_shield(shield_type)
    if c is not None and len(c) > 0:
        print(f'\n=== Top counters for {shield_type} ===')
        print(c.round(3).to_string(index=False))

### 4.6.5 What do shield builds dominate?

Inverse — for each shield type, what loadouts do they beat reliably?

In [ ]:
def shield_matchup_grid(shield_name, min_n=5):
    """Full attacker-retinue × defender-retinue favorable-matchup grid for a shield.

    For each (attacker retinue, defender retinue) cell, shows the shield's mean win rate
    and the top favorable weapon matchups within that cell — so you can read e.g.
    'Scutum-KT vs Sergeant-tier opponents' without Levy noise on either side.
    """
    rows = df_main_matchups[
        (df_main_matchups['a_shield'] == shield_name) &
        (df_main_matchups['a_name'] != df_main_matchups['b_name'])
    ].copy()
    if len(rows) == 0:
        print(f"\n(no data for {shield_name})")
        return
    rows['n_battles'] = rows['a_wins'] + rows['b_wins'] + rows['mut_wipe'] + rows['indecisive']
    rows['a_winrate'] = rows['a_wins'] / rows['n_battles']
    rows['b_winrate'] = rows['b_wins'] / rows['n_battles']

    RETS = ['Levy', 'Man-at-Arms', 'Sergeant', 'Knight Templar']

    # ── Summary grid: mean win rate per (attacker ret × defender ret) ──
    grid = rows.groupby(['a_retinue', 'b_retinue']).agg(
        a_wr=('a_winrate', 'mean'), n=('a_name', 'count')
    ).reset_index()
    pivot = grid.pivot(index='a_retinue', columns='b_retinue', values='a_wr')
    pivot = pivot.reindex(index=[r for r in RETS if r in pivot.index],
                          columns=[r for r in RETS if r in pivot.columns])
    print(f"\n{'='*70}\n{shield_name}: mean win rate by attacker retinue (rows) × defender retinue (cols)\n{'='*70}")
    print(pivot.round(3).to_string())

    # ── Detailed: top favorable weapon matchups per attacker×defender cell ──
    for a_ret in RETS:
        for b_ret in RETS:
            sub = rows[(rows['a_retinue'] == a_ret) & (rows['b_retinue'] == b_ret)]
            if len(sub) == 0:
                continue
            targets = sub.groupby(['b_weapon', 'b_shield']).agg(
                a_wr=('a_winrate', 'mean'),
                b_wr=('b_winrate', 'mean'),
                n=('a_name', 'count'),
            ).reset_index()
            targets['margin'] = targets['a_wr'] - targets['b_wr']
            targets = targets[targets['n'] >= min_n].sort_values('margin', ascending=False).head(4)
            if len(targets) == 0:
                continue
            print(f'\n  {shield_name} · {a_ret} vs {b_ret}-tier opponents:')
            print(targets.round(3).to_string(index=False).replace('\n', '\n    '))


for shield_type in ['Wooden Shield', 'Kite Shield', 'Scutum Shield', 'Tower Shield', 'Heater Shield']:
    shield_matchup_grid(shield_type)

### 4.6.6 Best tactics for shield builds

For one representative loadout per shield type, what's the best opening tactic?

In [ ]:
# Pick the top-winrate loadout for each shield type from shield_winrate
representatives = []
for shield_type in ['Wooden Shield', 'Kite Shield', 'Scutum Shield', 'Tower Shield', 'Heater Shield']:
    matches = shield_winrate[shield_winrate['a_shield'] == shield_type]
    if len(matches) > 0:
        representatives.append(matches.iloc[0]['a_name'])

# Use the empirical tactic matrix (loaded in Section 1E) as a proxy for tactic guidance.
# For per-loadout tactic profile, see Section 5.
if df_tactic_matrix is not None:
    _tac_pivot = df_tactic_matrix.pivot(index='a_tactic', columns='b_tactic', values='a_win_rate')
    _tac_pivot = _tac_pivot.reindex(index=TACTICS, columns=TACTICS)
    marginal_wr = _tac_pivot.mean(axis=1).sort_values(ascending=False)

    print('Marginal tactic win rates (best openings against a uniform-tactic opponent):')
    for t, wr in marginal_wr.items():
        print(f'  {t:<22} {wr:.3f}')

    print(f'\nRepresentative shield builds (best win-rate loadout per shield type):')
    for name in representatives:
        print(f'  {name}')
else:
    print('Tactic matrix not loaded — skip')

### 4.6.7 Shield summary

In [ ]:
print('=== Shield Findings Summary ===')
print()
print('1. Win rate by shield (mean across loadouts):')
print(by_type[['n_loadouts','avg_win_rate','avg_decisive_wr']].round(3).to_string())
print()
print('2. Durability ranking:')
print(dur_by_shield.round(3).to_string())
print()
print('3. Destruction frequency (lower = tougher shield):')
print(destroy_by_type.round(1).to_string())

## 4.7 Bastard Sword deep-dive

The dual-profile weapon: 1H mode (with shield) provides Shatter Armor + Steady; 2H mode (no shield, OR shield destroyed) provides Cleave + Unwieldy. Adaptive: a Bastard+shield build automatically switches to 2H mode if the shield breaks. How does this adaptive mechanic perform in practice?

### 4.7.1 Overall comparison — Bastard 1H vs 2H vs other

In [ ]:
bastard_df = df_main_matchups[df_main_matchups['a_weapon'] == 'Bastard Sword'].copy()
bastard_df['n_battles'] = bastard_df['a_wins'] + bastard_df['b_wins'] + bastard_df['mut_wipe'] + bastard_df['indecisive']
bastard_df['mode'] = bastard_df['a_shield'].apply(
    lambda s: '2H (no shield)' if (pd.isna(s) or s == '') else f'1H + {s}')
print(f'Bastard Sword matchup rows: {len(bastard_df):,}\n')

mode_winrate = bastard_df.groupby(['a_name', 'mode', 'a_retinue', 'a_armor']).agg(
    n_battles=('n_battles', 'sum'),
    wins=('a_wins', 'sum'),
    losses=('b_wins', 'sum'),
).reset_index()
mode_winrate['win_rate'] = mode_winrate['wins'] / mode_winrate['n_battles']
mode_winrate['decisive_wr'] = mode_winrate['wins'] / mode_winrate[['wins','losses']].sum(axis=1).clip(lower=1)
mode_winrate = mode_winrate.sort_values('win_rate', ascending=False)

print('=== Bastard variants — mean win rate by mode ===')
by_mode = mode_winrate.groupby('mode').agg(
    n_loadouts=('a_name', 'count'),
    avg_win_rate=('win_rate', 'mean'),
    avg_decisive_wr=('decisive_wr', 'mean'),
).sort_values('avg_win_rate', ascending=False)
print(by_mode.round(3).to_string())
print('\n=== Top 10 Bastard builds (any mode) ===')
print(mode_winrate.head(10)[['a_name','mode','a_retinue','win_rate','decisive_wr']].round(3).to_string(index=False))

### 4.7.2 Adaptive value — does mode-switching pay off?

How often does a Bastard+shield build have its shield destroyed mid-battle, forcing the 2H mode switch? Pair that with win rate to see whether the adaptive mechanic is actually delivering value.

### 4.7.3 Counters for each Bastard mode

In [ ]:
def bastard_counters(filter_func, label, n=8):
    rows = df_main_matchups[filter_func(df_main_matchups) &
                            (df_main_matchups['a_name'] != df_main_matchups['b_name'])].copy()
    if len(rows) == 0:
        return
    rows['n_battles'] = rows['a_wins'] + rows['b_wins'] + rows['mut_wipe'] + rows['indecisive']
    rows['a_wr'] = rows['a_wins'] / rows['n_battles']
    rows['b_wr'] = rows['b_wins'] / rows['n_battles']
    counters = rows.groupby(['b_weapon', 'b_retinue', 'b_shield']).agg(
        b_wr=('b_wr', 'mean'),
        a_wr=('a_wr', 'mean'),
        n_pairs=('a_name', 'count'),
    ).reset_index()
    counters['margin'] = counters['b_wr'] - counters['a_wr']
    counters = counters[counters['n_pairs'] >= 5].sort_values('margin', ascending=False).head(n)
    print(f'\n=== Top counters for {label} ===')
    print(counters.round(3).to_string(index=False))

# 1H mode (has shield)
bastard_counters(
    lambda d: (d['a_weapon'] == 'Bastard Sword') & d['a_shield'].notna() & (d['a_shield'] != ''),
    'Bastard 1H+shield')
# 2H mode (no shield)
bastard_counters(
    lambda d: (d['a_weapon'] == 'Bastard Sword') & ((d['a_shield'].isna()) | (d['a_shield'] == '')),
    'Bastard 2H')

### 4.7.4 Best shield pairing for Bastard 1H

In [ ]:
bastard_1h = bastard_df[(bastard_df['a_shield'].notna()) & (bastard_df['a_shield'] != '')].copy()
pairing = bastard_1h.groupby(['a_shield', 'a_retinue', 'a_armor']).agg(
    n_loadouts=('a_name', 'nunique'),
    n_battles=('n_battles', 'sum'),
    wins=('a_wins', 'sum'),
    avg_destroy_pct=('a_shield_destroyed_rate', lambda x: x.mean() * 100),
).reset_index()
pairing['win_rate'] = pairing['wins'] / pairing['n_battles']
pairing = pairing.sort_values('win_rate', ascending=False)
print('=== Bastard 1H × shield × armor combos (top 15 by win rate) ===')
print(pairing.head(15).round(3).to_string(index=False))

### 4.7.5 Bastard vs other top weapons

In [ ]:
focus_weapons = ['Bastard Sword', 'Poleaxe', 'Lance', 'Halberd', 'Pike', 'Battle Axe', 'Morningstar', 'War Hammer',"2HBastard"]
focus = df_main_matchups[
    df_main_matchups['a_weapon'].isin(focus_weapons) &
    (df_main_matchups['a_name'] != df_main_matchups['b_name'])
].copy()
focus['n_battles'] = focus['a_wins'] + focus['b_wins'] + focus['mut_wipe'] + focus['indecisive']

def to_mode(row):
    if row['a_weapon'] == 'Bastard Sword':
        return 'Bastard 1H' if pd.notna(row['a_shield']) and row['a_shield'] != '' else 'Bastard 2H'
    return row['a_weapon']
focus['mode'] = focus.apply(to_mode, axis=1)

comparison = focus.groupby(['mode', 'a_retinue']).agg(
    n_loadouts=('a_name', 'nunique'),
    n_battles=('n_battles', 'sum'),
    wins=('a_wins', 'sum'),
    losses=('b_wins', 'sum'),
).reset_index()
comparison['win_rate'] = comparison['wins'] / comparison['n_battles']
comparison['decisive_wr'] = comparison['wins'] / comparison[['wins','losses']].sum(axis=1).clip(lower=1)
comparison = comparison.sort_values(['a_retinue', 'win_rate'], ascending=[True, False])
print('=== Heavy weapons by retinue ===')
print(comparison[['a_retinue','mode','n_loadouts','win_rate','decisive_wr']].round(3).to_string(index=False))

### 4.7.6 Bastard summary

In [ ]:
print('=== Bastard Sword findings ===\n')
print('1. Mode comparison:')
print(by_mode.round(3).to_string())
print()

# 5. Tactics Analysis

Two complementary views:
1. **Empirical matrix** from the main tournament — what *actually happens* when players choose tactics under Random policy
2. **Forced matrix** — head-to-head under deterministic tactic forcing

## 5.1 Empirical tactic matrix (from main tournament)

These are the win rates when A and B chose their first-skirmish tactics organically (per Random policy) and the battle played out. No tactics were forced.

In [ ]:
pivot_emp = df_tactic_matrix.pivot(index="a_tactic", columns="b_tactic", values="a_win_rate")
pivot_emp = pivot_emp.reindex(index=TACTICS, columns=TACTICS)
pivot_indec = df_tactic_matrix.pivot(index="a_tactic", columns="b_tactic", values="stalemate_rate")
pivot_indec = pivot_indec.reindex(index=TACTICS, columns=TACTICS)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
im = ax.imshow(pivot_emp.values, cmap="RdYlGn", vmin=0, vmax=0.5, aspect="auto")
ax.set_xticks(range(7)); ax.set_xticklabels(TACTICS, rotation=45, ha="right")
ax.set_yticks(range(7)); ax.set_yticklabels(TACTICS)
ax.set_title("Empirical: A's win rate by (A_first_tactic, B_first_tactic)")
ax.set_xlabel("B opens with"); ax.set_ylabel("A opens with")
plt.colorbar(im, ax=ax)
for i in range(7):
    for j in range(7):
        v = pivot_emp.values[i, j]
        if not np.isnan(v):
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8)

ax = axes[1]
im = ax.imshow(pivot_indec.values, cmap="Reds", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(7)); ax.set_xticklabels(TACTICS, rotation=45, ha="right")
ax.set_yticks(range(7)); ax.set_yticklabels(TACTICS)
ax.set_title("Stalemate rate by tactic opening")
plt.colorbar(im, ax=ax)
for i in range(7):
    for j in range(7):
        v = pivot_indec.values[i, j]
        if not np.isnan(v):
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8)
plt.tight_layout(); plt.show()

## 5.2 Marginal value of each tactic

Average win rate when opening with each tactic, vs uniform opponent.

In [ ]:
marginal = pivot_emp.mean(axis=1).sort_values(ascending=False)
print("=== Marginal win rate when opening with each tactic ===")
for t, wr in marginal.items():
    print(f"  {t:<22} {wr:.3f}")
print()
print("Read: avg win rate when A opens with this tactic vs a random B opener.")
print("Higher = stronger default opening choice under random play.")

## 5.3 Forced tactics matrix (stratified sample)

For comparison. Every (a, b) tactic pair is forced — pure interaction effect without play-style noise.

In [ ]:
if df_forced_tactics is not None:
    pivot_forced = df_forced_tactics.pivot(index="a_tactic", columns="b_tactic", values="win_rate")
    pivot_forced = pivot_forced.reindex(index=TACTICS, columns=TACTICS)

    fig, ax = plt.subplots(figsize=(9, 7))
    im = ax.imshow(pivot_forced.values, cmap="RdYlGn", vmin=0, vmax=0.5, aspect="auto")
    ax.set_xticks(range(7)); ax.set_xticklabels(TACTICS, rotation=45, ha="right")
    ax.set_yticks(range(7)); ax.set_yticklabels(TACTICS)
    ax.set_title("Forced: A's win rate when A plays row vs B plays column (every skirmish)")
    ax.set_xlabel("B forced to"); ax.set_ylabel("A forced to")
    plt.colorbar(im, ax=ax)
    for i in range(7):
        for j in range(7):
            v = pivot_forced.values[i, j]
            if not np.isnan(v):
                ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8)
    plt.tight_layout(); plt.show()

    print("\nDifference (empirical - forced):")
    print((pivot_emp - pivot_forced).round(2).to_string())
else:
    print("Forced tactics CSV not loaded — skip.")

## 5.4 Counter table — which tactic counters which?

For each opponent tactic, which of YOUR tactics produces the best win rate?

In [ ]:
print("=== Best counter for each opponent opening ===")
print(f"{'Opponent plays':<22}  {'Best counter':<22}  {'Win rate':>9}")
for t in TACTICS:
    col = pivot_emp[t].dropna()
    best = col.idxmax(); wr = col.max()
    print(f"  {t:<20}  {best:<22}  {wr:>8.3f}")

# 6. Playstyle Analysis

Compare the playstyle-aware tournament (each loadout uses its assigned best playstyle) against the main Random tournament. Question: do default playstyles actually help loadouts perform?

In [ ]:
if df_ps_summary is None:
    print("Playstyle tournament not loaded — skip.")
else:
    merged = df_main_summary.merge(
        df_ps_summary[["name","win_rate","decisive_win_rate","kill_efficiency","playstyle"]],
        on="name", suffixes=("_random", "_style"))
    merged["wr_delta"] = merged["win_rate_style"] - merged["win_rate_random"]

    print("=== Playstyle vs Random — average delta in win rate ===")
    print(f"  Mean Δ win rate: {merged['wr_delta'].mean():+.4f}")
    print(f"  Median:          {merged['wr_delta'].median():+.4f}")
    print(f"  % helped:        {100*(merged['wr_delta']>0).mean():.1f}%")
    print(f"  % hurt:          {100*(merged['wr_delta']<0).mean():.1f}%")

    print("\n=== By assigned playstyle ===")
    ps_delta = merged.groupby("playstyle_style").agg(
        n=("name","count"),
        mean_delta=("wr_delta","mean"),
        median_delta=("wr_delta","median"),
        pct_helped=("wr_delta", lambda s: (s > 0).mean()),
    ).round(4)
    print(ps_delta.to_string())

In [ ]:
# Plot
if df_ps_summary is not None:
    fig, ax = plt.subplots(figsize=(11, 6))
    ps_means = merged.groupby("playstyle_style")["wr_delta"].mean().sort_values()
    colors = ["red" if v < 0 else "green" for v in ps_means.values]
    ax.barh(ps_means.index, ps_means.values, color=colors, alpha=0.7)
    ax.axvline(0, color="black", linewidth=1)
    ax.set_xlabel("Δ win rate (Playstyle - Random)")
    ax.set_title("Default playstyle effect on win rate")
    plt.tight_layout(); plt.show()

## 6.1 KT-Twins Experiment — Unshakable vs Natural Playstyle (within KT)

Each KT loadout was duplicated in the playstyle tournament: one entry uses Unshakable, the other uses the equipment-natural playstyle (Defender, Cavalry, Aggressor, etc). Both have the same KT retinue → same rout-immunity stat. The win-rate delta is the **pure playstyle contribution**, isolated from the retinue effect.

In [ ]:
# Extract KT-twin pairs and compare directly
if dont:
    pass
else:
    kt_orig = df_ps_summary[
        (df_ps_summary["retinue"] == "Knight Templar") &
        (df_ps_summary["playstyle"] == "Unshakable")
    ].copy()
    
    # Twins are named with " (NaturalPS)" suffix
    kt_twin = df_ps_summary[df_ps_summary["name"].str.contains(r"\(NaturalPS\)", regex=True)].copy()
    print(f"Found {len(kt_orig)} Unshakable KT loadouts and {len(kt_twin)} KT-twins\n")
    
    if len(kt_twin) == 0:
        print("No KT-twins found in summary. Did you re-run the playstyle tournament after adding twins?")
    else:
        # Build a join key from equipment (weapon, shield, armor, ranged, tiltyard, tags)
        def equip_key(row):
            return (row.get('weapon', ''), row.get('shield', ''), row.get('armor', ''),
                    row.get('ranged', ''), row.get('tiltyard', False), row.get('tags', ''))
        kt_orig['equip_key'] = kt_orig.apply(equip_key, axis=1)
        kt_twin['equip_key'] = kt_twin.apply(equip_key, axis=1)
    
        merged = kt_orig.merge(
            kt_twin[['equip_key', 'playstyle', 'win_rate', 'decisive_win_rate']],
            on='equip_key', suffixes=('_unshakable', '_natural')
        )
        merged['delta_wr'] = merged['win_rate_unshakable'] - merged['win_rate_natural']
        merged['delta_dec'] = merged['decisive_win_rate_unshakable'] - merged['decisive_win_rate_natural']
    
        print("=== Pair-level Unshakable vs Natural playstyle (within KT) ===")
        print(merged[['weapon', 'shield', 'armor', 'playstyle_natural',
                      'win_rate_unshakable', 'win_rate_natural', 'delta_wr',
                      'decisive_win_rate_unshakable', 'decisive_win_rate_natural', 'delta_dec']].round(3).to_string(index=False))
        print()
        print(f"=== Summary ===")
        print(f"Mean Δ win rate (Unshakable - Natural): {merged['delta_wr'].mean():+.3f}")
        print(f"Median:                                 {merged['delta_wr'].median():+.3f}")
        print(f"% pairs where Unshakable wins:          {100 * (merged['delta_wr'] > 0).mean():.0f}%")
        print(f"% pairs where Natural wins:             {100 * (merged['delta_wr'] < 0).mean():.0f}%")
        print()
        print(f"Mean Δ decisive win rate:               {merged['delta_dec'].mean():+.3f}")
        print()
        print("INTERPRETATION:")
        print("  Positive delta → Unshakable IS the better playstyle for KT equipment")
        print("  Negative delta → KT does better with the equipment-natural playstyle")
        print("  Near zero      → Playstyle doesn't matter for KT (rout-immunity carries either)")


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Similar-gear comparison: same retinue, weapon-tier within ±1, MPC within ±2
# ──────────────────────────────────────────────────────────────────────────────
# Builds a per-loadout "tier index" from its weapon, then groups matchups where
# A and B are the same retinue and gear-similar. Tells you whether one player's
# investment in a specific build configuration outperforms a similarly-equipped
# peer at the same retinue.

from renown_combat import WEAPONS, ARMORS, SHIELDS
from loadouts import TIER_ORDER

TIER_IDX = {t: i for i, t in enumerate(TIER_ORDER)}  # Crude=0 ... Crafted=4

def loadout_tier_idx(weapon, armor):
    """Effective gear tier index. Uses the higher of weapon-tier and armor-tier
    so a Levy with Cloth + Bow is Crude (0), and a Sgt with Halberd + Chainmail is Wrought (2)."""
    w_t = TIER_IDX.get(WEAPONS.get(weapon, {}).get("tier"), 0) if weapon and weapon != "Farm Tools" else 0
    a_t = TIER_IDX.get(ARMORS.get(armor, {}).get("tier"), 0)
    return max(w_t, a_t)

# Build (name → tier_idx, mpc) lookup from summary
tier_lookup = {}
for _, row in df_main_summary.iterrows():
    tier_lookup[row["name"]] = (
        loadout_tier_idx(row.get("weapon"), row.get("armor")),
        row["military_pursuit_count"],
    )

# Annotate matchups with tier/MPC info for A and B
m = df_main_matchups.copy()
m["a_tier"] = m["a_name"].map(lambda n: tier_lookup.get(n, (None, None))[0])
m["b_tier"] = m["b_name"].map(lambda n: tier_lookup.get(n, (None, None))[0])
m["a_mpc"]  = m["a_name"].map(lambda n: tier_lookup.get(n, (None, None))[1])
m["b_mpc"]  = m["b_name"].map(lambda n: tier_lookup.get(n, (None, None))[1])
m = m.dropna(subset=["a_tier","b_tier","a_mpc","b_mpc"])
m["tier_diff"] = (m["a_tier"] - m["b_tier"]).astype(int)
m["mpc_diff"]  = (m["a_mpc"] - m["b_mpc"]).astype(int)
m["n_battles"] = m["a_wins"] + m["b_wins"] + m["mut_wipe"] + m["indecisive"]

# Filter: same retinue, |tier_diff| ≤ 1, |mpc_diff| ≤ 2
similar = m[
    (m["a_retinue"] == m["b_retinue"]) &
    (m["tier_diff"].abs() <= 1) &
    (m["mpc_diff"].abs() <= 2)
].copy()

print(f"Similar-gear matchups: {len(similar):,} (out of {len(m):,} total)")
print(f"Battles in scope:      {similar['n_battles'].sum():,}\n")

# ── A. Headline by retinue ──
print("=== Aggregate win rates within retinue (same retinue, ±1 tier, ±2 MPC) ===")
agg = similar.groupby("a_retinue").agg(
    matchups=("a_name", "count"),
    n_battles=("n_battles", "sum"),
    a_wins=("a_wins", "sum"),
    b_wins=("b_wins", "sum"),
    mut=("mut_wipe", "sum"),
    indec=("indecisive", "sum"),
).reset_index()
agg["a_wr"]        = agg["a_wins"] / agg["n_battles"]
agg["decisive"]    = (agg["a_wins"] + agg["b_wins"]) / agg["n_battles"]
agg["mut_pct"]     = agg["mut"] / agg["n_battles"]
agg["indec_pct"]   = agg["indec"] / agg["n_battles"]
print(agg[["a_retinue","matchups","a_wr","decisive","mut_pct","indec_pct"]].round(3).to_string(index=False))

# ── B. How does MPC advantage matter when gear is comparable? ──
print("\n=== Effect of MPC advantage at similar tier (A.mpc - B.mpc) ===")
mpc_pivot = similar.groupby(["a_retinue","mpc_diff"]).agg(
    n=("n_battles", "sum"),
    a_wins=("a_wins", "sum"),
    b_wins=("b_wins", "sum"),
).reset_index()
mpc_pivot["a_wr"] = mpc_pivot["a_wins"] / mpc_pivot["n"]
mpc_pivot = mpc_pivot.pivot(index="a_retinue", columns="mpc_diff", values="a_wr")
print(mpc_pivot.round(3).to_string())

# ── C. How does tier matter when MPC is comparable? ──
print("\n=== Effect of tier advantage at similar MPC (A.tier - B.tier) ===")
tier_pivot = similar.groupby(["a_retinue","tier_diff"]).agg(
    n=("n_battles", "sum"),
    a_wins=("a_wins", "sum"),
    b_wins=("b_wins", "sum"),
).reset_index()
tier_pivot["a_wr"] = tier_pivot["a_wins"] / tier_pivot["n"]
tier_pivot = tier_pivot.pivot(index="a_retinue", columns="tier_diff", values="a_wr")
print(tier_pivot.round(3).to_string())

# ── D. Top punch-above-weight loadouts: high win rate vs same-retinue-similar-gear ──
per_loadout = similar.groupby(["a_name","a_retinue","a_weapon","a_shield","a_armor"]).agg(
    n=("n_battles","sum"),
    a_wins=("a_wins","sum"),
    b_wins=("b_wins","sum"),
).reset_index()
per_loadout["wr"] = per_loadout["a_wins"] / per_loadout["n"]
per_loadout["decisive_wr"] = per_loadout["a_wins"] / (per_loadout["a_wins"] + per_loadout["b_wins"]).clip(lower=1)
per_loadout = per_loadout[per_loadout["n"] >= 200]  # noise filter

print("\n=== Top 15 loadouts vs their similar-gear peers (each retinue) ===")
for ret in ["Levy","Man-at-Arms","Sergeant","Knight Templar"]:
    sub = per_loadout[per_loadout["a_retinue"] == ret].nlargest(15, "wr")
    if len(sub) == 0: continue
    print(f"\n--- {ret} ---")
    print(sub[["a_name","a_weapon","a_shield","a_armor","wr","decisive_wr","n"]].round(3).to_string(index=False))

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Weapon archetype comparison: Ranged-primary / 1H+Shield / 2H melee / Lance
# ──────────────────────────────────────────────────────────────────────────────
# Categorizes every loadout into one of four archetypes and analyzes
# matchup performance across the field, head-to-head between archetypes,
# and within-archetype best performers.

from renown_combat import WEAPONS

def archetype(weapon, shield, ranged):
    """Classify a loadout's combat role."""
    if weapon == "Lance":
        return "Lance"
    if weapon is None or weapon == "Farm Tools":
        return "Ranged primary" if ranged is not None else "1H no shield"
    if "2H" in WEAPONS.get(weapon, {}).get("tags", []):
        return "2H melee"
    return "1H + shield" if shield is not None else "1H no shield"

m = df_main_matchups.copy()

# Normalize blank/NaN -> None once (vectorized), then classify each UNIQUE (weapon,shield,ranged)
# combo a single time and map it back. Avoids per-row apply (800k Python calls -> ~dozens).
def _tag_side(df, w, s, r, out):
    wn = df[w].where(df[w].notna() & (df[w] != ""), None)
    sn = df[s].where(df[s].notna() & (df[s] != ""), None)
    rn = df[r].where(df[r].notna() & (df[r] != ""), None)
    combo = pd.DataFrame({"w": wn, "s": sn, "r": rn})
    uniq = combo.drop_duplicates()
    uniq[out] = [archetype(row.w, row.s, row.r) for row in uniq.itertuples(index=False)]
    return combo.merge(uniq, on=["w", "s", "r"], how="left")[out].values

m["a_arch"] = _tag_side(m, "a_weapon", "a_shield", "a_ranged", "a_arch")
m["b_arch"] = _tag_side(m, "b_weapon", "b_shield", "b_ranged", "b_arch")
m["n_battles"] = m["a_wins"] + m["b_wins"] + m["mut_wipe"] + m["indecisive"]

# ──────────────────────────────────────────────────────────────────────────────
# A. Aggregate performance: how does each archetype fare across the whole field?
# ──────────────────────────────────────────────────────────────────────────────
print("\n=== A. Aggregate win rate by archetype (vs the entire field) ===")
agg = m.groupby("a_arch").agg(
    n_battles=("n_battles","sum"),
    a_wins=("a_wins","sum"),
    b_wins=("b_wins","sum"),
    mut=("mut_wipe","sum"),
    indec=("indecisive","sum"),
).reset_index()
agg["win_rate"]    = agg["a_wins"] / agg["n_battles"]
agg["decisive"]    = (agg["a_wins"] + agg["b_wins"]) / agg["n_battles"]
agg["mut_pct"]     = agg["mut"] / agg["n_battles"]
agg["indec_pct"]   = agg["indec"] / agg["n_battles"]
agg = agg.sort_values("win_rate", ascending=False)
print(agg[["a_arch","n_battles","win_rate","decisive","mut_pct","indec_pct"]].round(3).to_string(index=False))

# ──────────────────────────────────────────────────────────────────────────────
# B. Head-to-head matrix: archetype vs archetype
# ──────────────────────────────────────────────────────────────────────────────
print("\n=== B. Head-to-head win rate matrix (rows=A archetype, cols=B archetype) ===")
hh = m.groupby(["a_arch","b_arch"]).agg(
    n=("n_battles","sum"),
    a_wins=("a_wins","sum"),
    b_wins=("b_wins","sum"),
    mut=("mut_wipe","sum"),
).reset_index()
hh["a_wr"] = hh["a_wins"] / hh["n"]
wr_matrix = hh.pivot(index="a_arch", columns="b_arch", values="a_wr")
print(wr_matrix.round(3).to_string())

print("\n=== Mutual wipe rate matrix ===")
hh["mut_pct"] = hh["mut"] / hh["n"]
mut_matrix = hh.pivot(index="a_arch", columns="b_arch", values="mut_pct")
print(mut_matrix.round(3).to_string())

# ──────────────────────────────────────────────────────────────────────────────
# C. By retinue: how does each archetype perform when controlling for retinue?
# ──────────────────────────────────────────────────────────────────────────────
print("\n=== C. Archetype win rate by retinue ===")
ret_arch = m.groupby(["a_retinue","a_arch"]).agg(
    n=("n_battles","sum"),
    a_wins=("a_wins","sum"),
).reset_index()
ret_arch["wr"] = ret_arch["a_wins"] / ret_arch["n"]
ret_pivot = ret_arch.pivot(index="a_retinue", columns="a_arch", values="wr")
print(ret_pivot.round(3).to_string())

# ──────────────────────────────────────────────────────────────────────────────
# D. Top performer in each archetype
# ──────────────────────────────────────────────────────────────────────────────
# Group by a_name ONLY (the other 6 columns are functions of a_name — grouping on all 7
# string cols is what made this slow). Aggregate wins, then attach descriptors via first().
per_ld = m.groupby("a_name", observed=True).agg(
    a_arch=("a_arch", "first"),
    a_retinue=("a_retinue", "first"),
    a_weapon=("a_weapon", "first"),
    a_shield=("a_shield", "first"),
    a_armor=("a_armor", "first"),
    a_ranged=("a_ranged", "first"),
    n=("n_battles", "sum"),
    a_wins=("a_wins", "sum"),
    b_wins=("b_wins", "sum"),
).reset_index()
per_ld["wr"] = per_ld["a_wins"] / per_ld["n"]
per_ld["decisive_wr"] = per_ld["a_wins"] / (per_ld["a_wins"] + per_ld["b_wins"]).clip(lower=1)
per_ld = per_ld[per_ld["n"] >= 200]  # noise filter

print("\n=== D. Top 5 loadouts in each archetype ===")
for arch in ["Ranged primary","1H + shield","2H melee","Lance"]:
    sub = per_ld[per_ld["a_arch"] == arch].nlargest(5, "wr")
    if len(sub) == 0:
        print(f"\n--- {arch}: no loadouts ---")
        continue
    print(f"\n--- {arch} (top 5 by win rate) ---")
    print(sub[["a_name","a_retinue","a_weapon","a_shield","a_armor","a_ranged","wr","decisive_wr"]]
          .round(3).to_string(index=False))

# ──────────────────────────────────────────────────────────────────────────────
# E. Upkeep efficiency by archetype (wins per gold of army upkeep)
# ──────────────────────────────────────────────────────────────────────────────
print("\n=== E. Upkeep efficiency by archetype ===")
# Use df_main_summary to get per-loadout upkeep
upk = df_main_summary[["name","army_upkeep","wins_per_1000_upkeep","win_rate"]].copy()
upk["arch"] = upk.apply(lambda r: archetype(
    r.get("weapon") if pd.notna(r.get("weapon")) else None,
    r.get("shield") if pd.notna(r.get("shield")) else None,
    r.get("ranged") if pd.notna(r.get("ranged")) else None,
), axis=1) if "weapon" in upk.columns else None

# Summary doesn't have weapon directly — pull from matchups
weapon_map = df_main_matchups.groupby("a_name").agg(
    weapon=("a_weapon","first"), shield=("a_shield","first"), ranged=("a_ranged","first"),
).to_dict("index")
upk["arch"] = upk["name"].map(lambda n: archetype(
    weapon_map.get(n, {}).get("weapon") or None,
    weapon_map.get(n, {}).get("shield") or None,
    weapon_map.get(n, {}).get("ranged") or None,
) if n in weapon_map else "unknown")

eff = upk.groupby("arch").agg(
    n=("name","count"),
    avg_upkeep=("army_upkeep","mean"),
    avg_wr=("win_rate","mean"),
    avg_wins_per_1k=("wins_per_1000_upkeep","mean"),
).sort_values("avg_wins_per_1k", ascending=False)
print(eff.round(2).to_string())

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Within-tier weapon comparison (ceteris paribus)
# ──────────────────────────────────────────────────────────────────────────────
from renown_combat import WEAPONS

RANGED_TIERS = {"Hunting Bow":"Crude","Longbow":"Cast","Javelin":"Wrought","Pilum":"Wrought","Crossbow":"Forged"}

def loadout_weapon_tier(weapon, ranged):
    has_real_melee = weapon and weapon != "Farm Tools"
    if not has_real_melee and ranged:
        return RANGED_TIERS.get(ranged, "Crude")
    return WEAPONS.get(weapon, {}).get("tier", "Crude")

def loadout_weapon_label(weapon, ranged):
    has_real_melee = weapon and weapon != "Farm Tools"
    if not has_real_melee and ranged:
        return ranged
    return weapon or "Farm Tools"

m = df_main_matchups.copy()

# Vectorized tagging: classify each UNIQUE (weapon, ranged) combo once, map back.
# Avoids 4× per-row apply (1.6M Python calls -> a few dozen).
def _tag(df, w, r, tier_out, label_out):
    wn = df[w].where(df[w].notna() & (df[w] != ""), None)
    rn = df[r].where(df[r].notna() & (df[r] != ""), None)
    combo = pd.DataFrame({"w": wn, "r": rn})
    uniq = combo.drop_duplicates().copy()
    uniq[tier_out]  = [loadout_weapon_tier(x.w, x.r)  for x in uniq.itertuples(index=False)]
    uniq[label_out] = [loadout_weapon_label(x.w, x.r) for x in uniq.itertuples(index=False)]
    merged = combo.merge(uniq, on=["w","r"], how="left")
    return merged[tier_out].values, merged[label_out].values

m["a_wtier"], m["a_wlabel"] = _tag(m, "a_weapon", "a_ranged", "a_wtier", "a_wlabel")
m["b_wtier"], m["b_wlabel"] = _tag(m, "b_weapon", "b_ranged", "b_wtier", "b_wlabel")
m["n_battles"] = m["a_wins"] + m["b_wins"] + m["mut_wipe"] + m["indecisive"]
m["a_shield_filled"] = m["a_shield"].astype("object").fillna("None").replace("", "None")
m["b_shield_filled"] = m["b_shield"].astype("object").fillna("None").replace("", "None")

TIER_WEAPONS = {
    "Crude":   ["Farm Tools", "Cudgel", "Hunting Bow"],
    "Cast":    ["Daggers", "Short Sword", "Spears", "Longbow"],
    "Wrought": ["Arming Sword", "Pike", "Flail", "Halberd", "Battle Axe", "Javelin", "Pilum"],
    "Forged":  ["Bastard Sword", "Lance", "Morningstar", "War Hammer", "Crossbow"],
}

for tier, weapons_in_tier in TIER_WEAPONS.items():
    print(f"\n{'='*78}\n=== {tier.upper()} TIER WEAPON COMPARISON ===\n{'='*78}")
    print(f"Weapons in tier: {weapons_in_tier}")
    sub = m[(m["a_wtier"] == tier) & (m["b_wtier"] == tier) &
            (m["a_retinue"] == m["b_retinue"]) & (m["a_armor"] == m["b_armor"]) &
            (m["a_shield_filled"] == m["b_shield_filled"])].copy()
    if len(sub) == 0:
        print(f"  No ceteris-paribus matchups found in {tier} tier (filter too strict)."); continue
    print(f"  Same-tier, same-retinue/armor/shield matchups: {len(sub):,} ({sub['n_battles'].sum():,} battles)")

    print(f"\n  --- Each weapon's win rate vs other {tier} weapons (controlled) ---")
    by_weapon = sub.groupby("a_wlabel", observed=True).agg(
        n_matchups=("a_name","count"), n_battles=("n_battles","sum"),
        wins=("a_wins","sum"), losses=("b_wins","sum"),
        mut=("mut_wipe","sum"), indec=("indecisive","sum")).reset_index()
    by_weapon["wr"]          = by_weapon["wins"] / by_weapon["n_battles"]
    by_weapon["decisive_wr"] = by_weapon["wins"] / (by_weapon["wins"] + by_weapon["losses"]).clip(lower=1)
    by_weapon["mut_pct"]     = by_weapon["mut"] / by_weapon["n_battles"]
    by_weapon["indec_pct"]   = by_weapon["indec"] / by_weapon["n_battles"]
    by_weapon = by_weapon.sort_values("wr", ascending=False)
    print(by_weapon[["a_wlabel","n_matchups","n_battles","wr","decisive_wr","mut_pct","indec_pct"]].round(3).to_string(index=False))

    print(f"\n  --- Head-to-head matrix (rows=A weapon, cols=B weapon) ---")
    hh = sub.groupby(["a_wlabel","b_wlabel"], observed=True).agg(
        n=("n_battles","sum"), wins=("a_wins","sum")).reset_index()
    hh["wr"] = hh["wins"] / hh["n"]
    print(hh.pivot(index="a_wlabel", columns="b_wlabel", values="wr").round(3).to_string())

    print(f"\n  --- Win rate by weapon × retinue (controlled within tier) ---")
    by_ret = sub.groupby(["a_retinue","a_wlabel"], observed=True).agg(
        n=("n_battles","sum"), wins=("a_wins","sum")).reset_index()
    by_ret["wr"] = by_ret["wins"] / by_ret["n"]
    print(by_ret.pivot(index="a_retinue", columns="a_wlabel", values="wr").round(3).to_string())

print(f"\n{'='*78}\nNotes:")
print("  - Tier mapping: Hunting Bow=Crude, Longbow=Cast, Javelin/Pilum=Wrought, Crossbow=Forged, Lance=Forged.")
print("  - 'Ceteris paribus' = same retinue + armor + shield (ranged config not pinned).")
print("  - Dual-equip loadouts categorized by melee weapon.")
print("  - High mut_pct in low tiers = weapon choice didn't decide it; rout did. Higher tiers = cleaner.")

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Within-tier weapon comparison: 1H+shield vs 2H vs Ranged
# ──────────────────────────────────────────────────────────────────────────────
from renown_combat import WEAPONS

RANGED_TIER = {"Hunting Bow":"Crude","Longbow":"Cast","Javelin":"Wrought","Pilum":"Wrought","Crossbow":"Forged"}

def weapon_handedness(weapon, ranged):
    has_real_melee = weapon and weapon != "Farm Tools"
    if not has_real_melee:
        return "Ranged" if ranged else "Other"
    return "2H melee" if "2H" in WEAPONS.get(weapon, {}).get("tags", []) else "1H+shield"

def loadout_tier(weapon, ranged):
    has_real_melee = weapon and weapon != "Farm Tools"
    if not has_real_melee and ranged:
        return RANGED_TIER.get(ranged, "Crude")
    return WEAPONS.get(weapon, {}).get("tier", "Crude")

def loadout_label(weapon, ranged):
    has_real_melee = weapon and weapon != "Farm Tools"
    if not has_real_melee and ranged:
        return ranged
    return weapon or "Farm Tools"

m = df_main_matchups.copy()

# Vectorized tagging: classify each UNIQUE (weapon, ranged) combo once, map back.
# Replaces 6× per-row apply (~2.4M Python calls) with a few dozen.
def _tag(df, w, r):
    wn = df[w].where(df[w].notna() & (df[w] != ""), None)
    rn = df[r].where(df[r].notna() & (df[r] != ""), None)
    combo = pd.DataFrame({"w": wn, "r": rn})
    uniq = combo.drop_duplicates().copy()
    uniq["hand"]  = [weapon_handedness(x.w, x.r) for x in uniq.itertuples(index=False)]
    uniq["tier"]  = [loadout_tier(x.w, x.r)      for x in uniq.itertuples(index=False)]
    uniq["label"] = [loadout_label(x.w, x.r)     for x in uniq.itertuples(index=False)]
    mg = combo.merge(uniq, on=["w","r"], how="left")
    return mg["hand"].values, mg["tier"].values, mg["label"].values

m["a_hand"], m["a_wtier"], m["a_wlabel"] = _tag(m, "a_weapon", "a_ranged")
m["b_hand"], m["b_wtier"], m["b_wlabel"] = _tag(m, "b_weapon", "b_ranged")
m["n_battles"] = m["a_wins"] + m["b_wins"] + m["mut_wipe"] + m["indecisive"]

# ──────────────────────────────────────────────────────────────────────────────
# Per-tier analysis loop
# ──────────────────────────────────────────────────────────────────────────────
for tier in ["Crude", "Cast", "Wrought", "Forged"]:
    print(f"\n{'='*82}")
    print(f"=== {tier.upper()} TIER WEAPON COMPARISON (controlled: same retinue) ===")
    print(f"{'='*82}")

    valid_hands = ("1H+shield", "2H melee", "Ranged")
    sub = m[
        (m["a_wtier"] == tier) & (m["b_wtier"] == tier) &
        (m["a_retinue"] == m["b_retinue"]) &
        (m["a_hand"].isin(valid_hands)) & (m["b_hand"].isin(valid_hands))
    ].copy()
    if len(sub) == 0:
        print(f"  No matchups in {tier} tier."); continue
    print(f"  Same-tier, same-retinue matchups: {len(sub):,} ({sub['n_battles'].sum():,} battles)")

    # ── A. Aggregate win rate by handedness ──
    print(f"\n  --- A. Aggregate win rate by handedness (vs other same-tier same-retinue) ---")
    hand_agg = sub.groupby("a_hand", observed=True).agg(
        n_battles=("n_battles","sum"), wins=("a_wins","sum"), losses=("b_wins","sum"),
        mut=("mut_wipe","sum"), indec=("indecisive","sum")).reset_index()
    hand_agg["wr"]          = hand_agg["wins"] / hand_agg["n_battles"]
    hand_agg["decisive_wr"] = hand_agg["wins"] / (hand_agg["wins"] + hand_agg["losses"]).clip(lower=1)
    hand_agg["mut_pct"]     = hand_agg["mut"] / hand_agg["n_battles"]
    hand_agg["indec_pct"]   = hand_agg["indec"] / hand_agg["n_battles"]
    hand_agg = hand_agg.sort_values("wr", ascending=False)
    print(hand_agg[["a_hand","n_battles","wr","decisive_wr","mut_pct","indec_pct"]].round(3).to_string(index=False))

    # ── B. Handedness vs handedness matrix ──
    print(f"\n  --- B. Handedness head-to-head matrix (A on row) ---")
    hh = sub.groupby(["a_hand","b_hand"], observed=True).agg(
        n=("n_battles","sum"), wins=("a_wins","sum")).reset_index()
    hh["wr"] = hh["wins"] / hh["n"]
    print(hh.pivot(index="a_hand", columns="b_hand", values="wr").round(3).to_string())

    # ── C. Per-weapon win rate within tier ──
    print(f"\n  --- C. Per-weapon win rate within tier ---")
    by_weapon = sub.groupby(["a_hand","a_wlabel"], observed=True).agg(
        n_battles=("n_battles","sum"), wins=("a_wins","sum"), losses=("b_wins","sum"),
        mut=("mut_wipe","sum"), indec=("indecisive","sum")).reset_index()
    by_weapon["wr"]          = by_weapon["wins"] / by_weapon["n_battles"]
    by_weapon["decisive_wr"] = by_weapon["wins"] / (by_weapon["wins"] + by_weapon["losses"]).clip(lower=1)
    by_weapon["mut_pct"]     = by_weapon["mut"] / by_weapon["n_battles"]
    by_weapon = by_weapon.sort_values(["a_hand","wr"], ascending=[True, False])
    print(by_weapon[["a_hand","a_wlabel","n_battles","wr","decisive_wr","mut_pct"]].round(3).to_string(index=False))

    # ── D. Weapon × weapon matrix within tier ──
    print(f"\n  --- D. Full within-tier head-to-head (A's win rate; rows=A, cols=B) ---")
    weapon_grid = sub.groupby(["a_wlabel","b_wlabel"], observed=True).agg(
        n=("n_battles","sum"), wins=("a_wins","sum")).reset_index()
    weapon_grid["wr"] = weapon_grid["wins"] / weapon_grid["n"]
    print(weapon_grid.pivot(index="a_wlabel", columns="b_wlabel", values="wr").round(3).to_string())

    # ── E. Per-retinue handedness performance ──
    print(f"\n  --- E. Handedness win rate by retinue (within tier) ---")
    by_ret = sub.groupby(["a_retinue","a_hand"], observed=True).agg(
        n=("n_battles","sum"), wins=("a_wins","sum")).reset_index()
    by_ret["wr"] = by_ret["wins"] / by_ret["n"]
    print(by_ret.pivot(index="a_retinue", columns="a_hand", values="wr").round(3).to_string())

print(f"\n{'='*82}")
print("Notes:")
print("  - Tier mapping: Hunting Bow=Crude, Longbow=Cast, Javelin/Pilum=Wrought, Crossbow=Forged, Lance=Forged.")
print("  - Dual-equip loadouts categorized by melee weapon; pure-ranged as 'Ranged' by ranged-tier.")
print("  - 1H+shield includes Lance. 'Same retinue' is the only control; armor varies between matchups.")
print("  - High mut_pct in low tiers means weapon doesn't decide outcomes; fatigue does.")

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Within-tier weapon comparison: 1H+shield vs 2H vs Ranged
# ──────────────────────────────────────────────────────────────────────────────
from renown_combat import WEAPONS

RANGED_TIER = {"Hunting Bow":"Crude","Longbow":"Cast","Javelin":"Wrought","Pilum":"Wrought","Crossbow":"Forged"}

def weapon_handedness(weapon, ranged):
    has_real_melee = weapon and weapon != "Farm Tools"
    if not has_real_melee:
        return "Ranged" if ranged else "Other"
    return "2H melee" if "2H" in WEAPONS.get(weapon, {}).get("tags", []) else "1H+shield"

def loadout_tier(weapon, ranged):
    has_real_melee = weapon and weapon != "Farm Tools"
    if not has_real_melee and ranged:
        return RANGED_TIER.get(ranged, "Crude")
    return WEAPONS.get(weapon, {}).get("tier", "Crude")

def loadout_label(weapon, ranged):
    has_real_melee = weapon and weapon != "Farm Tools"
    if not has_real_melee and ranged:
        return ranged
    return weapon or "Farm Tools"

m = df_main_matchups.copy()

# Vectorized tagging: classify each UNIQUE (weapon, ranged) combo once, map back.
# Replaces 6× per-row apply (~2.4M Python calls) with a few dozen.
def _tag(df, w, r):
    wn = df[w].where(df[w].notna() & (df[w] != ""), None)
    rn = df[r].where(df[r].notna() & (df[r] != ""), None)
    combo = pd.DataFrame({"w": wn, "r": rn})
    uniq = combo.drop_duplicates().copy()
    uniq["hand"]  = [weapon_handedness(x.w, x.r) for x in uniq.itertuples(index=False)]
    uniq["tier"]  = [loadout_tier(x.w, x.r)      for x in uniq.itertuples(index=False)]
    uniq["label"] = [loadout_label(x.w, x.r)     for x in uniq.itertuples(index=False)]
    mg = combo.merge(uniq, on=["w","r"], how="left")
    return mg["hand"].values, mg["tier"].values, mg["label"].values

m["a_hand"], m["a_wtier"], m["a_wlabel"] = _tag(m, "a_weapon", "a_ranged")
m["b_hand"], m["b_wtier"], m["b_wlabel"] = _tag(m, "b_weapon", "b_ranged")
m["n_battles"] = m["a_wins"] + m["b_wins"] + m["mut_wipe"] + m["indecisive"]

# ──────────────────────────────────────────────────────────────────────────────
# Per-tier analysis loop
# ──────────────────────────────────────────────────────────────────────────────
for tier in ["Crude", "Cast", "Wrought", "Forged"]:
    print(f"\n{'='*82}")
    print(f"=== {tier.upper()} TIER WEAPON COMPARISON (controlled: same retinue) ===")
    print(f"{'='*82}")

    valid_hands = ("1H+shield", "2H melee", "Ranged")
    sub = m[
        (m["a_wtier"] == tier) & (m["b_wtier"] == tier) &
        (m["a_retinue"] == m["b_retinue"]) &
        (m["a_hand"].isin(valid_hands)) & (m["b_hand"].isin(valid_hands))
    ].copy()
    if len(sub) == 0:
        print(f"  No matchups in {tier} tier."); continue
    print(f"  Same-tier, same-retinue matchups: {len(sub):,} ({sub['n_battles'].sum():,} battles)")

    # ── A. Aggregate win rate by handedness ──
    print(f"\n  --- A. Aggregate win rate by handedness (vs other same-tier same-retinue) ---")
    hand_agg = sub.groupby("a_hand", observed=True).agg(
        n_battles=("n_battles","sum"), wins=("a_wins","sum"), losses=("b_wins","sum"),
        mut=("mut_wipe","sum"), indec=("indecisive","sum")).reset_index()
    hand_agg["wr"]          = hand_agg["wins"] / hand_agg["n_battles"]
    hand_agg["decisive_wr"] = hand_agg["wins"] / (hand_agg["wins"] + hand_agg["losses"]).clip(lower=1)
    hand_agg["mut_pct"]     = hand_agg["mut"] / hand_agg["n_battles"]
    hand_agg["indec_pct"]   = hand_agg["indec"] / hand_agg["n_battles"]
    hand_agg = hand_agg.sort_values("wr", ascending=False)
    print(hand_agg[["a_hand","n_battles","wr","decisive_wr","mut_pct","indec_pct"]].round(3).to_string(index=False))

    # ── B. Handedness vs handedness matrix ──
    print(f"\n  --- B. Handedness head-to-head matrix (A on row) ---")
    hh = sub.groupby(["a_hand","b_hand"], observed=True).agg(
        n=("n_battles","sum"), wins=("a_wins","sum")).reset_index()
    hh["wr"] = hh["wins"] / hh["n"]
    print(hh.pivot(index="a_hand", columns="b_hand", values="wr").round(3).to_string())

    # ── C. Per-weapon win rate within tier ──
    print(f"\n  --- C. Per-weapon win rate within tier ---")
    by_weapon = sub.groupby(["a_hand","a_wlabel"], observed=True).agg(
        n_battles=("n_battles","sum"), wins=("a_wins","sum"), losses=("b_wins","sum"),
        mut=("mut_wipe","sum"), indec=("indecisive","sum")).reset_index()
    by_weapon["wr"]          = by_weapon["wins"] / by_weapon["n_battles"]
    by_weapon["decisive_wr"] = by_weapon["wins"] / (by_weapon["wins"] + by_weapon["losses"]).clip(lower=1)
    by_weapon["mut_pct"]     = by_weapon["mut"] / by_weapon["n_battles"]
    by_weapon = by_weapon.sort_values(["a_hand","wr"], ascending=[True, False])
    print(by_weapon[["a_hand","a_wlabel","n_battles","wr","decisive_wr","mut_pct"]].round(3).to_string(index=False))

    # ── D. Weapon × weapon matrix within tier ──
    print(f"\n  --- D. Full within-tier head-to-head (A's win rate; rows=A, cols=B) ---")
    weapon_grid = sub.groupby(["a_wlabel","b_wlabel"], observed=True).agg(
        n=("n_battles","sum"), wins=("a_wins","sum")).reset_index()
    weapon_grid["wr"] = weapon_grid["wins"] / weapon_grid["n"]
    print(weapon_grid.pivot(index="a_wlabel", columns="b_wlabel", values="wr").round(3).to_string())

    # ── E. Per-retinue handedness performance ──
    print(f"\n  --- E. Handedness win rate by retinue (within tier) ---")
    by_ret = sub.groupby(["a_retinue","a_hand"], observed=True).agg(
        n=("n_battles","sum"), wins=("a_wins","sum")).reset_index()
    by_ret["wr"] = by_ret["wins"] / by_ret["n"]
    print(by_ret.pivot(index="a_retinue", columns="a_hand", values="wr").round(3).to_string())

print(f"\n{'='*82}")
print("Notes:")
print("  - Tier mapping: Hunting Bow=Crude, Longbow=Cast, Javelin/Pilum=Wrought, Crossbow=Forged, Lance=Forged.")
print("  - Dual-equip loadouts categorized by melee weapon; pure-ranged as 'Ranged' by ranged-tier.")
print("  - 1H+shield includes Lance. 'Same retinue' is the only control; armor varies between matchups.")
print("  - High mut_pct in low tiers means weapon doesn't decide outcomes; fatigue does.")

# 7. Horde Mode — multi-battle survival

Loadouts fight `HORDE_BATTLES` consecutive battles with carry-over fatigue. Tests sustained performance, not single-battle peak.

In [ ]:
if df_horde is None:
    print("Horde data not loaded — skip.")
else:
    print("=== Survival by retinue ===")
    print(df_horde.groupby("retinue").agg(
        n=("name","count"),
        mean_battles=("battles_survived_mean","mean"),
        median_battles=("battles_survived_median","median"),
        size_remaining=("size_remaining_mean","mean"),
    ).round(2).to_string())

    print("\n=== Survival by MPC ===")
    print(df_horde.groupby("mpc").agg(
        n=("name","count"),
        mean_battles=("battles_survived_mean","mean"),
    ).round(2).to_string())

# 8. Loadout-Level Rollups

The detailed per-loadout views: durability, kill power, efficiency, robustness, crush rate, stalemate rate, casualty sources, cause of wipe, shaking, pursuit-presence effects, counters lookup.

## 8.1 Durability — % of battles the loadout's army survives intact

In [ ]:
dur = analysis.durability(df_main_matchups)
print("=== Top 15 most durable ===")
print(dur.head(15).round(3).to_string())

## 8.2 Kill power — average opponents killed per battle

In [ ]:
kp = analysis.kill_power(df_main_matchups)
print("=== Top 15 by kill power ===")
print(kp.head(15).round(3).to_string())

## 8.3 Battle efficiency — wins per 1000 upkeep

In [ ]:
eff = analysis.battle_efficiency(df_main_matchups)
print("=== Top 15 most efficient (wins per 1000 upkeep) ===")
print(eff.head(15).round(3).to_string())

## 8.4 Crush rate — how often this loadout achieves a decisive victory

In [ ]:
cr = analysis.crush_rate(df_main_matchups)
print("=== Top 15 by crush rate ===")
print(cr.head(15).round(3).to_string())

## 8.5 Stalemate rate — how often this loadout's matches go indecisive

In [ ]:
sr = analysis.stalemate_rate(df_main_matchups)
print("=== Top 15 most stalemate-prone ===")
print(sr.head(15).round(3).to_string())

## 8.6 Robustness — consistency of performance across opponents

In [ ]:
rb = analysis.robustness(df_main_matchups)
print("=== Top 15 most robust (consistent across opponents) ===")
print(rb.head(15).round(3).to_string())

## 8.7 Punch above weight — performance vs more expensive opponents

In [ ]:
pa = analysis.punch_above_weight(df_main_matchups)
print("=== Top 15 punch-above-weight ===")
print(pa.head(15).round(3).to_string())

## 8.8 Counters lookup — for a specific loadout, what beats it?

Customize the loadout name to query.

In [ ]:
# Top-winrate loadout by default — change `target` to any name in df_main_summary
target = df_main_summary.nlargest(1, "win_rate")["name"].iloc[0]
print(f"Loadout: {target}\n")
ctr = analysis.counters_for(df_main_matchups, target, top_n=10)
print(f"=== Top 10 counters for {target} ===")
print(ctr.round(3).to_string())

## 8.9 Casualty sources & cause of wipe

What kills armies? Combat strikes, shake tests, or rout?

In [ ]:
# Aggregate across all matchups
cas = df_main_matchups.agg({
    "avg_a_killed_combat": "mean", "avg_a_killed_shake": "mean", "avg_a_killed_rout": "mean",
})
print("=== Average casualties per battle (A's side) ===")
total = cas.sum()
for src, n in cas.items():
    print(f"  {src.replace('avg_a_killed_','').capitalize():<10} {n:.1f}  ({100*n/total:.1f}%)")

# Cause of wipe
wipe_combat = df_main_matchups["a_wipe_combat"].sum()
wipe_shake = df_main_matchups["a_wipe_shake"].sum()
wipe_rout = df_main_matchups["a_wipe_rout"].sum()
total_wipes = wipe_combat + wipe_shake + wipe_rout
if total_wipes > 0:
    print("\n=== Cause of A's wipe (when wiped) ===")
    print(f"  Combat: {wipe_combat:>8}  ({100*wipe_combat/total_wipes:.1f}%)")
    print(f"  Shake:  {wipe_shake:>8}  ({100*wipe_shake/total_wipes:.1f}%)")
    print(f"  Rout:   {wipe_rout:>8}  ({100*wipe_rout/total_wipes:.1f}%)")

## 8.10 Shaking analysis — which loadouts cause / resist shakes?

In [ ]:
# Loadouts whose attacks cause most shake casualties
print("=== Loadouts that inflict the most shake casualties (top 10) ===")
sh = df_main_matchups.groupby("a_name").agg(
    shake_dealt=("avg_b_killed_shake","sum"),
    n_matchups=("a_name","count"),
).reset_index()
sh["shake_per_matchup"] = sh["shake_dealt"] / sh["n_matchups"]
print(sh.nlargest(10, "shake_per_matchup")[["a_name","shake_per_matchup"]].round(2).to_string(index=False))

## 8.11 Pursuit-presence impact

For each pursuit, how much does owning it raise win rate (controlling for MPC)?

In [ ]:
controlled_rows = []
g_p = analysis._pursuit_summary_from_df(df_main_matchups)
# pursuits may be a categorical column (lean loader) — convert to object so fillna/str ops work.
g_p["pursuits"] = g_p["pursuits"].astype("object")

# First loop: collect the set of distinct pursuits.
all_pursuits = set()
for s in g_p["pursuits"].fillna(""):
    if s:
        all_pursuits.update(s.split("|"))

# Second loop: for each pursuit, compute the MPC-controlled win-rate delta.
for p in sorted(all_pursuits):
    mask = g_p["pursuits"].fillna("").str.contains(rf"(?:^|\|){p}(?:\||$)", regex=True)
    deltas = []
    weights = []
    for mpc in sorted(g_p["military_pursuit_count"].unique()):
        bucket = g_p[g_p["military_pursuit_count"] == mpc]
        with_p = bucket[mask.loc[bucket.index]]
        without_p = bucket[~mask.loc[bucket.index]]
        if len(with_p) >= 3 and len(without_p) >= 3:
            deltas.append(with_p["win_rate"].mean() - without_p["win_rate"].mean())
            weights.append(min(len(with_p), len(without_p)))
    if deltas:
        w = np.array(weights); d = np.array(deltas)
        controlled_rows.append({
            "pursuit": p, "n_buckets": len(deltas),
            "mean_delta_wr": (d * w).sum() / w.sum(),
        })

df_pursuit_effect = pd.DataFrame(controlled_rows).sort_values("mean_delta_wr", ascending=False)
print("=== Pursuit effect on win rate (MPC-controlled) ===")
print(df_pursuit_effect.round(4).to_string(index=False))

## 8.12 Composite dashboard

In [ ]:
print("=== Composite dashboard — top 20 by overall score ===")
dash = analysis.composite_dashboard(df_main_matchups, top_n=20)
print(dash.round(3).to_string())

## 8.13 Cost Analysis — punch-per-upkeep and cheap effective units

The new upkeep-reducing buildings (Levy Hall, Tannery, Armory, Joinery, Master Workshop, Gilded Foundry, ABF, Fletchery, Smokehouse, Butchery) let players field substantially cheaper armies. Which loadouts maximize *effective punch per gold of upkeep*? And which cheap-upkeep units perform well enough to be the backbone of a replenishable army?

In [ ]:
# Build a cost-efficiency dataframe
ce = df_main_summary.copy()
ce['army_size'] = 25  # all loadouts are 50-man

# We measure 3 cost-related metrics:
#   wins_per_1000_upkeep — already in summary
#   wins_per_gold        — same idea, finer granularity
#   replenishment_cost   — gold to replace average casualties = (25 - avg_survivors) * upkeep_per_retinue
ce['wins_per_gold'] = ce['wins'] / ce['army_upkeep'].clip(lower=1)
ce['expected_losses_per_battle'] = (ce['army_size'] - ce['avg_self_survivors']).clip(lower=0)
ce['replenishment_cost_per_battle'] = ce['expected_losses_per_battle'] * ce['upkeep_per_retinue']

# Tag value: gold-efficient with non-trivial win rate
ce['is_cheap_effective'] = (ce['army_upkeep'] <= ce['army_upkeep'].median()) & (ce['win_rate'] >= 0.40)
print(f"Cheap (≤ median upkeep {ce['army_upkeep'].median():.0f}) AND effective (≥40% wr): {ce['is_cheap_effective'].sum()} loadouts")

In [ ]:
# 1) Most efficient gold-per-win loadouts
print("=== Top 20 wins per 1000 upkeep ===")
top_eff = ce.nlargest(20, 'wins_per_1000_upkeep')[
    ['name','retinue','weapon','shield','armor','win_rate','decisive_win_rate',
     'upkeep_per_retinue','army_upkeep','wins_per_1000_upkeep']
]
print(top_eff.round(3).to_string(index=False))

In [ ]:
# 2) Cheap and effective — low upkeep with respectable win rates
print("=== Top 15 'cheap and effective' (low upkeep, decent win rate) ===")
cheap = ce[ce['is_cheap_effective']].copy()
cheap = cheap.nlargest(15, 'win_rate')[
    ['name','retinue','weapon','shield','armor','win_rate','decisive_win_rate',
     'upkeep_per_retinue','army_upkeep','wins_per_1000_upkeep']
]
print(cheap.round(3).to_string(index=False))

In [ ]:
# 3) Lowest replenishment cost — units that don't take heavy losses
print("=== Top 15 lowest replenishment cost per battle (cheap to keep alive) ===")
low_replen = ce[ce['win_rate'] >= 0.35].nsmallest(15, 'replenishment_cost_per_battle')[
    ['name','retinue','weapon','shield','armor','win_rate',
     'avg_self_survivors','upkeep_per_retinue','replenishment_cost_per_battle']
]
print(low_replen.round(2).to_string(index=False))

In [ ]:
# 4) Cost frontier — scatter of upkeep vs win rate, with the Pareto frontier highlighted
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 7))

# Color by retinue
ret_colors = {'Levy':'#888','Man-at-Arms':'#4a90e2','Sergeant':'#d4a800','Knight Templar':'#c43838'}
for ret, color in ret_colors.items():
    sub = ce[ce['retinue'] == ret]
    ax.scatter(sub['army_upkeep'], sub['win_rate'], c=color, alpha=0.4, s=15, label=ret)

# Pareto frontier: for each upkeep bucket, the best win_rate
ce_sorted = ce.sort_values('army_upkeep')
frontier = []
best_wr = -1
for _, row in ce_sorted.iterrows():
    if row['win_rate'] > best_wr:
        best_wr = row['win_rate']
        frontier.append((row['army_upkeep'], row['win_rate'], row['name']))
fx = [f[0] for f in frontier]
fy = [f[1] for f in frontier]
ax.plot(fx, fy, 'k--', linewidth=1.5, alpha=0.6, label='Pareto frontier')

ax.set_xlabel('Army upkeep (gold/turn for 25-man army)')
ax.set_ylabel('Win rate')
ax.set_title('Cost-efficiency frontier: upkeep vs win rate')
ax.legend()
ax.grid(alpha=0.3)
ax.axhline(0.5, color='black', linewidth=0.5, alpha=0.5)
plt.tight_layout(); plt.show()

print(f"\nPareto frontier has {len(frontier)} loadouts. Top of frontier (highest win rate):")
for x, y, name in frontier[-5:]:
    print(f"  upkeep={x:.0f}  wr={y:.3f}  {name[:70]}")

In [ ]:
# 5) Effect of specific upkeep-reducing pursuits on cost efficiency
upkeep_pursuits = ['Levy Hall', 'Tannery', 'Armory', 'Joinery', 'Master Workshop',
                   'Gilded Foundry', 'ABF', 'Fletchery', 'Smokehouse', 'Butchery']

rows = []
for p in upkeep_pursuits:
    mask = ce['pursuits'].fillna('').str.contains(rf"(?:^|\|){p}(?:\||$)", regex=True)
    with_p = ce[mask]
    without_p = ce[~mask]
    if len(with_p) >= 5 and len(without_p) >= 5:
        rows.append({
            'pursuit': p,
            'n_with': len(with_p),
            'n_without': len(without_p),
            'avg_upkeep_with': with_p['army_upkeep'].mean(),
            'avg_upkeep_without': without_p['army_upkeep'].mean(),
            'avg_wr_with': with_p['win_rate'].mean(),
            'avg_wr_without': without_p['win_rate'].mean(),
            'avg_wins_per_1k_with': with_p['wins_per_1000_upkeep'].mean(),
            'avg_wins_per_1k_without': without_p['wins_per_1000_upkeep'].mean(),
        })
df_pursuit_cost = pd.DataFrame(rows)
df_pursuit_cost['Δ_wr'] = df_pursuit_cost['avg_wr_with'] - df_pursuit_cost['avg_wr_without']
df_pursuit_cost['Δ_efficiency'] = df_pursuit_cost['avg_wins_per_1k_with'] - df_pursuit_cost['avg_wins_per_1k_without']
print("=== Impact of upkeep-reducing pursuits ===")
print(df_pursuit_cost.round(3).to_string(index=False))
print()
print("Δ_wr near zero means: the pursuit doesn't change which loadouts win, just makes them cheaper.")
print("Δ_efficiency positive means: the pursuit makes loadouts more cost-efficient (more wins per gold).")

In [ ]:
# 6) Replenishment scenarios — given a fixed reserve, how many battles can each loadout fight?
RESERVE_GOLD = 5000   # adjust to test different reserve budgets

ce['battles_per_reserve'] = RESERVE_GOLD / ce['replenishment_cost_per_battle'].clip(lower=1)
ce['effective_wins_per_reserve'] = ce['battles_per_reserve'] * ce['win_rate']

print(f"=== Given a {RESERVE_GOLD} gold reserve, expected wins before reinforcement gold runs out ===")
top_reserve = ce[ce['win_rate'] >= 0.30].nlargest(15, 'effective_wins_per_reserve')[
    ['name','retinue','weapon','win_rate','replenishment_cost_per_battle',
     'battles_per_reserve','effective_wins_per_reserve']
]
print(top_reserve.round(2).to_string(index=False))

# 9. Headline Findings — Design Targets

The key design target was: **win rate tightly correlates with Military Pursuit Count.** Restate and evaluate.

In [ ]:
print("=" * 60)
print("DESIGN TARGET CHECK")
print("=" * 60)
print()
print(f"1. MPC ↔ Win Rate correlation:")
print(f"   Pearson r  = {pearson_r:+.3f}  (target: > +0.7 for strong correlation)")
print(f"   Spearman r = {spearman_r:+.3f}")
print()
print(f"2. Higher MPC beats lower MPC?")
print(f"   Average win rate vs lower-MPC opponents: {df_vs['vs_lower_mean'].mean():.3f}")
print(f"   Average win rate vs higher-MPC opponents: {df_vs['vs_higher_mean'].mean():.3f}")
print(f"   Difference: {df_vs['vs_lower_mean'].mean() - df_vs['vs_higher_mean'].mean():+.3f}")
print()
print(f"3. Domain Count independent effect:")
print(f"   Pearson r (raw)  = {pearson_d:+.3f}")
print(f"   Residual r (after MPC removed) = {res_pearson:+.3f}")
print()
print(f"4. Retinue win-rate spread:")
print(f"   Best retinue:  {ret_summary['win_rate'].idxmax():<15} {ret_summary['win_rate'].max():.3f}")
print(f"   Worst retinue: {ret_summary['win_rate'].idxmin():<15} {ret_summary['win_rate'].min():.3f}")
print(f"   Spread:        {ret_summary['win_rate'].max() - ret_summary['win_rate'].min():.3f}")
print()
print(f"5. Decisive vs stalemate balance:")
print(f"   Avg decisive_rate: {df_main_summary['decisive_rate'].mean():.3f}")
print(f"   (target: > 0.7 for meaningful battles)")

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Keyword impact on win rate
# ──────────────────────────────────────────────────────────────────────────────
# For each equipment keyword, measures:
#   (A) Raw split: mean win rate of loadouts that HAVE the keyword vs those that don't.
#   (B) MPC-controlled coefficient: the keyword's marginal effect on win rate after
#       removing MPC, retinue, and tier — so we don't credit a keyword for just
#       co-occurring with high-investment builds.
# A keyword "active" on a loadout = the keyword appears on its weapon, ranged weapon,
# or shield profile.

import numpy as np, pandas as pd
from sklearn.linear_model import LinearRegression
from renown_combat import WEAPONS, RANGED, SHIELDS, ARMORS

# All equipment keywords to test
KEYWORDS = ["Shatter Armor","Destroy Shield","Cleave","Unstoppable","Nimble",
            "Steady","Unwieldy","2H","-1TBH","One Shot","+1TH"]

# Tier helpers
RANGED_TIER = {"Hunting Bow":"Crude","Longbow":"Cast","Javelin":"Wrought","Pilum":"Wrought","Crossbow":"Forged"}
TIER_ORDER  = ["Crude","Cast","Wrought","Forged","Crafted"]
TIER_IDX    = {t:i for i,t in enumerate(TIER_ORDER)}

def loadout_keywords(weapon, ranged, shield):
    """Union of keywords from the loadout's weapon, ranged weapon, and shield."""
    kws = set()
    if pd.notna(weapon) and weapon:
        kws |= set(WEAPONS.get(weapon, {}).get("tags", []))
    if pd.notna(ranged) and ranged:
        kws |= set(RANGED.get(ranged, {}).get("tags", []))
    if pd.notna(shield) and shield:
        kws |= set(SHIELDS.get(shield, {}).get("tags", []))
    return kws

def weapon_tier(weapon, ranged):
    has_melee = pd.notna(weapon) and weapon and weapon != "Farm Tools"
    if not has_melee and pd.notna(ranged) and ranged:
        return RANGED_TIER.get(ranged, "Crude")
    return WEAPONS.get(weapon, {}).get("tier", "Crude")

# Build per-loadout attribute table from matchups (a_* side) + summary win_rate
attrs = df_main_matchups.groupby("a_name").agg(
    retinue=("a_retinue","first"), weapon=("a_weapon","first"),
    shield=("a_shield","first"), armor=("a_armor","first"),
    ranged=("a_ranged","first"), mpc=("a_military_pursuit_count","first"),
).reset_index().rename(columns={"a_name":"name"})
attrs = attrs.merge(df_main_summary[["name","win_rate"]], on="name", how="inner")

# Tag each loadout with its keyword set + tier indices
attrs["kw"]        = attrs.apply(lambda r: loadout_keywords(r["weapon"], r["ranged"], r["shield"]), axis=1)
attrs["wtier_idx"] = attrs.apply(lambda r: TIER_IDX[weapon_tier(r["weapon"], r["ranged"])], axis=1)
attrs["atier_idx"] = attrs["armor"].map(lambda a: TIER_IDX[ARMORS[a]["tier"]] if pd.notna(a) else 0)
for kw in KEYWORDS:
    attrs[f"has_{kw}"] = attrs["kw"].apply(lambda s: kw in s).astype(int)

print(f"Loadouts analyzed: {len(attrs)}\n")

# ── (A) Raw split ──
rows = []
for kw in KEYWORDS:
    has  = attrs[attrs[f"has_{kw}"]==1]
    hasnt= attrs[attrs[f"has_{kw}"]==0]
    if len(has)==0 or len(hasnt)==0:
        continue
    rows.append({
        "keyword": kw, "n_with": len(has),
        "wr_with": has["win_rate"].mean(),
        "wr_without": hasnt["win_rate"].mean(),
        "raw_diff": has["win_rate"].mean() - hasnt["win_rate"].mean(),
        "avg_mpc_with": has["mpc"].mean(),
        "avg_mpc_without": hasnt["mpc"].mean(),
    })
raw = pd.DataFrame(rows).sort_values("raw_diff", ascending=False)

# ── (B) MPC/retinue/tier-controlled coefficients ──
# Single regression: win_rate ~ all keyword dummies + mpc + retinue + weapon_tier + armor_tier.
# Each keyword coefficient = its marginal win-rate effect holding the rest constant.
X_kw   = attrs[[f"has_{kw}" for kw in KEYWORDS]].astype(float)
X_kw.columns = KEYWORDS
X_ret  = pd.get_dummies(attrs["retinue"], prefix="ret", drop_first=True).astype(float)
X_ctrl = pd.concat([attrs[["mpc","wtier_idx","atier_idx"]].astype(float), X_ret], axis=1)
X = pd.concat([X_kw, X_ctrl], axis=1)
y = attrs["win_rate"].values
reg = LinearRegression().fit(X, y)

# Bootstrap CIs for the keyword coefficients
rng = np.random.default_rng(2026); n=len(X); nboot=400
Xa = X.values; boot = np.zeros((nboot, X.shape[1]))
for b in range(nboot):
    idx = rng.integers(0,n,n)
    boot[b] = LinearRegression().fit(Xa[idx], y[idx]).coef_
coef = pd.DataFrame({"feature": X.columns, "coef": reg.coef_,
                     "ci_lo": np.percentile(boot,2.5,axis=0),
                     "ci_hi": np.percentile(boot,97.5,axis=0)})
kw_coef = coef[coef["feature"].isin(KEYWORDS)].copy()
kw_coef["significant"] = (kw_coef["ci_lo"]>0) | (kw_coef["ci_hi"]<0)
kw_coef = kw_coef.sort_values("coef", ascending=False)

# ── Print ──
print("=== (A) RAW SPLIT: win rate with vs without keyword ===")
print("(raw_diff includes MPC confound — see avg_mpc columns)")
print(raw[["keyword","n_with","wr_with","wr_without","raw_diff","avg_mpc_with","avg_mpc_without"]].round(3).to_string(index=False))

print(f"\n=== (B) CONTROLLED EFFECT: marginal win-rate impact (holding MPC/retinue/tier constant) ===")
print(f"Model R² = {reg.score(X,y):.3f}  |  'significant' = 95% bootstrap CI excludes 0")
print(kw_coef[["feature","coef","ci_lo","ci_hi","significant"]].round(3).to_string(index=False))

# ── Plot (B) ──
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(10,6))
kc = kw_coef.sort_values("coef")
colors = ["#27ae60" if s else "#bbbbbb" for s in kc["significant"]]
yp = np.arange(len(kc))
ax.barh(yp, kc["coef"], xerr=[kc["coef"]-kc["ci_lo"], kc["ci_hi"]-kc["coef"]],
        color=colors, alpha=0.85, capsize=3, ecolor="black")
ax.set_yticks(yp); ax.set_yticklabels(kc["feature"])
ax.axvline(0, color="black", lw=0.5)
ax.set_xlabel("Marginal win-rate impact (Δ, MPC/retinue/tier controlled)")
ax.set_title(f"Keyword impact on win rate  |  R²={reg.score(X,y):.3f}\nGreen = statistically significant (CI excludes 0)")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "keyword_impact.png"), dpi=110, bbox_inches="tight")
plt.show()
print(f"\nSaved: {os.path.join(OUT_DIR,'keyword_impact.png')}")

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Keyword impact via controlled A/B matchups (toggle on/off, same loadout)
# ──────────────────────────────────────────────────────────────────────────────
# For each keyword, find loadouts that legitimately carry it, clone each with the
# keyword REMOVED, and run both versions against a fixed reference field. The
# win-rate delta (with − without) is the keyword's isolated effect — no regression,
# no collinearity. Works for both equipment tags and pursuit/MPC extra_tags.
#
# Requires: loadouts, renown_combat, vectorized_combat, playstyles already imported
# in the lab (Section 0). Uses the live pool, not the CSV.

import importlib, numpy as np, pandas as pd
import loadouts, renown_combat, vectorized_combat, playstyles
importlib.reload(loadouts); importlib.reload(vectorized_combat)
vectorized_combat.invalidate_tactic_tables()
from renown_combat import WEAPONS, RANGED, SHIELDS

pool = MAIN_POOL   # reuse the main tournament pool (no regeneration)

# Keywords that live in extra_tags (pursuit/MPC-granted) — these we can toggle directly.
EXTRA_TAG_KEYWORDS = [
    "Poison", "Apothecary Heal",
    "Regenerate", "Regenerate 5", "Regenerate 4", "Regenerate Reroll",
    "Immune Unwieldy",                          # only grantable immunity (Tiltyard mastery)
    "GF Armor", "MW Weapons", "Rend", "Steadfast", "Drilled", "Nimble", "Cond Field", "Parry",
    "Seize: first", "Seize: first_two", "Seize: every",
    "Outrider: once", "Outrider: first", "Outrider: first_two", "Outrider: every",
    "Riposte"
]

# Build a fixed reference field: a stratified spread of opponents across retinue/tier.
# We hold this constant so every A/B comparison faces the same gauntlet.
rng = np.random.default_rng(2026)
import random; random.seed(2026)
from collections import defaultdict
buckets = defaultdict(list)
for ld in pool:
    buckets[(ld.retinue, ld.military_pursuit_count)].append(ld)
ref_field = []
for key, lds in buckets.items():
    ref_field.extend(random.sample(lds, min(len(lds), 1)))
if len(ref_field) > 60:
    ref_field = random.sample(ref_field, 60)
ref_field = [r._replace(playstyle=playstyles.assign_default_playstyle(r)) for r in ref_field]
print(f"Reference field: {len(ref_field)} opponents\n")

def field_winrate(ld, field, n_runs=120):
    """Average win rate of `ld` across the whole reference field."""
    ld = ld._replace(playstyle=playstyles.assign_default_playstyle(ld))
    wins = total = 0
    for i, opp in enumerate(field):
        if opp.name == ld.name:
            continue
        r = vectorized_combat.run_matchup_vec(ld, opp, n_runs=n_runs, seed=4242+i, max_skirmishes=20)
        wins += r['a_wins']; total += (r['a_wins']+r['b_wins']+r['mut_wipe']+r['indecisive'])
    return wins/total if total else float('nan')

rows = []
for kw in EXTRA_TAG_KEYWORDS:
    carriers = [ld for ld in pool if kw in ld.extra_tags]
    if not carriers:
        continue
    # Sample up to N carriers spanning retinues for a representative delta
    sample = random.sample(carriers, min(len(carriers), 12))
    deltas = []
    for ld in sample:
        with_kw    = ld
        without_kw = ld._replace(extra_tags=[t for t in ld.extra_tags if t != kw])
        wr_with    = field_winrate(with_kw,    ref_field)
        wr_without = field_winrate(without_kw, ref_field)
        deltas.append(wr_with - wr_without)
    deltas = np.array(deltas)
    rows.append({
        "keyword": kw, "n_carriers": len(carriers), "n_tested": len(sample),
        "mean_delta": deltas.mean(), "median_delta": np.median(deltas),
        "min_delta": deltas.min(), "max_delta": deltas.max(),
        "std": deltas.std(),
    })
    print(f"  {kw:<20} Δ={deltas.mean():+.3f}  (n={len(sample)}, range [{deltas.min():+.3f}, {deltas.max():+.3f}])")

res = pd.DataFrame(rows).sort_values("mean_delta", ascending=False)
print("\n=== Keyword isolated win-rate impact (toggle on vs off, same loadout, vs fixed field) ===")
print(res[["keyword","n_carriers","n_tested","mean_delta","median_delta","min_delta","max_delta"]].round(3).to_string(index=False))

# Plot
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(10, 6))
rs = res.sort_values("mean_delta")
ax.barh(range(len(rs)), rs["mean_delta"],
        xerr=[rs["mean_delta"]-rs["min_delta"], rs["max_delta"]-rs["mean_delta"]],
        color="#2980b9", alpha=0.8, capsize=3, ecolor="gray")
ax.set_yticks(range(len(rs))); ax.set_yticklabels(rs["keyword"])
ax.axvline(0, color="black", lw=0.5)
ax.set_xlabel("Isolated win-rate impact (with keyword − without)")
ax.set_title("Keyword impact via controlled A/B toggle\n(error bars = min/max across tested carriers)")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "keyword_ab_impact.png"), dpi=110, bbox_inches="tight")
plt.show()
print(f"\nSaved: {os.path.join(OUT_DIR,'keyword_ab_impact.png')}")

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Ranged weapons performance analysis
# ──────────────────────────────────────────────────────────────────────────────
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from renown_combat import WEAPONS, RANGED, ARMORS

def ranged_role(weapon, ranged):
    has_melee  = weapon is not None and weapon != "" and weapon != "Farm Tools"
    has_ranged = ranged is not None and ranged != ""
    if has_ranged and not has_melee: return "pure-ranged"
    if has_ranged and has_melee:     return "dual-equip"
    return "melee-only"

m = df_main_matchups.copy()

# Vectorized role tagging: classify each UNIQUE (weapon, ranged) combo once, map back.
def _role_col(df, w, r):
    wn = df[w].where(df[w].notna() & (df[w] != ""), None)
    rn = df[r].where(df[r].notna() & (df[r] != ""), None)
    combo = pd.DataFrame({"w": wn, "r": rn})
    uniq = combo.drop_duplicates().copy()
    uniq["role"] = [ranged_role(x.w, x.r) for x in uniq.itertuples(index=False)]
    return combo.merge(uniq, on=["w","r"], how="left")["role"].values

m["a_role"] = _role_col(m, "a_weapon", "a_ranged")
m["b_role"] = _role_col(m, "b_weapon", "b_ranged")
m["n"]      = m["a_wins"] + m["b_wins"] + m["mut_wipe"] + m["indecisive"]

# Armor tier via unique-value map (a_armor is categorical now — .astype(object) first).
TIER_IDX = {"Crude":0,"Cast":1,"Wrought":2,"Forged":3,"Crafted":4}
def _atier(col):
    s = m[col].astype("object")
    return s.map(lambda a: TIER_IDX.get(ARMORS.get(a, {}).get("tier", "Crude"), 0) if pd.notna(a) else 0)
m["a_atier"] = _atier("a_armor")
m["b_atier"] = _atier("b_armor")

# ── A. Role aggregate ──
print("=== A. Role aggregate (whole-field win rate by ranged role) ===")
agg = m.groupby("a_role", observed=True).agg(
    n_battles=("n","sum"), wins=("a_wins","sum"), losses=("b_wins","sum"),
    mut=("mut_wipe","sum"), indec=("indecisive","sum")).reset_index()
agg["win_rate"]    = agg["wins"]/agg["n_battles"]
agg["decisive_wr"] = agg["wins"]/(agg["wins"]+agg["losses"]).clip(lower=1)
agg["mut_pct"]     = agg["mut"]/agg["n_battles"]
agg["indec_pct"]   = agg["indec"]/agg["n_battles"]
print(agg.sort_values("win_rate", ascending=False)[["a_role","n_battles","win_rate","decisive_wr","mut_pct","indec_pct"]].round(3).to_string(index=False))

# ── B. Per-ranged-weapon performance ──
print("\n=== B. Per-ranged-weapon win rate (loadouts carrying each ranged weapon) ===")
ranged_only = m[m["a_ranged"].notna() & (m["a_ranged"] != "")].copy()
per_rw = ranged_only.groupby("a_ranged", observed=True).agg(
    n_loadouts=("a_name","nunique"), n_battles=("n","sum"),
    wins=("a_wins","sum"), losses=("b_wins","sum"),
    mut=("mut_wipe","sum"), indec=("indecisive","sum")).reset_index()
per_rw["win_rate"]    = per_rw["wins"]/per_rw["n_battles"]
per_rw["decisive_wr"] = per_rw["wins"]/(per_rw["wins"]+per_rw["losses"]).clip(lower=1)
per_rw["mut_pct"]     = per_rw["mut"]/per_rw["n_battles"]
per_rw = per_rw[per_rw["n_battles"] > 0]
print(per_rw.sort_values("win_rate", ascending=False)[
    ["a_ranged","n_loadouts","n_battles","win_rate","decisive_wr","mut_pct"]].round(3).to_string(index=False))

# ── C. Ranged vs defender armor tier ──
print("\n=== C. Ranged carrier win rate vs defender armor tier ===")
TIER_NAME = {0:"Crude/Cloth", 1:"Cast/Leather", 2:"Wrought/Chain", 3:"Forged/Full", 4:"Crafted/Gothic"}
vs_armor = ranged_only.groupby(["a_ranged","b_atier"], observed=True).agg(
    n=("n","sum"), wins=("a_wins","sum")).reset_index()
vs_armor["wr"] = vs_armor["wins"]/vs_armor["n"]
vs_armor_pivot = vs_armor.pivot(index="a_ranged", columns="b_atier", values="wr").dropna(how="all")
vs_armor_pivot.columns = [TIER_NAME.get(c, c) for c in vs_armor_pivot.columns]
print(vs_armor_pivot.round(3).to_string())

# ── D. By retinue × role ──
print("\n=== D. Ranged role win rate by retinue ===")
ret_role = m.groupby(["a_retinue","a_role"], observed=True).agg(
    n=("n","sum"), wins=("a_wins","sum")).reset_index()
ret_role["wr"] = ret_role["wins"]/ret_role["n"]
print(ret_role.pivot(index="a_retinue", columns="a_role", values="wr").round(3).to_string())

# ── E. Tiltyard dual-equip ROI ──
print("\n=== E. Tiltyard dual-equip ROI: dual-equip vs melee-only at same MPC ===")
role_mpc = m.groupby(["a_role","a_military_pursuit_count"], observed=True).agg(
    n=("n","sum"), wins=("a_wins","sum")).reset_index()
role_mpc["wr"] = role_mpc["wins"]/role_mpc["n"]
roi_pivot = role_mpc.pivot(index="a_military_pursuit_count", columns="a_role", values="wr")
roi_pivot["dual_minus_melee"] = roi_pivot.get("dual-equip", 0) - roi_pivot.get("melee-only", 0)
print(roi_pivot.round(3).to_string())

# ── Plots ──
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
field_mean = m["a_wins"].sum()/m["n"].sum()
sub = per_rw.sort_values("win_rate")
axes[0].barh(range(len(sub)), sub["win_rate"], color="#4a90e2", alpha=0.85)
axes[0].set_yticks(range(len(sub))); axes[0].set_yticklabels(sub["a_ranged"])
axes[0].axvline(field_mean, color="red", ls="--", lw=1, label=f"field mean {field_mean:.2f}")
axes[0].set_xlabel("Win rate"); axes[0].set_title("Win rate by ranged weapon"); axes[0].legend()

hm = vs_armor_pivot.values
im = axes[1].imshow(hm, cmap="RdYlGn", vmin=0, vmax=0.5, aspect="auto")
axes[1].set_xticks(range(hm.shape[1])); axes[1].set_xticklabels(vs_armor_pivot.columns, rotation=30, ha="right")
axes[1].set_yticks(range(hm.shape[0])); axes[1].set_yticklabels(vs_armor_pivot.index)
axes[1].set_title("Ranged weapon win rate vs defender armor")
for i in range(hm.shape[0]):
    for j in range(hm.shape[1]):
        if not np.isnan(hm[i,j]):
            axes[1].text(j, i, f"{hm[i,j]:.2f}", ha="center", va="center", fontsize=9,
                         color="white" if (hm[i,j]<0.2 or hm[i,j]>0.4) else "black")
plt.colorbar(im, ax=axes[1], label="Win rate", shrink=0.8)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "ranged_analysis.png"), dpi=110, bbox_inches="tight")
plt.show()
print(f"\nSaved: {os.path.join(OUT_DIR,'ranged_analysis.png')}")

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Per-skirmish casualty analysis — first 3 skirmishes
# ──────────────────────────────────────────────────────────────────────────────
# For each side, each of the first 3 skirmishes: casualties taken, and whether that
# side struck first. Reports how often a side loses >5 and >10, actual counts, and
# splits by first-striker vs second-striker.
# REQUIRES vectorized_combat.py with log_skirmishes support.

import numpy as np, pandas as pd, random
import importlib, loadouts, vectorized_combat, playstyles
importlib.reload(loadouts); importlib.reload(vectorized_combat)
vectorized_combat.invalidate_tactic_tables()

pool = MAIN_POOL   # reuse the main tournament pool (no regeneration)

# Stratified matchup sample
random.seed(2026)
from collections import defaultdict
by_ret = defaultdict(list)
for ld in pool: by_ret[ld.retinue].append(ld)
sample = []
for ret, lds in by_ret.items():
    sample.extend(random.sample(lds, min(len(lds), 15)))
sample = [ld._replace(playstyle=playstyles.assign_default_playstyle(ld)) for ld in sample]

pairs = [(i,j) for i in range(len(sample)) for j in range(len(sample)) if i!=j]
random.shuffle(pairs); pairs = pairs[:1500]
N_RUNS = 100
print(f"{len(sample)} loadouts, {len(pairs)} matchups × {N_RUNS} runs, logging first 3 skirmishes...\n")

records = {1: {"cas": [], "first": []}, 2: {"cas": [], "first": []}, 3: {"cas": [], "first": []}}

for (i,j) in pairs:
    a, b = sample[i], sample[j]
    r = vectorized_combat.run_matchup_vec(a, b, n_runs=N_RUNS, max_skirmishes=20,
                                          seed=1000+i*97+j, log_skirmishes=3)
    for e in r["skirmish_log"]:
        sk = e["skirmish"]; act = e["active"]
        a_cas = e["a_casualties"][act]; a_first = (e["first_striker"][act] == 1)
        b_cas = e["b_casualties"][act]; b_first = (e["first_striker"][act] == -1)
        records[sk]["cas"].append(a_cas);   records[sk]["first"].append(a_first)
        records[sk]["cas"].append(b_cas);   records[sk]["first"].append(b_first)

rows = []
for sk in (1,2,3):
    cas   = np.concatenate(records[sk]["cas"])
    first = np.concatenate(records[sk]["first"])
    cas_f = cas[first]; cas_s = cas[~first]
    rows.append({
        "skirmish": sk, "n_obs": len(cas),
        "mean_cas": cas.mean(), "median_cas": np.median(cas),
        "pct_over_5": 100*(cas>5).mean(), "pct_over_10": 100*(cas>10).mean(),
        "p90": np.percentile(cas,90), "max": cas.max(),
        "mean_if_first": cas_f.mean() if len(cas_f) else np.nan,
        "mean_if_second": cas_s.mean() if len(cas_s) else np.nan,
        "pct>5_first": 100*(cas_f>5).mean() if len(cas_f) else np.nan,
        "pct>5_second": 100*(cas_s>5).mean() if len(cas_s) else np.nan,
    })
res = pd.DataFrame(rows)

print("=== Per-skirmish casualties (per side, first 3 skirmishes) ===")
print(res[["skirmish","n_obs","mean_cas","median_cas","pct_over_5","pct_over_10","p90","max"]].round(2).to_string(index=False))

print("\n=== First-striker vs second-striker casualties taken ===")
print(res[["skirmish","mean_if_first","mean_if_second","pct>5_first","pct>5_second"]].round(2).to_string(index=False))

print("\n=== Casualty-count distribution per skirmish (% of side-observations) ===")
dist_rows = []
for sk in (1,2,3):
    cas = np.concatenate(records[sk]["cas"])
    dist_rows.append({"skirmish": sk,
        "0":   round(100*(cas==0).mean(),1),
        "1-2": round(100*((cas>=1)&(cas<=2)).mean(),1),
        "3-5": round(100*((cas>=3)&(cas<=5)).mean(),1),
        "6-10":round(100*((cas>=6)&(cas<=10)).mean(),1),
        ">10": round(100*(cas>10).mean(),1)})
print(pd.DataFrame(dist_rows).to_string(index=False))

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(17,5))
axes[0].bar([f"Sk {r['skirmish']}" for r in rows], [r["pct_over_5"] for r in rows], color="#e67e22", alpha=.85, label=">5")
axes[0].bar([f"Sk {r['skirmish']}" for r in rows], [r["pct_over_10"] for r in rows], color="#c0392b", alpha=.9, label=">10")
axes[0].set_ylabel("% of side-skirmishes"); axes[0].set_title("Losing >5 / >10 in a skirmish"); axes[0].legend()
for idx,r in enumerate(rows):
    axes[0].text(idx, r["pct_over_5"]+0.5, f"{r['pct_over_5']:.0f}%", ha="center", fontsize=9)
x=np.arange(3); w=0.35
axes[1].bar(x-w/2, res["mean_if_first"], w, label="struck first", color="#27ae60", alpha=.85)
axes[1].bar(x+w/2, res["mean_if_second"], w, label="struck second", color="#8e44ad", alpha=.85)
axes[1].set_xticks(x); axes[1].set_xticklabels([f"Sk {s}" for s in (1,2,3)])
axes[1].set_ylabel("Mean casualties taken"); axes[1].set_title("Casualties: first vs second striker"); axes[1].legend()
dd = pd.DataFrame(dist_rows).set_index("skirmish")
dd.plot(kind="bar", stacked=True, ax=axes[2], colormap="RdYlGn_r")
axes[2].set_ylabel("% of side-observations"); axes[2].set_xlabel("Skirmish"); axes[2].set_title("Casualty-count distribution"); axes[2].legend(title="cas", fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "per_skirmish_casualties.png"), dpi=110, bbox_inches="tight")
plt.show()
print(f"\nSaved: {os.path.join(OUT_DIR,'per_skirmish_casualties.png')}")

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Rend value — controlled A/B comparison
# ──────────────────────────────────────────────────────────────────────────────
# NOTE: the -1 AP "MW Weapons" mechanic is retired. Master Workshop MASTERY now grants Rend
# directly (and ABF grants it too). So this tests Rend vs NO-Rend on the same builds.
# Rend raises the defender's regen save threshold, so it should matter most vs the
# regen-carrying opponents. Reports overall delta AND a regen-only breakout.

import numpy as np, pandas as pd, random
import importlib, loadouts, renown_combat, vectorized_combat, playstyles
importlib.reload(loadouts); importlib.reload(renown_combat); importlib.reload(vectorized_combat); importlib.reload(playstyles)
vectorized_combat.invalidate_tactic_tables()

pool = MAIN_POOL   # reuse the main tournament pool (no regeneration)
random.seed(2026)

from collections import defaultdict
by_ret = defaultdict(list)
for ld in pool: by_ret[ld.retinue].append(ld)
field = []
for ret, lds in by_ret.items():
    field.extend(random.sample(lds, min(len(lds), 14)))
field = [f._replace(playstyle=playstyles.assign_default_playstyle(f)) for f in field]
regen_field = [f for f in field if any('Regenerate' in t for t in f.extra_tags)]
print(f"Field: {len(field)} opponents ({len(regen_field)} carry Regenerate — Rend's targets)\n")

def field_wr(ld, opponents, n=100):
    ld = ld._replace(playstyle=playstyles.assign_default_playstyle(ld))
    w = t = 0
    for i, opp in enumerate(opponents):
        if opp.name == ld.name: continue
        r = vectorized_combat.run_matchup_vec(ld, opp, n_runs=n, seed=900+i)
        w += r['a_wins']; t += r['a_wins']+r['b_wins']+r['mut_wipe']+r['indecisive']
    return w/t if t else float('nan')

def strip_rend(ld):
    """Clone a Rend build with Rend removed — the baseline to measure Rend's value."""
    p = set(ld.pursuits); p.discard('MWRend')
    tags = [t for t in ld.extra_tags if t != 'Rend']
    return ld._replace(pursuits=frozenset(p), extra_tags=tags, name=ld.name + '/noRend')

# Builds that carry Rend via MWRend (exclude ABF — its Rend is bundled with other effects)
rend_builds = [ld for ld in pool if 'MWRend' in ld.pursuits and 'ABF' not in ld.pursuits]
print(f"MWRend (non-ABF) builds: {len(rend_builds)}")
if not rend_builds:
    print("No MWRend builds found — nothing to test."); 
else:
    sample = random.sample(rend_builds, min(30, len(rend_builds)))
    rows = []
    for ld in sample:
        wr_rend = field_wr(ld, field)
        wr_base = field_wr(strip_rend(ld), field)
        rows.append({"weapon": ld.weapon or ld.ranged, "retinue": ld.retinue, "mpc": ld.military_pursuit_count,
                     "wr_noRend": wr_base, "wr_Rend": wr_rend, "delta": wr_rend - wr_base})
    df = pd.DataFrame(rows)

    print("\n=== Rend vs no-Rend — same builds, full field ===")
    print(df.sort_values("delta", ascending=False).round(3).to_string(index=False))
    print(f"\nMean: noRend={df['wr_noRend'].mean():.3f}  Rend={df['wr_Rend'].mean():.3f}  Δ={df['delta'].mean():+.4f}")
    print(f"Builds where Rend helps: {(df['delta']>0).sum()}/{len(df)}")

    if regen_field:
        print(f"\n=== Targeted: same builds vs ONLY the {len(regen_field)} regen opponents ===")
        r2 = []
        for ld in random.sample(rend_builds, min(15, len(rend_builds))):
            r2.append({"weapon": ld.weapon or ld.ranged,
                       "wr_noRend": field_wr(strip_rend(ld), regen_field, n=150),
                       "wr_Rend": field_wr(ld, regen_field, n=150)})
        d2 = pd.DataFrame(r2); d2['delta'] = d2['wr_Rend'] - d2['wr_noRend']
        print(d2.sort_values("delta", ascending=False).round(3).to_string(index=False))
        print(f"vs regen opponents: noRend={d2['wr_noRend'].mean():.3f}  Rend={d2['wr_Rend'].mean():.3f}  Δ={d2['delta'].mean():+.4f}")

    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    ds = df.sort_values("delta")
    colors = ["#27ae60" if d > 0 else "#c0392b" for d in ds["delta"]]
    axes[0].barh(range(len(ds)), ds["delta"], color=colors, alpha=0.85)
    axes[0].set_yticks(range(len(ds))); axes[0].set_yticklabels([f"{w} ({r[:3]})" for w,r in zip(ds["weapon"],ds["retinue"])], fontsize=8)
    axes[0].axvline(0, color="black", lw=0.5)
    axes[0].set_xlabel("Δ win rate (Rend − noRend)"); axes[0].set_title("Per-build: Rend value (full field)\ngreen = Rend better")
    axes[1].scatter(df["wr_noRend"], df["wr_Rend"], c=df["mpc"], cmap="viridis", s=60, alpha=0.8)
    lims = [min(df["wr_noRend"].min(), df["wr_Rend"].min()), max(df["wr_noRend"].max(), df["wr_Rend"].max())]
    axes[1].plot(lims, lims, "k--", lw=0.5, label="equal")
    axes[1].set_xlabel("no-Rend win rate"); axes[1].set_ylabel("Rend win rate")
    axes[1].set_title("Rend vs noRend (points above line = Rend better)"); axes[1].legend()
    plt.colorbar(axes[1].collections[0], ax=axes[1], label="MPC")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "rend_value.png"), dpi=110, bbox_inches="tight")
    plt.show()
    print(f"\nSaved: {os.path.join(OUT_DIR,'rend_value.png')}")

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Initiative impact analysis
# ──────────────────────────────────────────────────────────────────────────────
#   A. Raw: win rate by base-initiative advantage (head-to-head pool sample)
#   B. Equipment init: sweep a weapon's init, measure win-rate slope
#   C. Tactics: Charge-priority vs neutral playstyle
#   D. Nimble: toggle Nimble on/off (first-skirmish +1 init)
#   E. Seize: toggle the Ministry "Seize: first" tag (innate), measure first-move edge

import numpy as np, pandas as pd, random
import importlib, loadouts, renown_combat, vectorized_combat, playstyles
importlib.reload(loadouts); importlib.reload(renown_combat); importlib.reload(vectorized_combat); importlib.reload(playstyles)
vectorized_combat.invalidate_tactic_tables()
from loadouts import Loadout, is_2h

pool = MAIN_POOL   # reuse the main tournament pool (no regeneration)
random.seed(2026)

def winrate(a, b, n=300, seed=7):
    a = a._replace(playstyle=playstyles.assign_default_playstyle(a)) if a.playstyle is None else a
    b = b._replace(playstyle=playstyles.assign_default_playstyle(b)) if b.playstyle is None else b
    r = vectorized_combat.run_matchup_vec(a, b, n_runs=n, seed=seed, max_skirmishes=20)
    tot = r['a_wins']+r['b_wins']+r['mut_wipe']+r['indecisive']
    return (r['a_wins']/tot if tot else float('nan')), (r['mut_wipe']/tot if tot else float('nan'))

SIZE=25
def chassis(weapon, shield, armor='Chainmail', ranged=None, extra=None, playstyle=None):
    return Loadout(name=f"{weapon or ranged}/{shield}", retinue='Man-at-Arms', weapon=weapon,
                   shield=shield, armor=armor, ranged=ranged, has_tiltyard=False, size=SIZE,
                   extra_tags=extra or ['Cond Field'], upkeep_per_retinue=30, playstyle=playstyle,
                   tiltyard_mastery=False, pursuits=frozenset(['Coliseum']),
                   military_pursuit_count=5, domain_count=0)

# ── A. Does higher base initiative win? ──
def base_init(ld):
    if ld.weapon and ld.weapon!='Farm Tools': wi=renown_combat.WEAPONS[ld.weapon]['init']
    elif ld.ranged: wi=renown_combat.RANGED[ld.ranged]['init']
    else: wi=0
    si=renown_combat.SHIELDS[ld.shield]['init'] if ld.shield else 0
    return wi+si

sample = [s._replace(playstyle=playstyles.assign_default_playstyle(s)) for s in random.sample(pool, 60)]
init_diff_wr = {}
for i in range(len(sample)):
    for j in range(i+1, len(sample)):
        a,b = sample[i], sample[j]
        di = base_init(a)-base_init(b)
        wr,_ = winrate(a,b,n=120,seed=100+i*60+j)
        init_diff_wr.setdefault(di, []).append(wr)
print("=== A. Win rate by base-initiative advantage ===")
print(pd.DataFrame([{"init_advantage":d,"n":len(v),"win_rate":np.mean(v)} for d,v in sorted(init_diff_wr.items())]).round(3).to_string(index=False))

# ── B. Equipment init slope ──
print("\n=== B. Equipment init slope (Arming Sword swept -2..+2, vs fixed Halberd) ===")
opp = chassis('Halberd', None)
orig = renown_combat.WEAPONS['Arming Sword']['init']
brows=[]
for test_init in [-2,-1,0,1,2]:
    renown_combat.WEAPONS['Arming Sword']['init']=test_init
    vectorized_combat.invalidate_tactic_tables()
    wr,mut = winrate(chassis('Arming Sword','Scutum Shield'),opp,n=400,seed=55)
    brows.append({"weapon_init":test_init,"base_init_w_shield":test_init-1,"win_rate":wr,"mut_pct":mut})
renown_combat.WEAPONS['Arming Sword']['init']=orig
vectorized_combat.invalidate_tactic_tables()
print(pd.DataFrame(brows).round(3).to_string(index=False))

# ── C. Tactic-driven init ──
print("\n=== C. Tactic-driven init: playstyle vs fixed defender ===")
opp = chassis('Arming Sword','Scutum Shield')
crows=[]
for ps in ['Aggressor','Cautious','Defender','Berserker','Skirmisher']:
    wr,mut = winrate(chassis('Arming Sword','Scutum Shield', playstyle=ps),opp,n=400,seed=88)
    crows.append({"playstyle":ps,"win_rate":wr,"mut_pct":mut})
print(pd.DataFrame(crows).sort_values("win_rate",ascending=False).round(3).to_string(index=False))

# ── D. Nimble toggle ──
print("\n=== D. Nimble impact (+1 init first skirmish) ===")
drows=[]
for wpn in ['Daggers','Short Sword']:
    if 'Nimble' not in renown_combat.WEAPONS[wpn]['tags']: continue
    opp = chassis('Arming Sword','Scutum Shield')
    sh = None if is_2h(wpn) else 'Scutum Shield'
    atk = chassis(wpn, sh, armor='Leather')
    wr_on,_ = winrate(atk,opp,n=400,seed=33)
    orig_tags = renown_combat.WEAPONS[wpn]['tags'][:]
    renown_combat.WEAPONS[wpn]['tags']=[t for t in orig_tags if t!='Nimble']
    vectorized_combat.invalidate_tactic_tables()
    wr_off,_ = winrate(atk,opp,n=400,seed=33)
    renown_combat.WEAPONS[wpn]['tags']=orig_tags; vectorized_combat.invalidate_tactic_tables()
    drows.append({"weapon":wpn,"wr_with_nimble":wr_on,"wr_without":wr_off,"nimble_delta":wr_on-wr_off})
print(pd.DataFrame(drows).round(3).to_string(index=False))

# ── E. Seize the Initiative (toggle the actual Seize tags) ──
# FIXED: old version used attacker_mode="attacker"/"defender" (invalid → both fell to the
# balanced split → identical matchups). Seize is driven by "Seize: first"/"Seize: every".
print("\n=== E. Seize the Initiative (toggle Ministry Seize tags) ===")
opp  = chassis('Arming Sword','Scutum Shield')
base = chassis('War Hammer', None, playstyle='Aggressor')   # init -1: benefits most
wr_none,_  = winrate(base, opp, n=600, seed=11)
wr_first,_ = winrate(base._replace(extra_tags=base.extra_tags+['Seize: first']), opp, n=600, seed=11)
wr_every,_ = winrate(base._replace(extra_tags=base.extra_tags+['Seize: first_two']), opp, n=600, seed=11)
both,_     = winrate(base._replace(extra_tags=base.extra_tags+['Seize: first']),
                     opp._replace(extra_tags=opp.extra_tags+['Seize: first']), n=600, seed=11)
print(f"  War Hammer (init -1) vs Arming Sword:")
print(f"    No Seize:      {wr_none:.3f}")
print(f"    Seize: first   {wr_first:.3f}  (Δ {wr_first-wr_none:+.3f})")
print(f"    Seize: fist two  {wr_every:.3f}  (Δ {wr_every-wr_none:+.3f})")
print(f"    Both have it:  {both:.3f}  (≈ No-Seize {wr_none:.3f}: mutual cancel)")
fast = chassis('Lance','Scutum Shield', playstyle='Cavalry')   # init +1: benefits least
fn,_ = winrate(fast, opp, n=600, seed=11)
fe,_ = winrate(fast._replace(extra_tags=fast.extra_tags+['Seize: every']), opp, n=600, seed=11)
print(f"  Lance (init +1): No Seize {fn:.3f} → Seize:every {fe:.3f}  (Δ {fe-fn:+.3f}, expect small)")

# ── Plot: A (headline) ──
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8,5))
dfA = pd.DataFrame([{"init_advantage":d,"win_rate":np.mean(v)} for d,v in sorted(init_diff_wr.items()) if len(v)>=5])
ax.plot(dfA["init_advantage"], dfA["win_rate"], marker="o", color="#2980b9")
ax.axhline(0.5, color="gray", ls="--", lw=1); ax.axvline(0, color="gray", ls=":", lw=1)
ax.set_xlabel("Base initiative advantage"); ax.set_ylabel("Win rate")
ax.set_title("Win rate vs initiative advantage (threshold effect at 0)")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR,"initiative_analysis.png"), dpi=110, bbox_inches="tight"); plt.show()
print(f"\nSaved: {os.path.join(OUT_DIR,'initiative_analysis.png')}")

In [ ]:
# Initiative i1 x i2 surface — AP/weapon identical on both sides (pure init isolation).
import importlib, renown_combat, loadouts, vectorized_combat, playstyles
import numpy as np, pandas as pd, matplotlib.pyplot as plt
for _m in [renown_combat, loadouts, vectorized_combat, playstyles]: importlib.reload(_m)

# Two clone weapons: identical AP/tags (based on Arming Sword: AP -2, Steady, +1TH); only
# init is set per cell. A uses _initA, B uses _initB, so each side's init is independent.
_BASE_TAGS = ["Steady", "+1TH"]; _BASE_AP = -2
renown_combat.WEAPONS["_initA"] = {"ap": _BASE_AP, "init": 0, "tags": list(_BASE_TAGS), "tier": "Wrought"}
renown_combat.WEAPONS["_initB"] = {"ap": _BASE_AP, "init": 0, "tags": list(_BASE_TAGS), "tier": "Wrought"}

def _mk(name, weap):
    return loadouts.Loadout(name=name, retinue="Man-at-Arms", weapon=weap, shield=None,
        armor="Chainmail", ranged=None, has_tiltyard=False, size=25, extra_tags=["Cond Field"],
        upkeep_per_retinue=30, playstyle="Defender", tiltyard_mastery=False,
        pursuits=frozenset(["Conditioning Field"]), military_pursuit_count=5, domain_count=0)

INITS = [-3, -2, -1, 0, 1, 2, 3]
_A, _B = _mk("A", "_initA"), _mk("B", "_initB")
grid = np.full((len(INITS), len(INITS)), np.nan)
for ia, va in enumerate(INITS):
    for ib, vb in enumerate(INITS):
        renown_combat.WEAPONS["_initA"]["init"] = va
        renown_combat.WEAPONS["_initB"]["init"] = vb
        vectorized_combat.invalidate_tactic_tables()
        r = vectorized_combat.run_matchup_vec(_A, _B, n_runs=400, seed=100+ia*7+ib,
                a_playstyle="Defender", b_playstyle="Defender", alternate_attacker=False)
        t = r["a_wins"]+r["b_wins"]+r["mut_wipe"]+r["indecisive"]
        grid[ia, ib] = r["a_wins"]/t if t else np.nan
del renown_combat.WEAPONS["_initA"], renown_combat.WEAPONS["_initB"]
vectorized_combat.invalidate_tactic_tables()

df_grid = pd.DataFrame(grid, index=[f"A={i}" for i in INITS], columns=[f"B={i}" for i in INITS])
print("A win rate by (A init rows x B init cols) - same AP/weapon both sides:")
print(df_grid.round(3).to_string())

# Gap collapse over the SAFE region (neither side tripped: both >= -1).
gaps = {}
for ia, va in enumerate(INITS):
    for ib, vb in enumerate(INITS):
        if va >= -1 and vb >= -1:
            gaps.setdefault(va - vb, []).append(grid[ia, ib])
gap_x = sorted(gaps)
gap_y = [np.mean(gaps[g]) for g in gap_x]
print("\nInit gap (A-B) -> mean A win rate (safe region, both >= -1):")
for g, y in zip(gap_x, gap_y):
    print(f"  gap {g:+d}: {y:.3f}")

# ── Two-panel plot ──
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5.5))

# Panel 1: heatmap
im = ax1.imshow(grid, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto", origin="lower")
ax1.set_xticks(range(len(INITS))); ax1.set_xticklabels(INITS)
ax1.set_yticks(range(len(INITS))); ax1.set_yticklabels(INITS)
ax1.set_xlabel("B initiative"); ax1.set_ylabel("A initiative")
ax1.set_title("A win-rate surface (AP/weapon identical)\ncliff: tripped side (init <= -2) auto-loses")
for ia in range(len(INITS)):
    for ib in range(len(INITS)):
        v = grid[ia, ib]
        ax1.text(ib, ia, f"{v:.2f}", ha="center", va="center", fontsize=8,
                 color="white" if (v < 0.25 or v > 0.7) else "black")
ax1.axhline(INITS.index(-2)+0.5, color="#c0392b", lw=1.2, ls="--")
ax1.axvline(INITS.index(-2)+0.5, color="#c0392b", lw=1.2, ls="--")
plt.colorbar(im, ax=ax1, label="A win rate", shrink=0.85)

# Panel 2: gap curve (the corrected "init advantage" chart)
ax2.plot(gap_x, gap_y, marker="o", color="#2c7fb8", lw=2)
ax2.axhline(0.5, color="gray", ls=":", lw=1)
ax2.axvline(0, color="gray", ls=":", lw=1)
_ylo = max(0.0, min(gap_y) - 0.05); _yhi = max(gap_y) + 0.05
ax2.set_ylim(_ylo, _yhi)
ax2.set_xlabel("Initiative gap (A init - B init)")
ax2.set_ylabel("A win rate")
ax2.set_title("Init gap -> win rate (safe region, no AP confound)\nnot flat: ~0.11/step, saturates past +-2")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "initiative_grid.png"), dpi=110, bbox_inches="tight"); plt.show()
print(f"\nSaved: {os.path.join(OUT_DIR, 'initiative_grid.png')}")

In [ ]:
# Tactic x Tactic matrix for a fixed WEAPON pairing.
# "Player A has weapon_a + tactic_x, Player B has weapon_b + tactic_y" for all tactic pairs.
# Tactics are FORCED pure (one-hot) so each cell is a clean single-tactic-vs-single-tactic read.
import importlib, renown_combat, loadouts, vectorized_combat, playstyles
import numpy as np, pandas as pd, matplotlib.pyplot as plt
for _m in [renown_combat, loadouts, vectorized_combat, playstyles]: importlib.reload(_m)
from renown_combat import TACTICS

# ── CONFIGURE ──
WEAPON_A = "Bastard Sword"      # Player A's weapon
WEAPON_B = "Poleaxe"    # Player B's weapon
ARMOR_A, ARMOR_B = "Gothic Plate", "Gothic Plate"
SHIELD_A, SHIELD_B = "Heater Shield", None
N_RUNS = 400

# Register pure forced-tactic playstyles (one-hot, bypassing the FB-reshaping in resolve()).
def _make_forced(idx):
    _oh = np.zeros(7); _oh[idx] = 1.0
    def f(state, n_runs, _oh=_oh): return np.tile(_oh, (n_runs, 1))
    return f
_forced_names = []
for _idx, _name in enumerate(TACTICS):
    _pname = f"_force_{_name}"
    playstyles.ADAPTIVE_PLAYSTYLES[_pname] = {"func": _make_forced(_idx), "description": "forced", "good_for": "", "initiate_rate": 0.5}
    playstyles.FB_INTENTIONAL_PLAYSTYLES.add(_pname)   # return weights as-is (pure one-hot)
    _forced_names.append(_pname)
playstyles.ALL_PLAYSTYLES = list(playstyles.STATIC_PLAYSTYLES.keys()) + list(playstyles.ADAPTIVE_PLAYSTYLES.keys())
vectorized_combat.invalidate_tactic_tables()

def _mk(name, weap, arm, shield):
    return loadouts.Loadout(name=name, retinue="Man-at-Arms", weapon=weap, shield=shield,
        armor=arm, ranged=None, has_tiltyard=False, size=25, extra_tags=["Cond Field"],
        upkeep_per_retinue=30, playstyle="Aggressor", tiltyard_mastery=False,
        pursuits=frozenset(["Conditioning Field"]), military_pursuit_count=5, domain_count=0)

_A = _mk("A", WEAPON_A, ARMOR_A, SHIELD_A)
_B = _mk("B", WEAPON_B, ARMOR_B, SHIELD_B)
M = np.full((7, 7), np.nan)
for i, ta in enumerate(TACTICS):
    for j, tb in enumerate(TACTICS):
        r = vectorized_combat.run_matchup_vec(_A, _B, n_runs=N_RUNS, seed=25+i*7+j,
                a_playstyle=f"_force_{ta}", b_playstyle=f"_force_{tb}", alternate_attacker=False)
        t = r["a_wins"]+r["b_wins"]+r["mut_wipe"]+r["indecisive"]
        M[i, j] = r["a_wins"]/t if t else np.nan

# cleanup the temporary forced playstyles
for _pname in _forced_names:
    playstyles.ADAPTIVE_PLAYSTYLES.pop(_pname, None)
    playstyles.FB_INTENTIONAL_PLAYSTYLES.discard(_pname)
playstyles.ALL_PLAYSTYLES = list(playstyles.STATIC_PLAYSTYLES.keys()) + list(playstyles.ADAPTIVE_PLAYSTYLES.keys())
vectorized_combat.invalidate_tactic_tables()

df_tac = pd.DataFrame(M, index=[f"A:{t}" for t in TACTICS], columns=[f"B:{t}" for t in TACTICS])
print(f"=== {WEAPON_A} (A, rows) vs {WEAPON_B} (B, cols) — A win rate by tactic pair ===")
print(df_tac.round(2).to_string())
# A's best tactic vs each B tactic, and the row/col averages
print("\nA's best response to each B tactic:")
for j, tb in enumerate(TACTICS):
    bi = np.nanargmax(M[:, j]); print(f"  vs B:{tb:18} → A plays {TACTICS[bi]:18} ({M[bi,j]:.2f})")
print(f"\nA tactic avg (over B's tactics): " + ", ".join(f"{TACTICS[i][:4]} {np.nanmean(M[i]):.2f}" for i in range(7)))

fig, ax = plt.subplots(figsize=(8, 6.5))
im = ax.imshow(M, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto", origin="upper")
ax.set_xticks(range(7)); ax.set_xticklabels(TACTICS, rotation=40, ha="right")
ax.set_yticks(range(7)); ax.set_yticklabels(TACTICS)
ax.set_xlabel(f"B tactic ({WEAPON_B})"); ax.set_ylabel(f"A tactic ({WEAPON_A})")
ax.set_title(f"{WEAPON_A} vs {WEAPON_B}\nA win rate by tactic pair (pure forced tactics)")
for i in range(7):
    for j in range(7):
        v = M[i, j]
        if not np.isnan(v):
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8,
                    color="white" if (v < 0.25 or v > 0.7) else "black")
plt.colorbar(im, ax=ax, label="A win rate", shrink=0.85)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "tactic_matrix_weaponpair.png"), dpi=110, bbox_inches="tight"); plt.show()
print(f"\nSaved: {os.path.join(OUT_DIR, 'tactic_matrix_weaponpair.png')}")

In [ ]:
# Tactic SENSITIVITY for one weapon: how much does A's tactic choice matter, averaged over a
# field of opponent weapons (each opponent plays its own assigned playstyle, i.e. realistic).
import importlib, renown_combat, loadouts, vectorized_combat, playstyles
import numpy as np, pandas as pd, matplotlib.pyplot as plt
for _m in [renown_combat, loadouts, vectorized_combat, playstyles]: importlib.reload(_m)
from renown_combat import TACTICS

# ── CONFIGURE ──
WEAPON_A = "War Hammer"      # the weapon whose tactic sensitivity we measure
ARMOR_A = "Chainmail"; SHIELD_A = None
N_RUNS = 300
# Opponent field: a representative set of enemy weapons (each on Chainmail, its own playstyle).
OPP_WEAPONS = ["Arming Sword", "Halberd", "Spears", "Lance", "Bastard Sword",
               "Morningstar", "Poleaxe", "Pike", "Battle Axe", "Flail"]

# Register pure forced-tactic playstyles for A.
def _make_forced(idx):
    _oh = np.zeros(7); _oh[idx] = 1.0
    def f(state, n_runs, _oh=_oh): return np.tile(_oh, (n_runs, 1))
    return f
_forced_names = []
for _idx, _name in enumerate(TACTICS):
    _pname = f"_force_{_name}"
    playstyles.ADAPTIVE_PLAYSTYLES[_pname] = {"func": _make_forced(_idx), "description": "forced", "good_for": "", "initiate_rate": 0.5}
    playstyles.FB_INTENTIONAL_PLAYSTYLES.add(_pname)
    _forced_names.append(_pname)
vectorized_combat.invalidate_tactic_tables()

def _mk(name, weap, arm="Chainmail", shield=None):
    return loadouts.Loadout(name=name, retinue="Man-at-Arms", weapon=weap, shield=shield,
        armor=arm, ranged=None, has_tiltyard=False, size=25, extra_tags=["Cond Field"],
        upkeep_per_retinue=30, playstyle="Aggressor", tiltyard_mastery=False,
        pursuits=frozenset(["Conditioning Field"]), military_pursuit_count=5, domain_count=0)

_A = _mk("A", WEAPON_A, ARMOR_A, SHIELD_A)
_field = []
for w in OPP_WEAPONS:
    o = _mk(f"opp_{w}", w)
    _field.append(o._replace(playstyle=playstyles.assign_default_playstyle(o)))

# For each A tactic, average A's win rate across the field (opponents play their own playstyle).
rows = []
per_tactic_vs_opp = np.full((7, len(_field)), np.nan)
for i, ta in enumerate(TACTICS):
    wins = tot = 0
    for k, opp in enumerate(_field):
        r = vectorized_combat.run_matchup_vec(_A, opp, n_runs=N_RUNS, seed=70+i*13+k,
                a_playstyle=f"_force_{ta}", b_playstyle=opp.playstyle, alternate_attacker=False)
        t = r["a_wins"]+r["b_wins"]+r["mut_wipe"]+r["indecisive"]
        wr = r["a_wins"]/t if t else np.nan
        per_tactic_vs_opp[i, k] = wr
        wins += r["a_wins"]; tot += t
    rows.append({"tactic": ta, "mean_wr": wins/tot if tot else np.nan})
df_sens = pd.DataFrame(rows).sort_values("mean_wr", ascending=False)

for _pname in _forced_names:
    playstyles.ADAPTIVE_PLAYSTYLES.pop(_pname, None)
    playstyles.FB_INTENTIONAL_PLAYSTYLES.discard(_pname)
vectorized_combat.invalidate_tactic_tables()

print(f"=== {WEAPON_A}: tactic sensitivity vs field of {len(_field)} opponents ===")
print(df_sens.round(3).to_string(index=False))
_best, _worst = df_sens["mean_wr"].max(), df_sens["mean_wr"].min()
print(f"\nTactic choice swing: {_worst:.3f} (worst) → {_best:.3f} (best) = {_best-_worst:+.3f} spread")
print(f"  → {'tactic choice matters a lot' if _best-_worst>0.15 else 'tactic choice is minor' if _best-_worst<0.05 else 'tactic choice is moderate'} for {WEAPON_A}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
# Panel 1: mean win rate per tactic
order = df_sens["tactic"].tolist()
ax1.bar(range(7), df_sens["mean_wr"], color="#2c7fb8", alpha=0.85)
ax1.set_xticks(range(7)); ax1.set_xticklabels([t[:4] for t in order], rotation=30)
ax1.set_ylabel("Mean win rate vs field"); ax1.set_title(f"{WEAPON_A}: which tactic is best (avg over opponents)")
ax1.axhline(df_sens["mean_wr"].mean(), color="gray", ls=":", lw=1, label="overall mean")
ax1.legend()
# Panel 2: heatmap tactic x opponent (where each tactic shines)
im = ax2.imshow(per_tactic_vs_opp, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
ax2.set_yticks(range(7)); ax2.set_yticklabels(TACTICS)
ax2.set_xticks(range(len(_field))); ax2.set_xticklabels(OPP_WEAPONS, rotation=40, ha="right")
ax2.set_xlabel("opponent weapon"); ax2.set_title(f"{WEAPON_A} win rate: A tactic × opponent")
plt.colorbar(im, ax=ax2, label="A win rate", shrink=0.85)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "tactic_sensitivity.png"), dpi=110, bbox_inches="tight"); plt.show()
print(f"\nSaved: {os.path.join(OUT_DIR, 'tactic_sensitivity.png')}")

In [ ]:
# #3 — Weapon x Weapon matrix at OPTIMAL TACTIC PLAY (tactic dimension solved out).
# For each weapon pairing, find a best tactic and report the win rate. Two MODEs:
#   "exploit" — A plays its BEST tactic vs B on its assigned playstyle. Discriminates weapons
#               well; ASYMMETRIC (measures how well A exploits B's fixed doctrine).
#   "fair"    — both sides best-respond (A_best vs B_best). Symmetric, but tends toward
#               defensive stalemate (low sums) since safe mutual play is non-committal.
# NOTE: in a game with strong defensive/non-engagement tactics, full game-theoretic "optimal"
# play drifts to mutual caution. "exploit" is usually the more informative weapon comparison.
import importlib, renown_combat, loadouts, vectorized_combat, playstyles
import numpy as np, pandas as pd, matplotlib.pyplot as plt
for _m in [renown_combat, loadouts, vectorized_combat, playstyles]: importlib.reload(_m)
from renown_combat import TACTICS

# ── CONFIGURE ──
MODE = "exploit"        # "exploit" or "fair"
ARMOR = "Gothic Plate"     # chassis held constant to isolate the weapon x tactic interaction
N_RUNS = 250
WEAPONS = ["Cudgel","Daggers","Short Sword","Spears","Arming Sword","Pike","Flail","Halberd",
           "Battle Axe","Bastard Sword","Lance","Morningstar","War Hammer","Poleaxe"]
# (drop/add weapons to taste; full 16 incl. Daggers/2HBastard/Farm Tools runs ~60-90s)

# Register pure forced-tactic playstyles (one-hot, bypassing FB reshaping).
def _make_forced(idx):
    _oh = np.zeros(7); _oh[idx] = 1.0
    def f(state, n_runs, _oh=_oh): return np.tile(_oh, (n_runs, 1))
    return f
_forced = []
for _idx, _name in enumerate(TACTICS):
    _p = f"_force_{_name}"
    playstyles.ADAPTIVE_PLAYSTYLES[_p] = {"func": _make_forced(_idx), "description": "forced", "good_for": "", "initiate_rate": 0.5}
    playstyles.FB_INTENTIONAL_PLAYSTYLES.add(_p); _forced.append(_p)
vectorized_combat.invalidate_tactic_tables()

def _mk(weap, arm=ARMOR):
    b = loadouts.Loadout(name=weap, retinue="Man-at-Arms", weapon=weap, shield=None, armor=arm,
        ranged=None, has_tiltyard=False, size=25, extra_tags=["Cond Field"], upkeep_per_retinue=30,
        playstyle="Aggressor", tiltyard_mastery=False, pursuits=frozenset(["Conditioning Field"]),
        military_pursuit_count=5, domain_count=0)
    return b._replace(playstyle=playstyles.assign_default_playstyle(b))

def _best_tactic(att, defn, defn_playstyle, seed0):
    best, bt = -1.0, TACTICS[0]
    for i, ta in enumerate(TACTICS):
        r = vectorized_combat.run_matchup_vec(att, defn, n_runs=N_RUNS, seed=seed0+i,
                a_playstyle=f"_force_{ta}", b_playstyle=defn_playstyle, alternate_attacker=False)
        t = r["a_wins"]+r["b_wins"]+r["mut_wipe"]+r["indecisive"]
        wr = r["a_wins"]/t if t else 0.0
        if wr > best: best, bt = wr, ta
    return bt

n = len(WEAPONS)
val = np.full((n, n), np.nan)
a_choice = np.empty((n, n), dtype=object)
b_choice = np.empty((n, n), dtype=object)
for wi, wa in enumerate(WEAPONS):
    for wj, wb in enumerate(WEAPONS):
        A, B = _mk(wa), _mk(wb)
        a_best = _best_tactic(A, B, B.playstyle, seed0=wi*991+wj*89)
        if MODE == "fair":
            b_best = _best_tactic(B, A, A.playstyle, seed0=wj*991+wi*89+7)
            bp = f"_force_{b_best}"
        else:
            b_best = B.playstyle; bp = B.playstyle
        r = vectorized_combat.run_matchup_vec(A, B, n_runs=N_RUNS*2, seed=wi*131+wj*61,
                a_playstyle=f"_force_{a_best}", b_playstyle=bp, alternate_attacker=False)
        t = r["a_wins"]+r["b_wins"]+r["mut_wipe"]+r["indecisive"]
        val[wi, wj] = r["a_wins"]/t if t else np.nan
        a_choice[wi, wj] = a_best; b_choice[wi, wj] = b_best

for _p in _forced:
    playstyles.ADAPTIVE_PLAYSTYLES.pop(_p, None); playstyles.FB_INTENTIONAL_PLAYSTYLES.discard(_p)
vectorized_combat.invalidate_tactic_tables()

df_w = pd.DataFrame(val, index=WEAPONS, columns=WEAPONS)
print(f"=== Weapon x Weapon at optimal tactic play (MODE={MODE}) — A win rate (rows=A, cols=B) ===")
print(df_w.round(2).to_string())
# Overall weapon strength = mean win rate across all opponents (excl. self-mirror)
strength = {w: np.nanmean([val[i, j] for j in range(n) if j != i]) for i, w in enumerate(WEAPONS)}
print("\nWeapon strength ranking (mean win rate vs field, excl. mirror):")
for w, s in sorted(strength.items(), key=lambda x: -x[1]):
    print(f"  {w:14} {s:.3f}")

fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(val, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(n)); ax.set_xticklabels(WEAPONS, rotation=45, ha="right")
ax.set_yticks(range(n)); ax.set_yticklabels(WEAPONS)
ax.set_xlabel("B weapon"); ax.set_ylabel("A weapon")
ax.set_title(f"Weapon vs Weapon at optimal tactic play (MODE={MODE})\nA win rate; rows that are uniformly green = strong weapons")
for i in range(n):
    for j in range(n):
        v = val[i, j]
        if not np.isnan(v):
            ax.text(j, i, f"{v:.0%}"[:-1], ha="center", va="center", fontsize=7,
                    color="white" if (v < 0.25 or v > 0.7) else "black")
plt.colorbar(im, ax=ax, label="A win rate", shrink=0.8)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, f"weapon_matrix_optimal_{MODE}.png"), dpi=110, bbox_inches="tight"); plt.show()
print(f"\nSaved: {os.path.join(OUT_DIR, f'weapon_matrix_optimal_{MODE}.png')}")

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Armor Penetration (AP) — ISOLATED, vs realistic armor+shield combinations
# ──────────────────────────────────────────────────────────────────────────────
# AP lowers the defender's save target; shields ALSO lower it (better save), so a
# defender's real durability = armor + shield together. We sweep one weapon's AP with
# init/tags fixed, and measure it against representative armor+shield defenders (the
# pairings that actually occur in the pool, by tier).

import numpy as np, pandas as pd, random
import importlib, loadouts, renown_combat, vectorized_combat, playstyles
importlib.reload(loadouts); importlib.reload(renown_combat); importlib.reload(vectorized_combat); importlib.reload(playstyles)
vectorized_combat.invalidate_tactic_tables()
from loadouts import Loadout

SIZE = 25
def mk(weapon, shield, armor='Chainmail', extra=None, playstyle='Aggressor'):
    return Loadout(name=f"{weapon}/{shield}/{armor}", retinue='Man-at-Arms', weapon=weapon, shield=shield,
                   armor=armor, ranged=None, has_tiltyard=False, size=SIZE,
                   extra_tags=extra or ['Cond Field'], upkeep_per_retinue=30, playstyle=playstyle,
                   tiltyard_mastery=False, pursuits=frozenset(['Conditioning Field']),
                   military_pursuit_count=5, domain_count=0)

def winrate(a, b, n=400, seed=11):
    a = a._replace(playstyle=playstyles.assign_default_playstyle(a)) if a.playstyle is None else a
    b = b._replace(playstyle=playstyles.assign_default_playstyle(b)) if b.playstyle is None else b
    r = vectorized_combat.run_matchup_vec(a, b, n_runs=n, seed=seed, max_skirmishes=20)
    t = r['a_wins']+r['b_wins']+r['mut_wipe']+r['indecisive']
    return (r['a_wins']/t if t else np.nan)

# Effective save target a defender presents (lower = tougher): armor_save - shield_bonus.
def eff_save(armor, shield):
    s = renown_combat.ARMORS[armor]['save']
    sb = renown_combat.SHIELDS[shield]['save_bonus'] if shield else 0
    return s - sb

# Representative defender profiles: armor paired with the shield that tier realistically uses.
# (None = the no-shield / 2H-defender case for that tier.)
DEFENDERS = [
    ("Cloth",        None),
    ("Cloth",        "Wooden Shield"),
    ("Leather",      "Kite Shield"),
    ("Chainmail",    None),
    ("Chainmail",    "Scutum Shield"),
    ("Full Plate",   None),
    ("Full Plate",   "Kite Shield"),
    ("Full Plate",   "Scutum Shield"),
    ("Full Plate",   "Tower Shield"),
    ("Full Plate",   "Heater Shield"),
    ("Gothic Plate", None),
    ("Gothic Plate", "Scutum Shield"),
    ("Gothic Plate", "Heater Shield"),
    ("Gothic Plate", "Tower Shield")
]
# Sort by effective save (toughest last) for readable axes
DEFENDERS = sorted(DEFENDERS, key=lambda ad: -eff_save(ad[0], ad[1]))  # weakest (high target) first
def_labels = [f"{a}+{s.split()[0]}" if s else f"{a}" for a, s in DEFENDERS]

print("Defender durability (effective save target, lower = tougher):")
for (a, s), lab in zip(DEFENDERS, def_labels):
    print(f"  {lab:22} save {eff_save(a,s)}+")

# ── AP sweep × defender (armor+shield) ──
print("\n=== AP win rate vs armor+shield defenders ===")
orig_ap = renown_combat.WEAPONS['Arming Sword']['ap']
attacker_armor = 'Chainmail'   # attacker fixed
grid = []
for test_ap in [0,-1, -2, -3, -4, -5, -6, -8]:
    renown_combat.WEAPONS['Arming Sword']['ap'] = test_ap
    vectorized_combat.invalidate_tactic_tables()
    row = {"weapon_AP": test_ap}
    for (arm, sh), lab in zip(DEFENDERS, def_labels):
        atk = mk('Arming Sword', 'Scutum Shield', armor=attacker_armor)   # attacker fixed
        # defender uses a Steady weapon so a shield is legal; Arming Sword is 1H, shield ok
        defn = mk('Arming Sword', sh, armor=arm)
        row[lab] = round(winrate(atk, defn, n=400, seed=77), 3)
    grid.append(row)
renown_combat.WEAPONS['Arming Sword']['ap'] = orig_ap
vectorized_combat.invalidate_tactic_tables()
df = pd.DataFrame(grid)
print(df.to_string(index=False))
print("\nRead down a column: how much extra AP helps against THAT armor+shield wall.")
print("Read across a row: how a fixed AP level decays as the enemy stacks armor+shield.")

# ── Saturation point per defender: lowest AP where win rate stops improving ──
print("\n=== AP saturation point per defender (where extra AP stops mattering) ===")
sat = []
for lab in def_labels:
    col = df[lab].values
    aps = df["weapon_AP"].values
    # find first AP (scanning most→least negative... here ascending order 0..-8) where
    # further AP gives <0.01 improvement
    best = col.max()
    sat_ap = aps[0]
    for k in range(1, len(col)):
        if col[k] - col[k-1] < 0.01 and col[k] >= best - 0.01:
            sat_ap = aps[k-1]; break
        sat_ap = aps[k]
    sat.append({"defender": lab, "save": eff_save(*DEFENDERS[def_labels.index(lab)]),
                "best_wr": round(best,3), "AP_saturates_at": sat_ap})
print(pd.DataFrame(sat).to_string(index=False))

# ── Plot heatmap ──
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(12, 5))
hm = df[def_labels].values
im = ax.imshow(hm, cmap="RdYlGn", aspect="auto", vmin=0, vmax=1)
ax.set_xticks(range(len(def_labels))); ax.set_xticklabels(def_labels, rotation=35, ha="right")
ax.set_yticks(range(len(df))); ax.set_yticklabels(df["weapon_AP"])
ax.set_ylabel("Weapon AP"); ax.set_xlabel("Defender (armor + shield, weakest → toughest)")
ax.set_title("AP win rate vs armor+shield combinations (init/tags fixed)")
for i in range(hm.shape[0]):
    for j in range(hm.shape[1]):
        ax.text(j, i, f"{hm[i,j]:.2f}", ha="center", va="center", fontsize=8,
                color="white" if (hm[i,j]<0.25 or hm[i,j]>0.7) else "black")
plt.colorbar(im, ax=ax, label="win rate", shrink=0.8)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "ap_vs_armor_shield.png"), dpi=110, bbox_inches="tight"); plt.show()
print(f"\nSaved: {os.path.join(OUT_DIR,'ap_vs_armor_shield.png')}")

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Armor Penetration (AP) — vs armor+shield, WITH an Unstoppable comparison
# ──────────────────────────────────────────────────────────────────────────────
# AP and shields both shift the save target. Shields ALSO impose -1TBH (attacker must
# roll +1 higher to hit). Unstoppable zeroes that shield -1TBH — so it should help ONLY
# against shielded defenders. We add three extra columns: vs Scutum / Tower / Heater
# defenders with the attacker carrying Unstoppable, to see how much it cracks shields.

import numpy as np, pandas as pd, random
import importlib, loadouts, renown_combat, vectorized_combat, playstyles
importlib.reload(loadouts); importlib.reload(renown_combat); importlib.reload(vectorized_combat); importlib.reload(playstyles)
vectorized_combat.invalidate_tactic_tables()
from loadouts import Loadout

SIZE = 25
def mk(weapon, shield, armor='Chainmail', extra=None, playstyle='Aggressor'):
    return Loadout(name=f"{weapon}/{shield}/{armor}", retinue='Man-at-Arms', weapon=weapon, shield=shield,
                   armor=armor, ranged=None, has_tiltyard=False, size=SIZE,
                   extra_tags=extra or ['Cond Field'], upkeep_per_retinue=30, playstyle=playstyle,
                   tiltyard_mastery=False, pursuits=frozenset(['Conditioning Field']),
                   military_pursuit_count=5, domain_count=0)

def winrate(a, b, n=400, seed=11):
    a = a._replace(playstyle=playstyles.assign_default_playstyle(a)) if a.playstyle is None else a
    b = b._replace(playstyle=playstyles.assign_default_playstyle(b)) if b.playstyle is None else b
    r = vectorized_combat.run_matchup_vec(a, b, n_runs=n, seed=seed, max_skirmishes=20)
    t = r['a_wins']+r['b_wins']+r['mut_wipe']+r['indecisive']
    return (r['a_wins']/t if t else np.nan)

def eff_save(armor, shield):
    s = renown_combat.ARMORS[armor]['save']
    sb = renown_combat.SHIELDS[shield]['save_bonus'] if shield else 0
    return s - sb

# Base defenders (armor + shield), sorted weakest → toughest by effective save.
DEFENDERS = [
    ("Cloth", None), ("Cloth", "Wooden Shield"), ("Leather", "Kite Shield"),
    ("Chainmail", None), ("Chainmail", "Scutum Shield"), ("Full Plate", None),
    ("Full Plate", "Tower Shield"), ("Gothic Plate", None), ("Gothic Plate", "Heater Shield"),
]
DEFENDERS = sorted(DEFENDERS, key=lambda ad: -eff_save(ad[0], ad[1]))
def_labels = [f"{a}+{s.split()[0]}" if s else a for a, s in DEFENDERS]

# The three shielded walls we want an Unstoppable comparison for.
UNSTOPPABLE_TARGETS = [
    ("Chainmail",    "Scutum Shield"),
    ("Full Plate",   "Tower Shield"),
    ("Gothic Plate", "Heater Shield"),
]

print("Defender durability (effective save, lower = tougher):")
for (a, s), lab in zip(DEFENDERS, def_labels):
    print(f"  {lab:22} save {eff_save(a,s)}+")

orig_ap = renown_combat.WEAPONS['Arming Sword']['ap']
ATTACKER_ARMOR = 'Chainmail'

print("\n=== AP win rate vs defenders, plus Unstoppable-vs-shield comparison ===")
grid = []
for test_ap in [0, -2, -3, -4, -5, -6, -8]:
    renown_combat.WEAPONS['Arming Sword']['ap'] = test_ap
    vectorized_combat.invalidate_tactic_tables()
    row = {"weapon_AP": test_ap}
    # Standard columns: normal attacker vs every defender
    for (arm, sh), lab in zip(DEFENDERS, def_labels):
        atk = mk('Arming Sword', 'Scutum Shield', armor=ATTACKER_ARMOR)
        defn = mk('Arming Sword', sh, armor=arm)
        row[lab] = round(winrate(atk, defn, n=400, seed=77), 3)
    # Three extra columns: attacker WITH Unstoppable vs the shielded walls
    for (arm, sh) in UNSTOPPABLE_TARGETS:
        atk_u = mk('Arming Sword', 'Scutum Shield', armor=ATTACKER_ARMOR,
                   extra=['Cond Field', 'Unstoppable'])
        defn = mk('Arming Sword', sh, armor=arm)
        col = f"UNSTOP→{sh.split()[0]}"
        row[col] = round(winrate(atk_u, defn, n=400, seed=77), 3)
    grid.append(row)
renown_combat.WEAPONS['Arming Sword']['ap'] = orig_ap
vectorized_combat.invalidate_tactic_tables()

df = pd.DataFrame(grid)
print(df.to_string(index=False))

# ── Delta table: how much Unstoppable adds vs each shield wall ──
print("\n=== Unstoppable delta (Unstoppable − normal) vs each shielded wall ===")
delta_rows = []
for (arm, sh) in UNSTOPPABLE_TARGETS:
    base_lab = f"{arm}+{sh.split()[0]}"
    unstop_lab = f"UNSTOP→{sh.split()[0]}"
    for ap, base_v, u_v in zip(df["weapon_AP"], df[base_lab], df[unstop_lab]):
        delta_rows.append({"AP": ap, "wall": base_lab, "normal": base_v,
                           "unstoppable": u_v, "delta": round(u_v - base_v, 3)})
dd = pd.DataFrame(delta_rows)
print(dd.to_string(index=False))
print("\nUnstoppable zeroes the shield's -1TBH, so the delta = the shield's to-hit value.")
print("Expect positive deltas (shields hurt the attacker; Unstoppable removes that).")

# ── Plot: the three walls, normal vs Unstoppable, across AP ──
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
for ax, (arm, sh) in zip(axes, UNSTOPPABLE_TARGETS):
    base_lab = f"{arm}+{sh.split()[0]}"; unstop_lab = f"UNSTOP→{sh.split()[0]}"
    ax.plot(df["weapon_AP"], df[base_lab], marker="o", color="#c0392b", label="normal")
    ax.plot(df["weapon_AP"], df[unstop_lab], marker="s", color="#27ae60", label="+ Unstoppable")
    ax.set_title(f"vs {base_lab} (save {eff_save(arm,sh)}+)")
    ax.set_xlabel("Weapon AP"); ax.invert_xaxis(); ax.axhline(0.5, color="gray", ls=":", lw=1)
    ax.legend()
axes[0].set_ylabel("Win rate")
plt.suptitle("AP vs shielded walls: normal vs Unstoppable (Unstoppable negates shield -1TBH)")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "ap_unstoppable_vs_shields.png"), dpi=110, bbox_inches="tight"); plt.show()
print(f"\nSaved: {os.path.join(OUT_DIR,'ap_unstoppable_vs_shields.png')}")

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Poison & Destroy Shield vs armor+shield walls (incl. the save-0+ Gothic+Tower wall)
# ──────────────────────────────────────────────────────────────────────────────
# Raw AP saturates and can't crack the heaviest shielded walls (save <2 = auto-pass).
# Two keywords bypass that:
#   - Poison: fails the defender's save on a natural 6 — works even vs auto-pass (save <2).
#   - Destroy Shield: removes the shield mid-fight, collapsing the wall to bare armor.
# We measure a baseline attacker vs each wall, then the same attacker with Poison, with
# Destroy Shield, and with both.

import numpy as np, pandas as pd, random
import importlib, loadouts, renown_combat, vectorized_combat, playstyles
importlib.reload(loadouts); importlib.reload(renown_combat); importlib.reload(vectorized_combat); importlib.reload(playstyles)
vectorized_combat.invalidate_tactic_tables()
from loadouts import Loadout

SIZE = 25
def mk(weapon, shield, armor='Chainmail', extra=None, playstyle='Aggressor'):
    return Loadout(name=f"{weapon}/{shield}/{armor}", retinue='Man-at-Arms', weapon=weapon, shield=shield,
                   armor=armor, ranged=None, has_tiltyard=False, size=SIZE,
                   extra_tags=extra or ['Cond Field'], upkeep_per_retinue=30, playstyle=playstyle,
                   tiltyard_mastery=False, pursuits=frozenset(['Conditioning Field']),
                   military_pursuit_count=5, domain_count=0)

def winrate(a, b, n=400, seed=11):
    a = a._replace(playstyle=playstyles.assign_default_playstyle(a)) if a.playstyle is None else a
    b = b._replace(playstyle=playstyles.assign_default_playstyle(b)) if b.playstyle is None else b
    r = vectorized_combat.run_matchup_vec(a, b, n_runs=n, seed=seed, max_skirmishes=20)
    t = r['a_wins']+r['b_wins']+r['mut_wipe']+r['indecisive']
    return (r['a_wins']/t if t else np.nan)

def eff_save(armor, shield):
    s = renown_combat.ARMORS[armor]['save']
    sb = renown_combat.SHIELDS[shield]['save_bonus'] if shield else 0
    return s - sb

# Shielded walls, weakest → toughest, INCLUDING Gothic+Tower (save 0+).
WALLS = [
    ("Chainmail",    "Scutum Shield"),
    ("Full Plate",   "Tower Shield"),
    ("Gothic Plate", "Heater Shield"),
    ("Gothic Plate", "Tower Shield"),   # save 0+ — the AP-proof wall
]
wall_labels = [f"{a}+{s.split()[0]} ({eff_save(a,s)}+)" for a, s in WALLS]

print("Walls under test (effective save):")
for (a, s), lab in zip(WALLS, wall_labels):
    print(f"  {lab}")

# Attacker: Arming Sword at a fixed strong AP (-4) so we isolate the keyword effect, not AP.
renown_combat.WEAPONS['Arming Sword']['ap'] = -4  # temp, restored at end
ATK_AP_NOTE = "Arming Sword @ AP -4 (fixed)"
orig_ap = -2  # actual original

print(f"\n=== {ATK_AP_NOTE}: baseline vs Poison vs Destroy Shield vs both ===")
variants = {
    "baseline":        ['Cond Field'],
    "+Poison":         ['Cond Field', 'Poison'],
    "+Destroy Shield": ['Cond Field', 'Destroy Shield'],
    "+both":           ['Cond Field', 'Poison', 'Destroy Shield'],
}
rows = []
for (arm, sh), lab in zip(WALLS, wall_labels):
    row = {"wall": lab}
    for vname, tags in variants.items():
        atk = mk('Arming Sword', 'Scutum Shield', armor='Chainmail', extra=tags)
        defn = mk('Arming Sword', sh, armor=arm)
        row[vname] = round(winrate(atk, defn, n=500, seed=77), 3)
    rows.append(row)
renown_combat.WEAPONS['Arming Sword']['ap'] = orig_ap
df = pd.DataFrame(rows)
print(df.to_string(index=False))
print("\n+Poison: bypasses save on 6s (works even vs save-0+ auto-pass).")
print("+Destroy Shield: collapses the shield, dropping the wall to bare armor.")

# Plot grouped bars
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(WALLS)); w = 0.2
for i, vname in enumerate(variants):
    ax.bar(x + (i-1.5)*w, df[vname], w, label=vname)
ax.set_xticks(x); ax.set_xticklabels(wall_labels, rotation=20, ha="right")
ax.set_ylabel("Win rate"); ax.set_title("Poison & Destroy Shield vs shielded walls (attacker AP -4)")
ax.axhline(0.5, color="gray", ls=":", lw=1); ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "poison_destroyshield_vs_walls.png"), dpi=110, bbox_inches="tight"); plt.show()
print(f"\nSaved: {os.path.join(OUT_DIR,'poison_destroyshield_vs_walls.png')}")

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Pursuit A/B testing — compare two specific pursuits head-to-head
# ──────────────────────────────────────────────────────────────────────────────
# For a set of base loadouts, builds two variants: one with pursuit A's tag(s), one with
# pursuit B's tag(s), holding everything else fixed. Runs both vs a shared field and reports
# the win-rate delta. Use to answer "Rend vs no-Rend?", "Preceptory (Steadfast) vs Royal
# Pavilion (Drilled+Nimble)?", "Cond Field vs none?", etc. NOTE: -1 AP is retired; Master
# Workshop / ABF grant Rend. Ministry innate = Seize: first; mastery = MaxInit3 + MinInit+1.

import numpy as np, pandas as pd, random
import importlib, loadouts, renown_combat, vectorized_combat, playstyles
importlib.reload(loadouts); importlib.reload(renown_combat); importlib.reload(vectorized_combat); importlib.reload(playstyles)
vectorized_combat.invalidate_tactic_tables()

pool = MAIN_POOL   # reuse the main tournament pool (no regeneration)
random.seed(2026)

# ── CONFIGURE THE A/B HERE ──
# Each side is (label, set_of_tags_to_ADD, set_of_tags_to_REMOVE). The harness clones a base
# loadout, strips the REMOVE tags, adds the ADD tags, reassigns playstyle, and runs it.
# Examples below — uncomment/edit the pair you want to test.

AB_PAIRS = {
    # "Master Workshop (AP-1)" vs "MWRend (Rend)"
    "MW (AP-1)":  {"add": {"MW Weapons"}, "remove": {"Rend"}},
    "MWRend (Rend)": {"add": {"Rend"},      "remove": {"MW Weapons"}},
}
# Other ready-to-use pairs (swap into AB_PAIRS):
#   Preceptory vs Royal Pavilion:
#     "Preceptory (Steadfast)": {"add": {"Steadfast"}, "remove": {"Drilled","Nimble","Immune Strain"}},
#     "Royal Pavilion (D/N/IS)": {"add": {"Drilled","Nimble","Immune Strain"}, "remove": {"Steadfast"}},
#   Ministry seize-first vs seize-every:
#     "Seize: first": {"add": {"Seize: first"}, "remove": {"Seize: every"}},
#     "Seize: every": {"add": {"Seize: every"}, "remove": {"Seize: first"}},
#   GF Armor on vs off:
#     "with GF Armor": {"add": {"GF Armor"}, "remove": set()},
#     "no GF Armor":   {"add": set(),        "remove": {"GF Armor"}},

# Restrict base loadouts to ones where the comparison is meaningful (e.g. only builds that
# could plausibly carry either pursuit). Default: a stratified sample of the whole pool.
def base_candidates():
    # exclude loadouts that already hard-commit to one side's tags in a way that makes the
    # swap incoherent; here we just take a broad sample across retinues.
    from collections import defaultdict
    by_ret = defaultdict(list)
    for ld in pool: by_ret[ld.retinue].append(ld)
    out = []
    for ret, lds in by_ret.items():
        out.extend(random.sample(lds, min(len(lds), 12)))
    return out

bases = base_candidates()
print(f"Base loadouts under test: {len(bases)}")

# Fixed reference field
field = [ld._replace(playstyle=playstyles.assign_default_playstyle(ld))
         for ld in random.sample(pool, 40)]

def make_variant(ld, add_tags, remove_tags):
    tags = [t for t in ld.extra_tags if t not in remove_tags]
    for t in add_tags:
        if t not in tags: tags.append(t)
    v = ld._replace(extra_tags=tags)
    return v._replace(playstyle=playstyles.assign_default_playstyle(v))

def field_wr(ld, n=60):
    w = t = 0
    for i, opp in enumerate(field):
        if opp.name == ld.name: continue
        r = vectorized_combat.run_matchup_vec(ld, opp, n_runs=n, seed=400+i,
                                              a_playstyle=ld.playstyle, b_playstyle=opp.playstyle)
        w += r['a_wins']; t += r['a_wins']+r['b_wins']+r['mut_wipe']+r['indecisive']
    return w/t if t else np.nan

labels = list(AB_PAIRS.keys())
assert len(labels) == 2, "AB_PAIRS must have exactly two entries"
A, B = labels

rows = []
for ld in bases:
    va = make_variant(ld, AB_PAIRS[A]["add"], AB_PAIRS[A]["remove"])
    vb = make_variant(ld, AB_PAIRS[B]["add"], AB_PAIRS[B]["remove"])
    wr_a = field_wr(va); wr_b = field_wr(vb)
    rows.append({"weapon": ld.weapon or ld.ranged, "retinue": ld.retinue,
                 "armor": ld.armor, f"wr_{A}": wr_a, f"wr_{B}": wr_b,
                 "delta_B_minus_A": wr_b - wr_a})
df = pd.DataFrame(rows)

print(f"\n=== {A}  vs  {B}  (same loadouts, same field) ===")
print(df.sort_values("delta_B_minus_A", ascending=False).round(3).to_string(index=False))
print(f"\nMean:  {A}={df[f'wr_{A}'].mean():.3f}   {B}={df[f'wr_{B}'].mean():.3f}   Δ(B−A)={df['delta_B_minus_A'].mean():+.4f}")
print(f"{B} better in {(df['delta_B_minus_A']>0.005).sum()}/{len(df)} builds; "
      f"{A} better in {(df['delta_B_minus_A']<-0.005).sum()}/{len(df)}")

# Per-retinue and per-weapon breakdown (where does each pursuit win?)
print(f"\n=== Δ(B−A) by retinue ===")
print(df.groupby("retinue")["delta_B_minus_A"].agg(["mean","count"]).round(4).to_string())
print(f"\n=== Δ(B−A) by weapon (top/bottom 5) ===")
byw = df.groupby("weapon")["delta_B_minus_A"].mean().sort_values()
print(pd.concat([byw.head(5), byw.tail(5)]).round(4).to_string())

# Plot
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(10, 5))
ds = df.sort_values("delta_B_minus_A")
colors = ["#27ae60" if d > 0 else "#c0392b" for d in ds["delta_B_minus_A"]]
ax.barh(range(len(ds)), ds["delta_B_minus_A"], color=colors, alpha=0.85)
ax.axvline(0, color="black", lw=0.5)
ax.set_yticks([]); ax.set_xlabel(f"Δ win rate  ({B} − {A})")
ax.set_title(f"{A} vs {B}: per-build win-rate delta\n(green = {B} better, red = {A} better)")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "pursuit_ab_test.png"), dpi=110, bbox_inches="tight"); plt.show()
print(f"\nSaved: {os.path.join(OUT_DIR,'pursuit_ab_test.png')}")

In [ ]:
# === Monument comparison: Sergeant vs Man-at-Arms delta win rate ===
# Each monument's effect added to a controlled baseline (Forged weapon / Full Plate / size 25),
# measured as Δ win rate vs that same retinue's bare baseline, across a fixed opponent field.
# Two columns: how much the monument is worth on a Sergeant vs on a Man-at-Arms.
import importlib, renown_combat, loadouts, playstyles, vectorized_combat
import numpy as np, random, matplotlib.pyplot as plt
for _m in [renown_combat, loadouts, playstyles, vectorized_combat]: importlib.reload(_m)
vectorized_combat.invalidate_tactic_tables()
from loadouts import Loadout

N_RUNS = 100
pool = MAIN_POOL   # reuse the main tournament pool (no regeneration)
random.seed(7)
field = [ld._replace(playstyle=playstyles.assign_default_playstyle(ld)) for ld in random.sample(pool, 40)]

def fwr(ld, n=N_RUNS):
    w = t = 0
    for i, opp in enumerate(field):
        if opp.name == ld.name: continue
        r = vectorized_combat.run_matchup_vec(ld, opp, n_runs=n, seed=300+i,
                a_playstyle=ld.playstyle, b_playstyle=opp.playstyle)
        w += r["a_wins"]; t += r["a_wins"]+r["b_wins"]+r["mut_wipe"]+r["indecisive"]
    return w/t if t else float("nan")

def mk(weap, sh, ret, tags=None, arm="Full Plate"):
    b = Loadout(name=f"{ret}/{weap}", retinue=ret, weapon=weap, shield=sh, armor=arm,
        ranged=None, has_tiltyard=False, size=25, extra_tags=(tags or [])+["Cond Field"],
        upkeep_per_retinue=30, playstyle="Aggressor", tiltyard_mastery=False,
        pursuits=frozenset(["Coliseum"]), military_pursuit_count=5, domain_count=0)
    return b._replace(playstyle=playstyles.assign_default_playstyle(b))

FORGED = [("Halberd",None),("Bastard Sword","Tower Shield"),("Lance","Scutum Shield"),("War Hammer",None),
          ("Battle Axe",None),("Flail","Tower Shield"),("Morningstar","Tower Shield"),("2HBastard",None)]

# Monument effects. "KT" = retinue swap to Knight Templar (Preceptory mastery: unlocks KT +
# grants Steadfast on top of the innate Unshakable). Royal Pavilion's Immune Strain auto-grants
# +1 endurance in the engine, so no manual endurance override is needed here.
EFFECTS = {
    "Preceptory innate (Unshakable)":                    ["Unshakable"],
    "Preceptory mastery (->KT, +Steadfast)":             "KT",
    "Royal Pavilion (Drilled/Nimble/ImmStrain +1End)":   ["Drilled","Nimble","Immune Strain"],
    "Outrider mastery (first_two)":                      ["Outrider: first_two"],
    "Ministry innate (Seize: first)":                    ["Seize: first"],
    "Ministry mastery (Seize: every)":                   ["Seize: every"],
}
# ABF: tier-unlock value = best Crafted build vs Forged+Full Plate baseline (same retinue).
ABF_PAIRS = [
    (("War Hammer","Full Plate",None),     ("Poleaxe","Gothic Plate",None)),
    (("Bastard Sword","Full Plate","Tower Shield"),  ("Bastard Sword","Gothic Plate","Heater Shield")),
    (("2HBastard","Full Plate",None),        ("Poleaxe","Gothic Plate",None)),
    (("Lance","Full Plate","Scutum Shield"),          ("Bastard Sword","Gothic Plate","Heater Shield")),
]

def col(ret):
    out = {}
    for weap, sh in FORGED:
        bw = fwr(mk(weap, sh, ret))
        for name, spec in EFFECTS.items():
            if spec == "KT":
                v = fwr(mk(weap, sh, "Knight Templar", tags=["Steadfast"])) - bw
            else:
                v = fwr(mk(weap, sh, ret, tags=spec)) - bw
            out.setdefault(name, []).append(v)
    abf = []
    for (fw, fa, fs), (cw, ca, cs) in ABF_PAIRS:
        abf.append(fwr(mk(cw, cs, ret, arm=ca)) - fwr(mk(fw, fs, ret, arm=fa)))
    out["ABF monument (Crafted equipment)"] = abf
    return {k: np.mean(v) for k, v in out.items()}

sgt = col("Sergeant")
maa = col("Man-at-Arms")
order = sorted(sgt, key=lambda k: -sgt[k])

print("=== Monument delta WR (Forged+Full Plate baseline, size 25) ===")
print(f"{'Monument':52} {'Sgt ΔWR':>9} {'MaA ΔWR':>9}")
for k in order:
    print(f"  {k:25} {sgt[k]:+.4f} {maa[k]:+.4f}")

# Grouped bar chart
fig, ax = plt.subplots(figsize=(11, 6))
y = np.arange(len(order)); h = 0.4
ax.barh(y+h/2, [sgt[k] for k in order], h, label="Sergeant", color="#2c7fb8")
ax.barh(y-h/2, [maa[k] for k in order], h, label="Man-at-Arms", color="#d95f0e")
ax.set_yticks(y); ax.set_yticklabels(order, fontsize=9)
ax.invert_yaxis(); ax.axvline(0, color="gray", lw=0.8)
ax.set_xlabel("Δ win rate vs bare baseline"); ax.legend()
ax.set_title("Monument value: Sergeant vs Man-at-Arms")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "monument_sgt_vs_maa.png"), dpi=110, bbox_inches="tight"); plt.show()

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Keyword impact via SYNTHETIC injection — ALL keywords, not just pursuit-granted
# ──────────────────────────────────────────────────────────────────────────────
# Instead of finding existing carriers, we take a fixed set of base loadouts and
# ADD each keyword to extra_tags (toggle ON), then compare to the same base (OFF)
# against a fixed reference field. This tests every keyword the engine reads,
# including innate-only ones no pursuit grants.
#
# NOTE: this is a "what if this keyword were added" experiment. Some keywords here
# are normally innate (weapon/retinue) or unobtainable via pursuits — injecting
# them measures their raw mechanical value, not their in-game availability.

import importlib, numpy as np, pandas as pd, random
import loadouts, renown_combat, vectorized_combat, playstyles
importlib.reload(loadouts); importlib.reload(vectorized_combat)
vectorized_combat.invalidate_tactic_tables()
from collections import defaultdict

pool = MAIN_POOL

# EVERY keyword the engine reads (build-relevant). Grouped for readability.
ALL_KEYWORDS = [
    # offensive / weapon effects
    "Shatter Armor", "Cleave", "Destroy Shield", "Rend", "Poison", "Unstoppable",
    "Steady", "Nimble", "Unwieldy",
    # defensive / regen / heal
    "Regenerate", "Regenerate 5", "Regenerate 4", "Regenerate Reroll", "Apothecary Heal",
    # morale / discipline
   "Unshakable", "Drilled", "Cond Field",
    # immunities
    "Immune Poison", "Immune Strain", "Immune Unwieldy",
    "Immune Blocked", "Immune Destroy Shield", "Immune Nimble",
    # spec gear effects
    "GF Armor", "MW Weapons", "Parry",
    # initiative tempo (Seize / Outrider)
    "Seize: first", "Seize: first_two", "Seize: every",
    "Outrider: once", "Outrider: first", "Outrider: first_two", "Outrider: every",
    "Riposte"
]

# Fixed reference field (same gauntlet for every comparison).
random.seed(2026)
buckets = defaultdict(list)
for ld in pool:
    buckets[(ld.retinue, ld.military_pursuit_count)].append(ld)
ref_field = []
for key, lds in buckets.items():
    ref_field.extend(random.sample(lds, min(len(lds), 1)))
if len(ref_field) > 60:
    ref_field = random.sample(ref_field, 60)
ref_field = [r._replace(playstyle=playstyles.assign_default_playstyle(r)) for r in ref_field]
print(f"Reference field: {len(ref_field)} opponents\n")

# Fixed set of base loadouts to inject keywords onto. Spread across retinues so the
# delta isn't dominated by one chassis. We pick builds that DON'T already carry the kw.
random.seed(99)
base_candidates = []
seen = defaultdict(int)
for ld in random.sample(pool, len(pool)):           # shuffled
    if seen[ld.retinue] < 6:                         # up to 6 per retinue
        base_candidates.append(ld); seen[ld.retinue] += 1
print(f"Base loadouts for injection: {len(base_candidates)} ({dict(seen)})\n")

def field_winrate(ld, field, n_runs=100):
    ld = ld._replace(playstyle=playstyles.assign_default_playstyle(ld))
    wins = total = 0
    for i, opp in enumerate(field):
        if opp.name == ld.name:
            continue
        r = vectorized_combat.run_matchup_vec(ld, opp, n_runs=n_runs, seed=4242+i, max_skirmishes=20)
        wins += r['a_wins']; total += (r['a_wins'] + r['b_wins'] + r['mut_wipe'] + r['indecisive'])
    return wins / total if total else float('nan')

rows = []
for kw in ALL_KEYWORDS:
    # Use base loadouts that don't already have the keyword
    bases = [ld for ld in base_candidates if kw not in ld.extra_tags][:12]
    if not bases:
        print(f"  {kw:<22} (no eligible base loadouts — skipped)")
        continue
    deltas = []
    for ld in bases:
        without_kw = ld._replace(extra_tags=[t for t in ld.extra_tags if t != kw])
        with_kw    = ld._replace(extra_tags=list(ld.extra_tags) + [kw])
        wr_off = field_winrate(without_kw, ref_field)
        wr_on  = field_winrate(with_kw,    ref_field)
        deltas.append(wr_on - wr_off)
    deltas = np.array(deltas)
    rows.append({
        "keyword": kw, "n_tested": len(bases),
        "mean_delta": deltas.mean(), "median_delta": np.median(deltas),
        "min_delta": deltas.min(), "max_delta": deltas.max(), "std": deltas.std(),
    })
    print(f"  {kw:<22} Δ={deltas.mean():+.3f}  (n={len(bases)}, range [{deltas.min():+.3f}, {deltas.max():+.3f}])")

res = pd.DataFrame(rows).sort_values("mean_delta", ascending=False)
print("\n=== Keyword isolated win-rate impact (SYNTHETIC inject on vs off, vs fixed field) ===")
print(res[["keyword","n_tested","mean_delta","median_delta","min_delta","max_delta"]].round(3).to_string(index=False))

# Plot
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(10, max(6, 0.32*len(res))))
rs = res.sort_values("mean_delta")
ax.barh(range(len(rs)), rs["mean_delta"],
        xerr=[rs["mean_delta"]-rs["min_delta"], rs["max_delta"]-rs["mean_delta"]],
        color="#2980b9", alpha=0.8, capsize=3, ecolor="gray")
ax.set_yticks(range(len(rs))); ax.set_yticklabels(rs["keyword"])
ax.axvline(0, color="black", lw=0.5)
ax.set_xlabel("Isolated win-rate impact (with keyword − without)")
ax.set_title("Keyword impact via SYNTHETIC injection (all engine keywords)\n(error bars = min/max across tested base loadouts)")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "keyword_ab_all.png"), dpi=110, bbox_inches="tight")
plt.show()
print(f"\nSaved: {os.path.join(OUT_DIR,'keyword_ab_all.png')}")

# 10. Retinue-vs-Retinue Matrices

A x B retinue matrices (A = attacker/initiator slot) for kills, shakes, skirmish length,
rout kills, win rate, and mutual-wipe rate. Reads `df_main_matchups`.

In [ ]:
# Retinue matchup matrices — kills / shakes / skirmish length / rout / win rate
mats = analysis.retinue_matchup_matrices(df_main_matchups)

_titles = {
    "a_win_rate":       "A win rate (A=row retinue vs B=col retinue)",
    "mut_wipe_rate":    "Mutual-wipe rate",
    "indecisive_rate":  "Indecisive rate",
    "avg_skirm":        "Avg battle length (skirmishes)",
    "a_kills_combat":   "A combat kills on B / battle",
    "b_kills_combat":   "B combat kills on A / battle (A's losses)",
    "a_kills_shake":    "A kills via B failed shaking / battle",
    "a_rout_kills":     "A kills when B routs / battle",
    "b_rout_kills":     "B kills when A routs / battle",
    "a_shield_destroyed": "A shield-destroyed rate",
    "n_battles":        "N battles (sample weight)",
}
for key, title in _titles.items():
    print(f"\n=== {title} ===")
    print(mats[key].round(3).to_string())

# Heatmap grid of the headline matrices
import matplotlib.pyplot as plt
_show = ["a_win_rate","mut_wipe_rate","avg_skirm","a_kills_combat","b_kills_combat","a_kills_shake"]
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, key in zip(axes.flat, _show):
    M = mats[key]
    im = ax.imshow(M.values.astype(float), cmap="viridis", aspect="auto")
    ax.set_xticks(range(len(M.columns))); ax.set_xticklabels(M.columns, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(len(M.index))); ax.set_yticklabels(M.index, fontsize=8)
    ax.set_title(_titles[key], fontsize=10)
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            v = M.values[i, j]
            if v == v:  # not NaN
                ax.text(j, i, f"{v:.2f}", ha="center", va="center", color="w", fontsize=8)
    fig.colorbar(im, ax=ax, fraction=0.046)
axes.flat[0].set_ylabel("A retinue"); 
plt.suptitle("Retinue x Retinue matchup matrices", fontsize=13)
plt.tight_layout(); plt.show()

# 11. Strongest Performers by MPC

Top builds within each Military Pursuit Count bucket, plus a per-MPC rollup. Reads
`df_main_summary` (one row per build).

In [ ]:
# Top performers within each MPC bucket
TOP_N_PER_MPC = 8
top_mpc = analysis.top_performers_by_mpc(df_main_summary, top_n=TOP_N_PER_MPC, metric="win_rate")

print("=== Per-MPC rollup ===")
mpc_tbl = analysis.mpc_summary_table(df_main_summary)
print(mpc_tbl.round(3).to_string(index=False))

print(f"\n=== Top {TOP_N_PER_MPC} builds per MPC ===")
for mpc, g in top_mpc.groupby("military_pursuit_count"):
    print(f"\n--- MPC {mpc} ---")
    print(g[["mpc_rank","name","retinue","win_rate","decisive_win_rate"]].to_string(index=False)
          if "decisive_win_rate" in g.columns
          else g[["mpc_rank","name","retinue","win_rate"]].to_string(index=False))

# Plot: win-rate distribution per MPC (box) + best build trend
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
ax = axes[0]
mpcs = sorted(df_main_summary["military_pursuit_count"].unique())
data = [df_main_summary[df_main_summary["military_pursuit_count"]==m]["win_rate"].values for m in mpcs]
try:
    ax.boxplot(data, tick_labels=mpcs, showmeans=True)   # matplotlib >= 3.9
except TypeError:
    ax.boxplot(data, labels=mpcs, showmeans=True)         # older matplotlib
ax.axhline(0.5, color="gray", ls=":", alpha=0.5)
ax.set_xlabel("MPC"); ax.set_ylabel("Win rate"); ax.set_title("Win-rate distribution by MPC")
ax.grid(alpha=0.3, axis="y")
ax = axes[1]
ax.plot(mpc_tbl["military_pursuit_count"], mpc_tbl["max_wr"], "o-", label="best build", color="#c43838")
ax.plot(mpc_tbl["military_pursuit_count"], mpc_tbl["mean_wr"], "s-", label="mean", color="#4a90e2")
ax.plot(mpc_tbl["military_pursuit_count"], mpc_tbl["median_wr"], "^-", label="median", color="#d4a800")
ax.axhline(0.5, color="gray", ls=":", alpha=0.5)
ax.set_xlabel("MPC"); ax.set_ylabel("Win rate"); ax.set_title("Win rate vs MPC")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

# 12. Confound-Reduced Views

`analysis.filter_confounds` controls for the main confounds: mutual-wipe inflation,
identical-gear mirror matchups, and unequal MPC. Compare raw vs filtered retinue win rates.

In [ ]:
# Confound filtering: compare raw vs controlled retinue win rates
# 1) Decisive-only + mirror-gear excluded: removes mutual-wipe washout (the Sgt-vs-MaA case)
df_decisive = analysis.filter_confounds(df_main_matchups, decisive_only=True, exclude_mirror_gear=True)

def _retinue_wr(d, col):
    g = d.groupby("a_retinue").apply(
        lambda x: (x[col] * x["n_runs"].clip(lower=1)).sum() / x["n_runs"].clip(lower=1).sum())
    return g.reindex(analysis.RETINUE_ORDER)

raw   = _retinue_wr(df_main_matchups, "a_win_rate")
dec   = _retinue_wr(df_decisive, "a_win_rate_decisive")
cmp = pd.DataFrame({"raw_win_rate": raw, "decisive_only_win_rate": dec})
cmp["delta"] = cmp["decisive_only_win_rate"] - cmp["raw_win_rate"]
print("=== Retinue win rate: raw vs decisive-only (mirror gear excluded) ===")
print(cmp.round(3).to_string())

# 2) Same-MPC only: control for military investment
df_sameMPC = analysis.filter_confounds(df_main_matchups, same_mpc=True, mpc_tolerance=0)
print(f"\nSame-MPC filter: {len(df_main_matchups)} -> {len(df_sameMPC)} rows (|a_mpc - b_mpc| == 0)")
if len(df_sameMPC):
    same = _retinue_wr(df_sameMPC, "a_win_rate")
    print("Retinue win rate, same-MPC matchups only:")
    print(same.round(3).to_string())

# 3) High-decisiveness only: drop stalematey matchups
df_decisiveOnly = analysis.filter_confounds(df_main_matchups, min_decisive_rate=0.5)
print(f"\nDecisive-rate >= 0.5 filter: {len(df_main_matchups)} -> {len(df_decisiveOnly)} rows")

# 13. The Story of a Battle — High-Level View

Battle-as-the-unit overview: win / loss / mutual / indecisive rates, survivor counts, the kill mix (combat vs failed-morale vs rout), and how the underlying combat stats (to-hit, initiative, save, endurance) correlate with winning. Reads `df_main_matchups`, `df_main_summary`, `MAIN_POOL`.

In [ ]:
# ── Battle overview + combat-stat correlations + narrative ──
import matplotlib.pyplot as plt
import analysis

ov   = analysis.battle_overview(df_main_matchups)
st   = analysis.build_stat_table(MAIN_POOL)
m, corr = analysis.stat_winrate_correlation(df_main_summary, st)

pop = {"avg_to_hit": st["to_hit"].mean(), "avg_init": st["init"].mean(),
       "avg_eff_save": st["eff_save"].mean(), "avg_endurance": st["endurance"].mean()}

print("=" * 68)
print("  THE STORY OF A BATTLE  —  {:,} battles".format(ov["total_battles"]))
print("=" * 68)
print(f"""
A typical engagement runs about {ov['avg_skirmishes']:.1f} skirmishes. When the dust settles,
{ov['decisive_rate']*100:.0f}% of battles have a clear winner; the attacker takes
{ov['attacker_win_rate']*100:.0f}% and the defender {ov['defender_win_rate']*100:.0f}% — close to
even, so the initiator slot is not a strong edge. {ov['mutual_wipe_rate']*100:.0f}% of battles end
in MUTUAL annihilation, and only {ov['indecisive_rate']*100:.1f}% peter out indecisively.

The two sides leave roughly {ov['avg_attacker_survivors']:.0f} and {ov['avg_defender_survivors']:.0f}
survivors on average. Of every casualty inflicted, {ov['kill_mix_combat_pct']*100:.0f}% fall in direct
combat, {ov['kill_mix_shake_pct']*100:.0f}% break from exhaustion (shaken), {ov['kill_mix_waver_pct']*100:.0f}% break
under heavy losses (waver), and {ov['kill_mix_rout_pct']*100:.0f}% are cut down in the rout. Shields are destroyed in
{ov['avg_shield_destroyed_rate']*100:.1f}% of battles.

WHAT WINS BATTLES (correlation of each combat stat with win rate):""")
for _, r in corr.iterrows():
    s = r["stat"]; c = r["corr_with_win_rate"]
    better = "lower is better" if s in ("to_hit","eff_save","shaking","weapon_ap") else "higher is better"
    print(f"   {s:11s} {c:+.2f}  {'#'*int(abs(c)*30):30s} ({better})")
print(f"""
Innate ACCURACY (to-hit) and MORALE (shaking) dominate — they outweigh weapon AP and initiative.
Endurance is the strongest positive driver. Population averages: to-hit {pop['avg_to_hit']:.1f},
initiative {pop['avg_init']:+.1f}, effective save {pop['avg_eff_save']:.1f}, endurance {pop['avg_endurance']:.1f}.
""")

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
ax = axes[0,0]
labels = ["Attacker win","Defender win","Mutual wipe","Indecisive"]
vals = [ov["attacker_win_rate"], ov["defender_win_rate"], ov["mutual_wipe_rate"], ov["indecisive_rate"]]
ax.bar(labels, vals, color=["#4a90e2","#d4a800","#c43838","#888"])
ax.set_title("Battle outcome split"); ax.set_ylabel("Fraction of battles")
ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=20, ha="right")
for i,v in enumerate(vals): ax.text(i, v+0.005, f"{v*100:.0f}%", ha="center", fontsize=9)
ax = axes[0,1]
mix = [ov["kill_mix_combat_pct"], ov["kill_mix_shake_pct"], ov.get("kill_mix_waver_pct",0.0), ov["kill_mix_rout_pct"]]
ax.pie(mix, labels=["Combat","Shaken","Waver","Rout"], autopct="%1.0f%%",
       colors=["#c43838","#d4a800","#e07b39","#4a90e2"], startangle=90)
ax.set_title("How casualties are inflicted")
ax = axes[1,0]
cc = corr.sort_values("corr_with_win_rate")
ax.barh(cc["stat"], cc["corr_with_win_rate"], color=["#c43838" if v<0 else "#4a90e2" for v in cc["corr_with_win_rate"]])
ax.axvline(0, color="k", lw=0.8); ax.set_title("Combat stat vs win rate (Pearson r)")
ax.set_xlabel("correlation with win rate")
ax = axes[1,1]
ret_colors={"Levy":"#888","Man-at-Arms":"#4a90e2","Sergeant":"#d4a800","Knight Templar":"#c43838"}
for ret,grp in m.groupby("retinue"):
    ax.scatter(grp["to_hit"]+(hash(ret)%5-2)*0.03, grp["win_rate"], s=8, alpha=0.4, label=ret, color=ret_colors.get(ret,"#555"))
ax.set_xlabel("to-hit target (lower = more accurate)"); ax.set_ylabel("win rate")
ax.set_title("Win rate vs to-hit, by retinue"); ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

# 14. Controlled Initiative Sweep

Population-level initiative correlation is near-zero, but that averages over a confound (high-init weapons tend to be low-AP and vice versa). This cell isolates initiative: it holds the CHASSIS fixed (retinue + weapon + shield + armor) and varies ONLY initiative via a synthetic `TestInit±N` modifier, against a fixed opponent field.

Note the engine clamps base initiative to [-2, max_init] (default max_init=2; Ministry MaxInit3 raises it to 3, but that only matters for breaking init-2 ties), and init <= -2 is a "trip cliff". So the meaningful band is -1..+2.

In [ ]:
# Controlled initiative sweep — fixed chassis, vary only initiative
import matplotlib.pyplot as plt
LD = loadouts.Loadout
def _mk(name, ret, wep, shield, armor, tags=frozenset()):
    return LD(name=name, retinue=ret, weapon=wep, shield=shield, armor=armor, ranged=None,
              has_tiltyard=False, size=25, extra_tags=tags, upkeep_per_retinue=0,
              playstyle=None, tiltyard_mastery=False, pursuits=frozenset())

# Chassis to sweep (edit freely). (retinue, weapon, shield, armor)
SWEEP_CHASSIS = [
    ("Sergeant",    "Arming Sword", "Heater Shield", "Chainmail"),
    ("Man-at-Arms", "Bastard Sword","Kite Shield",   "Chainmail"),
    ("Sergeant",    "Poleaxe",      None,            "Full Plate"),
    ("Knight Templar","Bastard Sword","Heater Shield","Full Plate"),
]
INIT_DELTAS = [-3, -2, -1, 0, 1, 2, 3]
SWEEP_NR = 60

import random as _random
_random.seed(7)
_field = _random.sample(MAIN_POOL, min(50, len(MAIN_POOL)))

results = {}
for ret, wep, sh, ar in SWEEP_CHASSIS:
    wrs = []
    for d in INIT_DELTAS:
        tag = frozenset({f"TestInit{d:+d}"}) if d != 0 else frozenset()
        actor = _mk(f"{ret}/{wep}", ret, wep, sh, ar, tag)
        pairs = [(actor, b) for b in _field if b.name != actor.name]
        r = batch_engine.run_batch_random(pairs, n_runs=SWEEP_NR, seed=2026, mode="random")
        tot = r["a_wins"] + r["b_wins"] + r["mut_wipe"] + r["indecisive"]
        wrs.append(r["a_wins"].sum() / tot.sum())
    label = f"{ret}/{wep}/{sh or 'None'}"
    results[label] = wrs

print(f"{'chassis':48s} " + " ".join(f"{d:+d}I" for d in INIT_DELTAS) + "   spread (viable band -1..+2)")
for label, wrs in results.items():
    band = [w for d, w in zip(INIT_DELTAS, wrs) if -1 <= d <= 2]
    print(f"{label:48s} " + " ".join(f"{w:.3f}" for w in wrs) + f"   {max(band)-min(band):.3f}")

fig, ax = plt.subplots(figsize=(11, 6))
for label, wrs in results.items():
    ax.plot(INIT_DELTAS, wrs, "o-", label=label)
ax.axvspan(-3, -2, color="red", alpha=0.08)
ax.axvline(-2, color="red", ls="--", alpha=0.5)
ax.text(-2.9, ax.get_ylim()[1]*0.95, "trip cliff\n(init <= -2)", color="red", fontsize=8, va="top")
ax.axvline(2, color="gray", ls=":", alpha=0.5)
ax.text(2.05, ax.get_ylim()[0]+0.02, "max_init clamp", color="gray", fontsize=8)
ax.set_xlabel("Initiative delta on a FIXED chassis")
ax.set_ylabel("Win rate vs fixed field")
ax.set_title("Controlled initiative sweep (only initiative varies)")
ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print("\nReading: a flat line = initiative-agnostic balance. A rising line = initiative helps on")
print("that chassis. The drop below -2 is the trip cliff; the plateau at +2 is the max_init clamp.")

In [ ]:
fb = analysis.feature_bucket_matrix(df_main_matchups, min_support=20)
# raw vs controlled for the key metrics
cols = ["feature","n_has","n_not",
        "win_rate_raw","win_rate_ctrl",
        "death_combat_raw","death_combat_ctrl",
        "death_shake_raw","death_shake_ctrl",
        "death_rout_raw","death_rout_ctrl"]
print(fb[cols].round(3).to_string(index=False))
fb.to_csv(os.path.join(OUT_DIR, "feature_bucket_matrix.csv"), index=False)  # full matrix, all metrics

In [ ]:
import analysis, importlib; importlib.reload(analysis)
df = df_main_matchups   # your loaded tournament

# 1. DOMINANCE — which builds are trap picks (cheaper build wins more)
front, dom = analysis.dominance_frontier(df)
print(f"Frontier: {len(front)} efficient builds | Dominated: {len(dom)} traps")
display(dom.head(20))                    # worst traps first — these need buffs or cost cuts
display(front.head(15))

# plot the frontier:
import matplotlib.pyplot as plt
plt.scatter(front.mpc, front.win_rate, c='tab:blue', label='frontier')
if len(dom): plt.scatter(dom.mpc, dom.win_rate, c='lightgray', s=8, label='dominated')
plt.xlabel('MPC (cost)'); plt.ylabel('win rate'); plt.legend(); plt.show()

# 2. CHEAPEST COUNTER — over-tuned builds you can't answer on-budget
cc = analysis.cheapest_counter(df, win_threshold=0.55)
print("Uncounterable builds:", (cc.n_counters==0).sum())
display(cc.head(20))                      # top = uncounterable / must-overpay-to-beat

# 3. TAG SYNERGY — combos that exceed the sum of their parts
syn = analysis.tag_synergy_matrix(df, min_support=30)
display(syn.head(20))                      # watch Parry+Riposte, Unstoppable+Shatter, etc.

# 4. INTRANSITIVITY — is the meta deep RPS or a solved ladder?
print(analysis.intransitivity(df, win_margin=0.55))